# 전처리 연습 (Tokenize, Cleansing & Normalization, Stemming & Lemmatization)

1. 데이터셋을 찾는다. (만들어진 데이터셋, 크롤링, ...)
2. 전처리<br>
    2-1. 의미가 있는 단어로 vocabulary<br>
    2-2. corpus -> 토큰화 + 전처리 -> 문장

In [1]:
import pandas as pd
import numpy as np

In [ ]:
# 위키피디아
import bz2
import xml.etree.ElementTree as ET

# 파일 경로
file_path = 'data/kowiki-latest-pages-articles.xml.bz2'

def parse_kowiki():
    # bz2 파일을 압축 해제 없이 바로 읽기
    with bz2.open(file_path, 'rb') as f:
        # XML의 네임스페이스(Namespace) 정의
        namespace = {'ns': 'http://www.mediawiki.org/xml/export-0.11/'}
        
        # 반복적으로 트리 요소를 읽어 메모리 효율 극대화 (iterparse)
        context = ET.iterparse(f, events=('end',))
        
        count = 0
        for event, elem in context:
            # 태그 이름에서 네임스페이스 제거 후 'page' 태그만 찾기
            tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
            
            if tag == 'page':
                title = elem.find('ns:title', namespace).text
                # 리다이렉트 문서는 건너뛰고 본문이 있는 문서만 출력
                revision = elem.find('ns:revision', namespace)
                text_elem = revision.find('ns:text', namespace)
                
                if text_elem is not None and text_elem.text:
                    content = text_elem.text[:100] # 앞부분 100자만 샘플로 확인
                    print(f"제목: {title}")
                    print(f"내용 요약: {content}...")
                    print("-" * 30)
                    
                count += 1
                if count >= 5: # 너무 많으니까 5개만 보고 멈춤
                    break
                
                # 처리가 끝난 요소는 메모리에서 삭제
                elem.clear()

if __name__ == "__main__":
    parse_kowiki()

제목: 위키백과:대문
내용 요약: <templatestyles src="틀:대문/styles.css" />
<div class="main-box main-top" style="flex: 10;">
	<div cla...
------------------------------
제목: 지미 카터
내용 요약: {{대통령 정보
| 이름 = 지미 카터
| 원어명 = {{lang|en|Jimmy Carter}}
| 그림 = JimmyCarterPortrait.jpg
| 크기 = 300px
|...
------------------------------
제목: 수학
내용 요약: {{학문 정보
| 학문명 = 수학
| 그림 = Portal Math Banner Background ka.jpg
| 그림크기 = 330
| 그림설명 = 
| 다른 이름 = 
| 연...
------------------------------
제목: 수학 상수
내용 요약: '''[[수학]]'''에서 '''상수'''(常數, {{lang|en|constant}})란 그 값이 변하지 않는 불변량으로, [[변수 (수학)|변수]]의 반대말이다. [[물리 상수...
------------------------------
제목: 도움말:도움말
내용 요약: {{다른 뜻|위키백과:도움말 이름공간||위키백과의 이름공간}}
{{도움말 문서 머리말}}
<div class="cp_contentsbox" style="border:1px soli...
------------------------------


### 유튜브 댓글 크롤링

In [ ]:
# !pip install selenium

  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 6.3 MB/s  0:00:01 eta 0:00:01
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached websocket_client-1.9.0-py3-none-any.whl (82 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached outcome-1.3.0.post0-py2.py3-none-any.whl (10 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
Using cached wsproto-1.3.2-py3-none-any.whl (24 kB)
Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [20]:
# import time
# from selenium.webdriver import Chrome
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.keys import Keys
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC

# driver = webdriver.Chrome() 

# youtube_video_url = "https://youtu.be/_SErD-l0F18?si=gPf93I7JS5kVEfoi"
# data = []

# try:
#     driver.get(youtube_video_url)
#     wait = WebDriverWait(driver, 20)

#     driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.PAGE_DOWN)
#     time.sleep(3)
    

#     for i in range(100): 
#         driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.END)
#         time.sleep(2)
#         print(f'{i}페이지 탐색 완료')


#     comments = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#content-text")))

#     for comment in comments:
#         data.append(comment.text)

#     print(f"총 {len(data)}개의 댓글을 수집했습니다.")

# finally:
#     time.sleep(5)
#     driver.quit()

In [9]:
data = ["리포트 받아보기: https://bit.ly/44Jfos7 \n\n저희는 국내에서 유일한 기술경제언론 '하이젠버그'입니다.\n현재 유튜브 멤버십 가입을 통해 이공계 석박사 현업자들이 각 기업과 기술에 대해 분석한 리포트를 매일 받아보실 수 있습니다.\n하이젠버그 '스탠다드' 등급은 리포트를 받아보실 수 있고, '비즈니스' 등급은 투자 의견과 함께 연구자와 컨택도 가능합니다.\n\n비용이 부담스러우신 분들은 아래의 뉴스레터를 통해 무료로 맛보기를 하실 수 있으니 많은 신청 바랍니다.\n뉴스레터 신청하기 : ",
 '회사 문닫고 떠나면 장사안된다고 징징거릴 상인들.',
 '구구절절 맞는 말씀입니다. 기업이 살아야 나라가 살고 국민이 삽니다.',
 '감사합니다- 정확한 지적이십니다. 삼성 현대   sk 등 우리 기업들과 우리나라를 응원합니다.',
 '이재용회장님을 언제나 응원합니다!\n또한 대한민국 국민의 한 국민으로서 그 노고를 ...참 치하합니다 저는 할머니입니다',
 '국내 투자하는 기업에 우대해줘야 합니다\n그래야 일자리가 생기죠',
 '삼성, SK응원합니다 겨우4명 먹여 살리는 초초소기업 운영하는데도 이리 힘든데 삼성,SK 같이 몇십만명 몇백만명 먹여 살리는 두분보면 존경스럽기까지 하다',
 '너무 좋은 내용입니다.\n 기업을 키워야 국가가 부흥하지요.\n응원합니다.',
 '부산살다가 청주산지 10년 되었습니다.  전국이 다 불경기인데 청주는 하이닉스 때문인지   호황기인거 같습니다.  월세가 없습니다.  전국에서 몰려온 하이닉스 공사인부들때문에   원룸 . 모텔.아파트까지 월세를 구할수가없을정도입니다.  대기업의 역활이 얼마나 큰지 세삼 느낍니다.  한국이 계속 경제가발전하는데  모든국민이  노력해주는게 맞는거같습니다. 결국은 나라가 결국은 국민이 혜택을 입는것 같습니다.',
 '저 지역주민들은 굶어봐야 정신차리지.',
 '이런방송 잘하고 있어요.\n우리가 미처몰랏던 사실들 알려줘서 고마워요 ~',
 '구내식당 만들려고하면 주변 식당가들에서 시위하는게 짜치는게 전 회사가 신사옥 지을때 구내식당 만들려다가 구청장 까지와서 주변 식당이랑 상생하라고 만들지 말라고 압박하더니 결국 구내식당 없는데 주변식당 가격들 한끼에 10000~12000원 수준. 판교에서도 주변식당가들 직원식대 올랐다고 식당 사장님이 이번에 식대 올랐다면서요? 물어보고 오른만큼 가격올린것도 있고 뭔 주변에 죄다 남 피빨아 먹으려는것들밖에 없음',
 '이재용 회장님  힘내세요\n대한민국  화이팅\n올바른 사람이  한명이라도 많으면  그것이  승리  가  될것임',
 '군부대 떠나는 강원도 한도시의 업자들이 생각나네요...선을 넘는 인간들은 어디든 있죠.',
 '이런방송 정말  멋진 방송 입니다 국민은 아직도 모르고  왜 지연될까  걱정 만 했죠',
 '대기업을 욕하고 미워하고 끌어내리는게 아니라 대기업을 많이 만들 생각을 해야함',
 '너무 감동 입니당 ~^^  참 잘 함요',
 '정말 맞는 말씀입니다.\n우리나라가 하루빨리 기업하기 좋은 나라가 되어 일자리가 넘쳐나는 나라로 변모하길 바랍니다.',
 '국가에 스파이들이 하는 제일원칙은 국가의 제도를 복잡하게 만들고 관료주의적으로 만드는게 주된 일이라고 어디선가 본듯함..',
 '이런 방송 구독과좋아요는 당연히 필수입니다',
 '인간적인 기업 삼성... 나라가 어려울 때마다 항상 도왔다. 국민들은 이것을 잊으면 안된다.',
 '우리 기업은 적이 아닙니다.그들은 우리나라 발전의 공신입니다.개인의 이익을 위해서 기업을 해하는 행위는 멈추어야 합니다.',
 '그놈의 보상 그놈의 보상...\n기업이 떠나고\n기업이 망하고\n기업이 사라진\n지방 도시를 봐라...\n그게 너희의 미래다....',
 '이재용 회장님 하염없이 작은 일용직들 고덕p3(코로나때) 정직원식당 식권도 구매할수있게 해주시고 이용도 하게  해주셔서 너무나 감사했습니다^^♡사랑합니다',
 '몰랐던 이야기 알수 있게 해주셔서 감사합니다\n감사합니디ㅡ',
 '정말 정확한 지적이십니다.\n이런 목소리가 더 커져  조국의 앞날에 도움이 되었으면 좋겠어요.',
 '양구가 군인들한테 갑질하다가 깡통 찬걸 기억해라',
 '마지막 멘트 넘 슬프고 가슴아픈 현실이네여.기업이 잘되어야 나라가 성장합니다.',
 '구구 절절 동의 합니다.  이런 유투부 방송이 있는것이 자랑스럽습니다.   도대채 언제까지  남이야 어찌되건 내 배만 채우기 급급한 3등 시민의식에서 벗어날찌.  같이 살자는 공생의식이 없이는 서로를 파멸시키는 결과만 갖어올뿐인것을 다 알면서도 말입니다.',
 "반도체뿐이 아니죠 요즘 조선업이 좀 된다니까 거제시장은 한화오션, 삼성중공업한테 상생기금이라고 천억원씩 내라고 했죠. \n나라 전체가 '상생' 이라는 명목으로 기업들 삥뜽어가는거 하루이틀 일이 아닙니다. 제가 기업인이면 이나라에 1원 한 푼 투자 안하죠. \n미국가서 공장지으면 그 기업 이름으로 도로도 내주고 고맙다고 주지사가 찾아와서 인사까지 하고 각종 인허가 단계부터 맞춤형으로 지원해주는데 뭐하러 쌩돈 물어가면서 이나라에 투자하나요?\n그런다고 국민들이 기업들 고마운줄 아는것도아니고.",
 '옳고 맞는 좋은 방송입니다\n\n옳지못한 관행은 효율적으로 \n바꾸어야 합니다\n\n지역의 이기적 단체의 힘은 \n바뀌어야 합니다\n\n그런데 이것 저것 다 나쁘다고\n외국으로 나가면\n또 그것이 옳다고\n주장하는 허황된 못된 말은  하면 \n안됩니다',
 '기업하기 좋은 나라가 되도록, 국민들이 변화하면 좋겠습니다',
 '기업이 성장하여야 나라가 발전 또한 청년들도 장래도 힘,꿈이죠? 파이팅 삼성️',
 '걸핏하면 피해자인 척 손 벌리는 문화부터 바뀌어야 됨\n온갖 혜택을 누리면서 당연한 것처럼 무리하게 요구하는 국민성\n조금이라도 마음에 안들면 무조건 반대하는 국민성\n과연 정상입니까?',
 '이렇게 확실한 사실가지고 정확히 짚어주는 방송은 나라발전에 꼭 필요한 방송이다. 수고했습니다.',
 '우리나라 국민들은 정말 배가 불렀읍니다...반성해야합니다...',
 '삼성은 애국기업이다 우리나라을 먹여살림니다',
 '반도체기업 전세계에서 서로 끌어들이려하는데\n우리나라 관공서는\n뜯어먹을 생각만 하고있으니\n안타깝네요',
 '무식한 정치인 ,무능한 관료, 개념없는 국민  망할 수 밖에 없는 3박자 조건이네요.\n사회 지도자, 국가 지도자가 부재한 대한민국의 미래는 없읍니다.\n국민이 햔명해야 나라 꼴이 바로 설 수 있는데 지식인들은 주둥이를 쳐 닫고 있는데\n그래도 그대가 깃발을 흔들어 주네요.....',
 '항상 언제든 양심없는 자영업자, 소상공인이 문제임',
 '영상 잘 봤습니다. 우리나라 기업들을 응원합니다.',
 '기업이 사내식당을 만드는건 직원 복지를 위해 당연하며 외부 상인들이 참견할 사안이 아닙니다. 기업에서는 상인 및 주민들과 상생하는 차원에서 일부러 사내식당을 안두는거지 마음만 먹으면 당장 사내식당을 둘수 있습니다. 그럼에도 불구하고 상인들은 퀄리티가 낮은 음식을 제공하고 게다가 가격까지 비싸니 기업 노동자들 입장에서는 점심시간, 저녁시간이 두려워진다라고 말할정도로 밖에서 밥먹는게 꺼려집니다. 그래서 결국 직원들의 한탄과 요구로 사내식당을 만드는 겁니다. 그래놓고 상인들은 상권 무너진다, 자기들 생존권이 달려있다라고 그제서야 후회하죠. 그럼 처음부터 질좋은 퀄리티와 가격 올려치기를 하지 말던지요. 기업 노동자들을 무슨 봉으로 봅니까? 제가사는 대구에서도 비슷한일 있었습니다. 대학에서 기숙사를 건립하려하자 주변 원룸업자들의 반대가 심했던 겁니다. 대학은 학생 복지를 위해 기숙사 건립은 당연하지만 원룸 업자들은 기숙사가 건립되면 원룸 공실률이 높으니 자신들의 이익을 위해 반대를 했던겁니다. 대학측이 하는일이고 학생 복지차원인데 주변에서 간섭을 하는게 웃깁니다. 이렇듯 뭐든지 욕심이 과하면 결국 부메랑이 되어 돌아오게 됩니다. 주민들도 기업을 막을게 아니라 문제가 발생하면 어떠한 조치가 이루어진다라는 명확한 답변을 받고 서로 믿고 신뢰해야 지역이 발전하고 기업이 들어옴으로서 일자리가 생겨 지역 경제발전에 보탬이 됩니다.',
 '상인들  진짜 악질이네. 제주도  상인들  바가지 요금을 받아  제주도 안간다\n중국인들이  제주도를 먹어버리려고 하는데 정신차려야 한다.',
 '좋은 영상 잘봤습니다. 대한민국의 기업들이 저렇게 큰 사업을 하고 글로벌 경쟁력을 가지는데 이렇게 연기시키고 반대하고 막지만 말고 다른 국가처럼 정부와 주변 시민들이 도움을 주면 좋겠네요.',
 '이웃이잘되야나도잘됩니다이재용회장님얼마나힘드실까요',
 '국민들이 모두 뭉쳐야 삽니다\n앞길을 멀리보고 살아가야 나라가 살아요\n지역주민들 협조해야 나라도 살아요\n‘있을때 잘해!!‘',
 '말씀아다 정확한 지적. 공감합니다\n앞으로도 대한민국 의 올바른 선택이 무엇인지\n생각할 수 있도록. \n응원합니딘',
 '이런 좋은 내용들이 널리널리 퍼져서 모두가 알수있었으면 합니다  빨리 깨우쳐야하는데ᆢ\n감사함을 알아야하는데ᆢ 안타깝다',
 '옳은 지적이십니다 ,,대한민국  이  국민 스스로  자멸시키는중입니다 ,,',
 '그넘의 시민단체와 선동이 문제',
 '멍청한 국민들은 거시경제도 모르고 장기적 손익비도 전혀 모릅니다.\n오로지 단기적 기분나쁨, 단기적 손실에 사로잡혀서 그것만 봅니다. \n그 결과 그에맞는 단기적기분좋음, 단기적 이익을 준다는 국회의원 ,정치인이 꽉차고있습니다.\n멍청하니까 가만히있으면 죄가아닙니다. 멍청한데 나대면 중죄입니다. 본인과 아들, 딸, 손자 잡아먹는거죠.',
 '구내식당 안가고 밖에 식당을 다녀오면\n오고가는 시간이 전부 점심시간에서 손해보는거임\n근무자 전체로 따지면 기업입장에서는 엄청난 손해임\n[상생]이라는 이유로 선 넘는 갑질은 그만해야 할 듯\n솔직히 저중 1/10만 저녘을 동네가서 먹어도 동네상권은 그야말로 초대박아님?',
 '자영업 활성화 위한 대기업의 결정에 감사 합니다, 우리 모두는 더불어,고통을\n감내하고 같이 성장하는 한국민이 되어야 합니다 우리는 한민족 입니다 감사 합니다',
 '지금 자기 코앞에 놓인 이득만 바라보느라 미래의 윤택한삶을 내다 던져버리는 안성과 용인 참 멋집니다 :)\n대기업때문에 이 나라가 먹고살 수 있는데, 가면갈수록 푸대접이고, 결국 기업이 이 나라를 떠나게되면\n더 힘들어지는건 당신들과 우리 다음 세대들입니다. 아 다음세대는 없을거니까 내 앞만 보는걸수도 있겠네요\n각성하시기 바랍니다.',
 '언론은 언급도 하지 않는아주 공정하고 애국계몽적인 내용입니다.  감사합니다',
 '누군가 해야할말을 해주셔서 감사 합니다\n서로 협조하고 살아가야 나라가 부흥합니다',
 '이재용회장님!늘존경합니다',
 '대부분 시민단체가  주도',
 '우연히 알고리즘 타고 들어와서 봤는데, 이런 예리한 시각을 가지신 젊은 분이 있다는것 만으로 대한민국은 희망이 보입니다. 저는 미국사는 사람으로서 한국 삼성이나 SK 보면 대단한 생각이 듭니다. 정권이 바뀔때 마다 경제인들을 쥐고 흔들려는 무식한 정치인들 사이에서  살아 남는다는게 얼마나 힘들까요? 예전엔 생각도 못했던 한국 자동차들이 미국거리를 누비는걸 보면서 마음 한구석이 뿌듯하고 자랑스럽기까지 합니다. 구독 눌렀어요. 앞으로도 이런 좋은 영상 부탁합니다.',
 '진짜 이나라는 답이 없다 그래도 굴러가는거보면    기적이다',
 '정말 좋은정보\n감사하고요~\n많이많이 올려주세요~\n️️️️️',
 "현대제철이 들어가기로 한 루이지애나는 산업용전기료가 우리 절반입니다. 부지는 주정부가 무상제공, 심지어 공장 내부 외부 인프라를 주정부예산으로 건설해줍니다. 파격적인 세금감면과 금융지원은 기본입니다. 뿐만 아니라 주변대학들에게 관련 산업 인재 양성을 의뢰한다고 합니다. 거기에 생산대기업이 들어오면 미국주정부는 '~의 날' 이라고 기념하면서 그 대기업을 칭송하고 무엇을 더 지원해줄까 위원회까지 만든다고 하죠?",
 '동네 상인들이 구내식당 질투한다는 소리가 진짜 있는 소리면 거기는 진짜 근시안적이고 자기생각밖에 안한다는 반증이죠 \n일하면서 안에서 먹는 점심 한끼 좋아봐야 얼마나 좋겠습니까. 결국 거기서 먹는 밥은 일 하려고 먹는 밥이기 때문에 그냥 평범하면 됩니다 \n근데 그 사람들 평일에 퇴근하고서, 주말에 밥 어디서 먹습니까? 집에서도 먹지만 결국 다 그 상권으로 그 지역으로 갈텐데요 \n\n맨날 인프라 깔아달라, 좋은 일자리 얘기하는데 막상 저렇게 뭐 하려고하면 지역주민 반발이 심하다는건 \n결국 텃세 아닙니까 돈 내놔라 이거죠',
 '민주주의 단점: 그 주민이 멍청하고 근시안적이면 \n정말로 답이 없음',
 '제대로 보셨네요.  \n현실을 자세히 보면 삼성은 매우 긍정적인 영향력을 가지고 있습니다.',
 '안성은 에버랜드도 들어오는거 반대해서 용인으로 가게했고, 평택 안성간 철도도 안성쪽에서 끊어버린 동네에요. 그냥 육지판 신안이에요. 우물안에선 지들이 맘대로 할 수 있는데, 외부에서 잘난 기업이나 사람들 들어오면 지들 맘대로 못할까봐 발전을 거부하는 동네',
 '글로벌 시대에 삼성 같은 기업은 더 이상 한국만의 기업이 아니다. 한국이 좋은 조건을 제공하지 못하면, 삼성은 언제든지 해외로 나가버릴 수 있다. 그런데도 마치 삼성이 한국에 당연히 남아있을 거라고 착각하며, 기업에 불리한 정책이나 환경을 만드는 것은 어리석은 짓이다. 이제는 그런 바보 같은 태도는 그만두어야 한다. 특히 정치인...',
 '세종대 졸업생인데유.. 지방에서 올라오는 학생이 많아서 학교에서 기숙사 지었는디.. 동네 임대 상권 무너진다고 주민들 현수막 붙이고 난리났던 때가 떠오르네유.. 참..',
 '상인들 참 어이없다 구멍 가게에서 그 많은 인원을 소화도 못시킬 뿐만아니라 왜 거기서 점심을 팔아줘야 하는데 국내 식당이 있는데 지나친 말도 안되는 욕심이지 직원이 몇명인데 몇십명도 못 받는 가게를 운영하면서 식당으로 오라는 건지 만명 먹을 설비하고 요구하라 10,000명 받을 식당 있냐고? 상인들 욕심이 과하다. 저녁에 회식이나 가끔 회식 때나 가는 거만해도 상권이 사는데 니들이 회사에 투자했어? 왜 강제로 말도 안되는 것 요구하는데? 큰 회사 운영 사업권을 방해하고 집단들 대거 구속해야 한다. 업무 방해다. 수용인원을 할 수 있는지 분수애 맞게 말해야지 갖추지도 않고 억지 부리는 몰상식한 짓이다.  상인들 갑질이 지니치다. 대기업 건드리지 마라 회식 때 팔아줘도 돈이 된다.인구수가 많잖아 주변에 공장이 있어서 돈 많이 벌면서 갑질 부리면 안되는 것이다 저런 상인들 구속 좀 하세요. 나라 발전에 방해자입니다.옆에  공장만 있어도 상권이 산다.',
 '공산당이냐?\n구내식당을 니들이 무슨권리로 반대하냐?',
 '지방자치제 없애야함. 좁은 땅떵어리 진짜. 예산도 부족한데 ... 몇백억 배(여객선) 사놓고 쓰지도 않고... 쓸데 없는데 꽃심고 도로 깔고 포장하고 공원조성하고... 강가에 벤치 나무 데크 만들고.... 진짜 필요한데 써야지... 너무너무 낭비가 많음.',
 '황금낳는 거위 데려다 줬더니 조금만 욕심에 눈이멀어 배를 가르는 어리석은 주민들. 절대 그들의 요구를 들어주지 마라. 떼쓰는 놈들은 법대로 처리하라.',
 '잘보면 우리나라 소상공인 진짜 날강도 도둑놈들임',
 '지방자치제가 문제다. 표는 주민에게 있으니 기업들 의견을 안듣는거지. 돈은 기업이 벌어다 주는데 투표권은 지역주민에게 있는 지방자치제가 제일 문제.  역사적으로 지방에 권한 많이 주면 토호들만 배불리는거다. 시스템을 투명하게 한 중앙집권방식이 제일 좋은거임.',
 '삼성  이재용  회장님 !  최고 !  존경합니다  .',
 '구구절절 옳은 말씀입니다.기업이 정리할 수 없는, 동네\n식당 개개인들을 \n정리 해줄 수 있는 올바른 정부의 힘이 필요 하다고 봅니다.',
 '와 정말 정부에서 기업 많이 도와주세요\n그래야 일자리도 많이 생기죠 ㅠㅠ',
 '삼성 대한민국의  미래입니다   국민여러분 기업은  우리후손들에게  희망입니다',
 '솔직한 의견 속 시원히~~~ 잘 봤습니다. 에스오디님이 말씀하시는게 처음부터 끝까지 다 맞습니다. 격한 동의!',
 '근래에 본 영상중에 가장 좋은 내용이네요. 복잡한 사안을 알기쉽게 설명해주셔서 감사합니다. 쉽게 설명하는 게 더 어렵죠',
 '제정신인가?? 소상공인 보호할려고 많은 혜택을 주다보니 호의가 계속되면 권리인줄 안다 우리 원래 안그랬음 예전에 지방자치단체가 자기네동네에 공장지으라고 길내주고 세금면제해주고 했음 제발 정신차리세요 기업들떠나면 우리도 일자리잃고 그럼 상인도 많이 망하고 결국 집값도 떨어져요',
 '진짜 저 국민들은 정신차려야합니다.. 나라 위해 일 하는 기업을 이런식으로 대우하면 대한민국은 중국한테 금방 먹힙니다..',
 '대한민국 리더가 잘 세워져야할텐데요',
 '나라가있고내가있지요죽어서도응원할께요',
 '안성은 뭘해도 안되는 동네임 ㅋㅋㅋㅋㅋ진짜 꽉막힌 동네 ..옛날에 삼성에서 애버랜드 안성에다가 만들겠다고 대림동산달라고 했는데 그거 죽어라 반대시위하고 거부해서\n용인으로 넘어가서 용인은 엄청성장함 ㅋㅋ근데 안성은 아직도 20년전이나 지금이나 발전된게 없음 그나마 공도쪽이나 신도시처럼 아파트나 줄기차게 지어졌지 바뀐게없음 인구계속 줄어듬',
 '삼성 국유화 하자는 나라..',
 '정말 한마디로 딱 맞는말만 한다',
 'ㅋㅋㅋ 원전 도 우리동내만 안지어면된다..',
 '요즘 진짜..기본소득 같은 거 주장하며 노력없이 무임승차하려는 사람들과\n국익이 아닌 표심에 따라서만 움직이는 정치인들 보면 역겹고 이 나라의 미래가 암울한 것 같아 안타까운 마음입니다..',
 '너무 답답한 심정으로 봤습니다, 이걸 166만 조회수나 나올 수 있게 만들어준 SOD님에게 감사드립니다. 정말 영향력이 큰 언론사로 발전하시길 기원합니다',
 '돈 내놔라 이러는 거지? 이참에 한몫 잡아보자 이거 아녀?',
 '멋진 방송 극 지지합니다. 국민들 생각을 대변해서 말해주니 시원하네요….',
 '뒤에서 선동하고, 이득 보려는  인간이 많다.. 어느 정당이라고 말하지 않겠지만..',
 '나라도 진짜 외국으로 도망감.\n이재용이 진짜 애국자임',
 '참.. 어이가 없는 인간들임..  기업이 봉이냐? 망해봐야..  알지..',
 '지역상인들 정말 이기적이네요 ㅡㅡ 이러다 기업들 다떠나겠다',
 '와..이영상 좋네요 구독합니다',
 '이건희 회장 자서전에서 우리나라는 공장을 지으려면 몇천가지 승인과 평당 가격을 매긴다네요 민원부터 행정까지 회의감든다고... 영국이나 미국은 그냥 들어오게 인프라시설 다 구축해준다고...',
 '진짜 말도 안되는 소리....\n일 좀 하게 해 줘라\n깡패돌도 아니고....\n정말 우리 기업들 불쌍하다',
 '그냥 셔터 내리고 "일할테니까 밥만 주세요 " 이소리 나올때 까지 망해봐야 합니다.',
 '우리나라는 열심히  노력해서돈버는자와 기업하는자들의 고혈을빨아먹고사는 인간들이 누굴까요. 일은적게하고 복지와임금은올리고 나라에서 꽁돈받아먹는자들만 특혜를받는나라. 정치가 미쳤습니다. 우리나라는다좋은데  나쁜정치인들이 이나라를망치고있습니다.꽁돈좋아하다가 어어 하다가 공산주의국가되는것 시간문제입니다',
 '한국 반도체 산업 무력화 시키려는 특히 삼성, 국내 사회단체 정치와 국외 조직화돤 세력이 있어요',
 '군대에 있을때 위수지역 풀어버리니까\n현수막 걸렸던 생각이 납니다\n그 이후 pc방 요금 ,숙박요금이 내려가더군요',
 '대한민국 대기업 인내력이 대단합니다.',
 '정말 맞는 말이네요.  정말 기가막힌 한국이다.',
 '모든 우트버가 보고 배으ㅟ야할 내용입니다 지역민들 자기 잇속만 챙기지말고  다같이 상생하는 대한민국을 만들어야합니다',
 '당신 말이 진실입니다 나라를 위해서 많은 조언바랍니다  감사합니다',
 '주민들 정신줄 꽉 잡으셔야겠어요 .',
 '정상적인 대기업들 진정한 애국자들 이시지요',
 '진짜 우리나라 사람들 국민 평균 수준이 얼마나 낮은지 보여주는 사례임. 이런 국민들이 투표권을 다 가져야한다니 통탄스러울따름 ㅠㅠ 민주주의국가의 단점임. 예로부터 지식인,깨어있는 사람들 전체 인구의 20%가 안된다는게 팩트고 그 20%가 나머지 80%를 설득해야 하는 시스템임',
 '노조는 강성이고 국가는 규제투성이고,\n일반인은,이기적으로 소소한것으로 소송하고 대기업은,한국에서 기업하기 너무힘들것같아요. 국민성과 풍토가 바뀌지않는한 세계적기업되기가 정말 어렵겠습니다',
 '소중한  방송감합니다^^\n\n이기적인 인간들 이지요 \n시민이나\n정치인이나!',
 '원삼면 원주민들 땅부자 다 되었습니다.\n그곳가면 원주민들 하이닉스에 돈 뜯어먹으려고 탐욕의 눈으로 보고있습니다.\n  강력한정부에서 대기업 지원해줘야해요.\n안성시 , 여주시 등 주변에서 숟가락 얹히고있어요.\n  나라를 위해서 반드시 빠른 진행을 기원합니다.',
 '이번 영상 정말 내용 좋네요 !',
 '정말 똑 부러지게  현실직시 뼈져리게 느끼게 하는 말씀 입니다  \n제발  우리 정치인들 정신좀 차렸음 좋겠습니다  재벌기업을 두둔하자는 말이 아니라 우리 국민들이 입는 이익을 왜 생각못하는지 한심 하고  아둔한 일부 시민과 정치인들  빨리 정신차려야합니다',
 '진짜해도 해도 너무하네요 ㅠㅜ\n우리나라 인구붕괴에 대기업들마저 무너지면, 끔찍한 미래가 있겠네요 ',
 '나같애도 해외로 나가겠다. \n나라에 거지같은 규제하며\n그저 피빨아먹으려는 버러지들뿐인데\n조빤다고 국내서 사업하냐 ㅡ\n없어봐야 궁한줄 안다 ㅡ\n없어봐야 정신차린다ㅡ\n기업들 다 해외로 나가 잘벌어먹고 승승장구하시길  진심으로 기원한다.',
 '수고해 주셔서 고맙습니다.',
 '1:57 근데 미국은 불체자 아닌 사람도 불체자로 장갑차 가지고와서 잡아들임....',
 '구독했어요!\n좋은 내용 계속 부탁합니다!',
 '기업을 망하게 하는 해커,브로커들부터 잡아야 할듯하네요..\n기업이 살아야 합니다.\n대기업들 응원합니다.',
 '이 영상 최상위 노출 되어야 합니다!!!!\n우리나라 미래 정말 걱정됩니다.',
 '삼성 이재용회장님 \n대구로 오십시요..\n시민들이 최대한 협조 할겁니다.\n시장 구청장 말 안들으면 시민들이 쫒아 내서라고 ㅎㅎㅎ',
 '현장에서 밖에 나가는 거리가 얼만데 나가서 먹냐ㅋ 안먹고 말지요ㅋㅋ',
 'ㅋㅋㅋ  보상이라는 달콤한 사탕에 맛들인 사람들이 너무 많고, 사탕을 잘 뜯어내는 정치인 뽑는 나라.',
 '국가 기업 못 뜯어먹어서 난리인 대한민국의 현실...',
 '대기업 그냥저냥 만들어 지는거 아니라 생각합니다 수십년간 노력하고 세계경쟁사들과 경쟁력 있는 기업을 만들기위해 근로자 분들과 오너와의 각자가 맏은 일에 충실했던 결과라 생각합니다 기업에 미래를 위해 대한민국 후손을 위해 우리가정마다 행복한 삶을 위해 더 노력하고 최선을...화이팅 입니다 \n이기적변명 남탓 헐뜯으며 가르치려 하지말고 배웁시다 이제 시작일뿐 화이팅 초심으로 미래를 위해 장단점을 개선하고 존중 과 배려 올바른 인성으로 \n내가누군데 내가제일잘났어 가  아니라\n삶에 동반자 가족입니다',
 '동의합니다. 제발 우리나라 정신 차려야한다 눈 앞에 이익을 위해 내 모든걸 갈아넣고 있는줄도 모루고 달리는중이지',
 '정신차려야한다 다 조진다.기업업ㅎ으면거지된다 나라 다!',
 '심성 멋집니다\n주변 개인 음식 상가를 배려 하셨군요',
 '우리나라삼성복지는내가삼성출신이라잘~~~안다진짜서계최고라할수있고삼성지역주민들에게도복지가은근펼치고있습니다그것은이병철고회장님의모토이기도합니다',
 '이미 공산당이거든\n선언만안했지\n대한중국 공산인민민주주의 공화국',
 '이방송이 널리퍼져야할거같아요',
 '지역이 망하는 이유죠. 대규모 공장 지어도 이거줘 저거줘 내놔 마인드. 공장이 지어지면 사람이 모여들고 자연스레 인프라 구축되고 수도 집중화도 해소되고. 이런 긍정적 미래보단 근거없는 선동에 휘말리는 낮은 국민성, 미래를 위한 비전보단 당장의 뜯어먹을 수 있는 작은 돈이 훨씬 중요한 이기심. 그러면서 젊은이들 떠난다, 오지 않는다 불평불만. 올 이유를 만들어준다고 해도 거부함.',
 '노란봉투법 으로 모든 기업 들 멈추고 문 닫아야 됩니다.',
 '해당 지자체의 장들\n소속 정당을 살펴보시고\n표로 심판합시다',
 '너무나 정확하고 옳은 말씀입니다',
 '멍청항 국민들 때문에 공산당이 판치는거  자업자득입니다',
 '이제야 봤습니다 좋은방송\n감사합니다 구독 좋아요 꾸~욱\n하고 갑니다',
 '정말 나라를 망치는것들이 다 자리에 앉아 있어니',
 '삼성 잡아끌어내리는 행위가 어떤 결과를 갖고올지 예상하고 정신차리자.',
 '이런 이유로 한쿡의 경쟁력과 혁신은 요원함~   그래서 국가 전략 산업만틈은 법적 예외나 일부 독재가 필요함!',
 '우리나라는 기회만있으면 기업이나 부자들한테 돈뜯어 먹을려는 더러운 인간들이 너무 많아...',
 '저런 일이 평택에서도 있었어요. 그때 삼성전자에 지역 업체 안쓴다고 단체로 시위했었죠~ㅋㅋ\n\n지금 시위하는 사람들이 다 용인 사람이라고 볼 수 없는게 함바식당 사장들이 반도체 공장 신축현장 따라다니면서 식당을 만들어요. 언론에 반도체 공장 신축현장 계획 발표되면 미리 와서 자리를 잡은거죠. 건물을 짓던 임대하던 어떻게든 자리를 잡습니다.\n\n화성 동탄, 평택, 아산 탕정, 청주, 이천 등에서 삼성전자와 SK하이닉스의 FAB신축현장 따라다니며 식당 열어서 돈 많이들 버셨거든요. 현장 하나 잘 잡으면 건물한채 올릴 수 있는 돈을 법니다. 아파트 현장 따위와는 비교가 안되요. 왜냐하면 FAB공사는 조단위의 돈이 투입되는 메가 프로젝트이기 때문입니다. \n\n우선 작업 투입 인원이 최소 수천명에서 몇 만명 단위라 규모가 큽니다. 반도체 공장 신축 현장에 참여하는 협력사가 수십개인데 한 업체당 최소 100명은 잡아요. 특히 설비나 전기공종의 회사 근로자들이 인원이 많습니다. 계약금액도 크고 투입인원도 몇백~천명 단위까지 올라가요. 그런 회사(삼성, SK 협력사) 하나 잘 잡으면 팔자를 고치는거죠. \n\n이게 순서가 있습니다. 처음에는 토목으로 시작하는데 장비만 많고 인원은 별로 없습니다. 토목이 끝나면 건물을 구조를 형성하는 골조(기둥, 보, 바닥 형성 등)를 지으며 인원이 늘어나기 시작해 전기, 설비 공정이 투입되면서 절정이 됩니다. 특히 설비나 전기공종 인원이 많고 공사 기간도 1년~1년 반정도 잡습니다.(골조 완성 후 기준) 이 설비, 전기 공종 협력사들을 잡기 위한 함바식당들 경쟁도 치열합니다~ 이 식당들도 한 곳이 아니라 몇십 개의 식당이 경쟁을 해요. 당연한 것이 인원이 저렇게 많은데 너도 나도 달려드는게 당연합니다. 근데 영양사가 없어요. 그래서 여름에 사설 함바식당에서 밥먹는 근로자들이 배탈이 나는 일이 종종 있습니다^^; 물론 삼성이나 SK에서 현장마다 자체적으로 구내식당을 운영합니다. 그런 곳은 영양사도 있고 나름 관리를 하는데 만명 단위까지는 감당을 못하더군요. 식대도 사설함바식당보다 훨씬 저렴합니다. 사설함바 사장들은 싫어하겠지만 개인적으로는 삼성이나 SK에서 현장내 구내식당을 더 늘려주었으면 하는 바램이 있네요~ 근로자들도 다들 비슷한 심정일겁니다. 안에서 밥먹고 근처에서 쉬는게 낫지 밖에까지 나가서 밥먹고 다시 들어오는 것도 고역이거든요~ 근데 사람 입맛이 다 달라서 구내식당밥 맛없다고 외부나가서 먹는 사람도 많긴 해요^^\n\n저렇게 공사가 진행되면서 근처 부동산도 들썩여요. \n일단 월세가 급등합니다. 반도체 현장 전문 팀 단위로 움직이는 숙식 노가다 형님들이 몇천~몇만명이 모이니 당연히 월세가 뜁니다. 평택의 경우에는 월세가 비싸서 회사에서 월세 지원금을 지급하니 그 액수만큼 집주인들이 집값을 올려버리는 횡포를 부렸었죠~~ㅎㅎ 이렇게 인원이 많으니 먹고 마시는 공간도 필요하겠죠? 현장 근처 편의점은 매출액이 전국 1위를 찍고 유흥업도 성황을 이룹니다. 단, 공사 종료할 때 까지만 이러한 호황이 유지됩니다. \n\n이처럼 지역경제 활성화 효과는 나름 있습니다. 근데 여러 부작용이 있죠~ 여기저기서 몰려와서 뜯어먹으려고 난리부르스 더군요. 나중에는 지자체도 숟가락을 내밀어요. 얼마전에 거제시에서 삼성중공업과 한화오션(구 대우조선해양)에게 1천억원의 상생발전기금을 내라고 논란이 있던거 다들 잘 아실겁니다.ㅎㅎ\n\n이런 거 보면 우리나라는 기업 하기 참 힘든 나라라는 생각이 들더군요.',
 '상속세가 60% 가 넘어서 세대 걸치면  3대가 이루었던 평생 가업을 뺏겨야하고\n국회의원들은 기업을 악마화하고 뺏어서 국민들한테 나눠준다 그러고\n기업을 부수면 본인들한테 10만원이라도 떨어지는줄 알고 좋아하는 국민들이 넘치는 나라에서\n어떻게 기업을 유지하고 발전시킬 수 있겠냐 에휴...',
 '바보같은 정치인들 한심안 국가 정책이 경제가 붕괴한다. 정신을 차려라. 기업이 살아야 국가 경제가 살아난다.',
 '그래서 지도자를 잘 뽑아야 합니다\n힘들게 이룩한 대한민국을 \n수십년간 누구당 지도자로 인해 나라가 망하게  생겼어요\n이번대선은 진정한 애국자이신 분을 뽑아서 경제를 부흥시켜야 합니디ㅣ',
 '천만번 지당한 말입니다\n이기적인 지역민들과 무지한 정치인들 때문에 한국에서 기업하기가 너무 힘듭니다.',
 '자사주 다처분하라는 상법개정은 경영권 방어를 못하게해서 우리나라기업 외국에 넘기는 법입니다.',
 '그런 우리 기업을 잘 할 수 있도록 배려하는 것이 잘 사는 길입니다',
 '진짜 시민의식 너무 한심해서 한숨만 나오네요 솔직히 자기 지역에 매일 만명이 왔다갔다하면 구내식당도 자주 쓰겠지만 가끔씩 나와서도 먹고 할 텐데 그런 낙수효과를 보고 장기적으로 내다봐야지... 설사 자기네 식당에 안 오면 스스로 경쟁력이 떨어지는 거니까 고객 유치를 하려고 더 노력을 해야지 노력은 하기 싫고 낙수만 누리고 싶고... 에휴 이래놓고 젊은이들 보고는 왜 노력 안 하냐고 그러고 있겠죠',
 '기업이 살아야 사람이 살고사람이 살아야 굶지도 못하지.\n사람의 욕심은 끝이없다 진짜 지역주민들이',
 'ㅋㅋㅋ 돈을 내서라도 먹을만한걸 내놓을 생각은 안하는 지역상인들. 뭔짓거리를 하는지 조만간 깨닫게 될 날이 올꺼다.',
 '항상 적은 내부에 있는거여... 그리고 침략은 내부에서 부터 시작되는 거고...',
 '하여튼 나라를 생각한게아니라  자기 개인만  생각함  못된놈들\n\n이런 상황을 알려주신분에게   감사드립니다',
 '한국에 공장을 짓게해야합니다.자기나라에서 공장짓는데 왜어렵게할까요.',
 '영원한건 하나도 없다.\n진짜 기업들이 각자도생 , 차라리 미국이나 해외가는게 더 대접 잘 받을듯..\n상생이 뭐냐. 칼만 안들었지 양아치에 징징거림.. 어휴ㅠㅠ....',
 '동네주민들  욕심이 목아지까지 찼구먼...',
 '우리나라 기업하기 어려운 문제 누구한사람의 잘못이아니라 전국민이 정신똑바로 차리고 개인이기주의가아닌 전체나라를 생각하는 의식개혁이필요한시기인것같습니다\n남탓아닌 내스스로 개혁하고 변화하지않는다면 우리나라는 모든분야에서 뒤떨어지고말거에요.  더늦기전에. 모두 정신차려보아요 모두좋은것만 바라고 어렵고힘든일을 회피한다면  스스로 망하는지름길임을 잊지맙시다',
 '이야 진짜 조선답다. \n대한민국은 조선으로 되야한다.\n너무 큰 복을 받았다.\n재매이햄께서 우릴 베네수엘라로 이끌것.',
 '수원시에 지네한테 버스안줬자고 삼전한테 랄지 떨었던거 보면 진짜 ㅋㅋㅋ 삼전이 탈반도해도 할말없음',
 '대한민국은 삼성, 현대\n덕에 먹고 산다고해도\n과언이 아니다..',
 '청주로 오세요. 청주에는 하이닉스가 있지만 삼성이 와도 주민들 모두 대환영이에요. 무조건 청주로! KTX가 지나가는 청주로!  국제공항이 있는 청주로!',
 '어후.. 화가 너무 많이 나서 머리가 아파요',
 '좋은영상 감사합니다.',
 '국민들한테 이런쓴소리 해주는 분들이 점점 많아져야합니다.     구구절절 옳은 소리만 하시네요 제가 하고싶은 소리이기도합니다',
 '진짜 개짜침. 나라를 먹여살리는 대기업을 우대해주지 못할망정 욕이나해대는 국민수준이 진짜 처참합니다..... 언제쯤 철이들지원...ㅉ',
 '큰일났다 정치인들이 욕을 먹어도 버팀목이 되어 열어 줘야 하는데  창조적인 역량을 국회가 열어줘야 하는데  어쩌다가 이나라가 이지경으로 가고 있는지  막막합니다.',
 '삼성 세계적인 초일류 기업 한국의 국민으로서 자부심을 느낍니다',
 '진짜 제가 생각하던 내용 그대로의 유투브네요 이런 구독자 많은 분들이 널리 퍼트렸으면 좋겠어요.',
 '귀한 영상',
 '좋은 기사 감사요',
 '대한민국은 환경보호와 자연보호가 중요해서 굶어 죽어도 자연으로 돌아가기를 원하는 사람들이 많다.',
 '이재용회장님사랑합니다',
 '정말 감사합니다^^',
 '구내식당 밥 너무너무 맛남~',
 '제정신 아닌이상 일부러 그러는거임..\n그럼 왜 일부러 그럴까?\n잘 생각해보면 나라에 도움되고 나라를 지키는 것들을 줄이고 없애는 세력들이 있음..\n왜그럴까? 왜? ㅎㅎ',
 '이렇게 해도 머만 하면 파업 ㅋㅋ',
 '대한민국의 집단 이기주의는 끝이없습니다.\n대기업이 들어와서 집값 오르고 일자리 많이 생기고 얼마나 좋습니까 !  욕심은 끝없고 정부가 나서서 강제로 해야 한다고 생각합니다.',
 ' 절대공감',
 '한국의 정치인들의 두뇌에 우동사리 뭉치가 들어있어\n대한민국의 미래보다는 자신들의 국회의원 유지~~ 자신 소속당의 권력유지를 위한 \n노조 이용한다. 기업들이 살아나야 미래가 있는데 우리 정치인들은 코앞 코딱지 떼어먹기 바쁘니 한심 하다.\n특히 노란 봉투법 같은 악법은 노동자들의 신성한 노동의 댓가를 떠나 꽁돈이나 바라는 노동자로 변하게 만들것이고\n기업은 망하고 국가는 빚을 내여 배급하여 사회주의로 만들어 베네수엘라와 같은 독재국가를 만들게 될것이다.',
 'ㅋㅋㅋ 다 망해봐야 정신차림, 세금이 어디서 많이 나오며, 나라발전이 어떻게 되는지 아무 관심도없는사람들 같네요~ ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ',
 '옳소!!!',
 '아니 인원이 많으면 사내 식당 당연한거 아닌가 ?? 우리나라는 기업하긴 최악이야   너무 걸뱅이들이 많아',
 '삼성이 살아야 나라가 산다',
 '이런 큰 사업보면 숟가락 꽂으려는 정치인과 사업제동장치로싸 한입 하려는 시민단체 들이 가장 큰 문제겠지만, 결국 시민들도 문제임\n안전, 환경 생각해서 감사하는 것도 좋지만 그거 패스 했으면 일단 제발 만들고 나서 숟가락을 꼽던 말던 했으면\n숟가락 꼽는거야 그렇다 치는데 저 질질 끌리는동안 갈려나가는 직원들, 협력사들 몇명의 피를 말려 죽이는 것인지 좀 인지 했으면 함',
 '옳소. 지당한말씀  정보 말 동감합니다. 위대한 자유대한민국 부강하기위해서는. 대기업들과 함께해야된다!!!🇰🇷🇰🇷🇰🇷🇰🇷',
 '진짜진짜 맞는말이오',
 '이재용회장님박정희대통령과같이정말로애국자이십니다내가회장님입장이라도외국으로나가지여기안있을건데너무나감사하고정말넓으신분입니다대한민국의은인같은존재십니다건강하십시요우리나라늘위해서라도개인의몸이아니십니다부디~~',
 '삼성과 함께 미국으로 진출하는 국내 기업들에 주목해야--특히 의료 로봇 기업 고영',
 '예전에 신고리 원자력 발전소 공사할때였습니다. 현대 SK 컨소시엄 이었는데, 주민들 민원으로 현장내 함바도 결국 못지었습니다. 어짜피.그사람들 저녁엔 근처에서 밥먹는데 말이죠. 점심 식사마저 마음대로 못하게 하더군요. 이러면서 기업이 왜 해외로 나가냐고 노동자의 권리 권리 하면서 이런건 온데간데 없습니다.',
 'ㅎㅎㅎ 국민들이 기업 발목잡았네요.',
 '내이익만 추구하다 보면 나라의 끝이 보인다. 내가먼저, 내고장이 먼저, 국가의 장래는 후순위로 밀리는 나라- 한국- 국제경쟁력이 있을까 의심이 든다.',
 '괜찮아요. 우리나라는 무적의 아파트가 있고, 주 4일제 한다잖아요.. 선배세대들이 만들어낸 탄력받은 경제성장이 그냥 공짜로 나오는줄 알고 권리만 외쳐대는 사람들이 많은가 봅니다. 손가락만 빨아먹으면서 그 좋아하는 아파트에서 실컷 살아봐야 정신차리죠',
 '대단히 지당하신 말씀입니다!',
 '기업의 이기심은 정당하다는 말씀 백퍼 공감합니다.\n한국은 늦었습니다. 삼성이 살려면 나스닥에 상장해야 합니다.\n한국이 굶어봐야 정신 차리죠.',
 '옛날부터 썩어가고 있는 상인들의 모습은 지금 현재도 여전하다...',
 '고거      쌤통이다ㆍ\n대기업이         적이냐고\n왜들     그러냐고\n세금       무지하게       내고    잇는\n대기업을        괴롭히냐',
 '참 훌륭한 말씀이네요. 우리의 생명줄인 고용에 발목을 잡아도 지지를 받는 나라가 대한민국이라는 사실이 부끄러워요. \n대기업의 부정적인 면은 동감하지만 더 큰 틀에서는 조만간 후회할 날이 올 것  같아요.',
 '다 전북 새만금으로 보내라 무한한 땅과 무한한 재섕에너지와 아름다운 자연이 일할 맛날거다',
 '이 재용 회장님, \n힘들게 한국에서 함량 미달 정치꾼들과 지자체 철밥통들에게 시달리지 마시고 미국으로 가셔서 마음 편하게 일류 삼성 만들어 주세요!!  응원합니다!!',
 '옳은 지적에 공감합니다. 생산 기반이 한국에서 없어지고 손가락을 빨아봐야 후회하고 시간을 되돌리자고 할건지, 지금의 이기적인 마음이 후대의 배고픔으로 물려주게되는 일이 없어야합니다.  참 걱정스럽군요.',
 '삼성 외국으로 가라 민노총종업원들 실업자 만들어라..',
 '정치가 가장 문제지요. 긍정적인 부분과 희망으로 가야 할 길을 갈등을 부추기고, 훼방만 놓고 권력투쟁만 열을 올리면서 국민을 이간시키면 결국 그 끝은 남미의 나라들이 결과를 이미 보여주고 있다고 생각됩니다. 방법이 없어요.  끝을 보아야 다른 길을 찾을겁니다. 에혀!',
 '마지막 말이 공감가네 ㅇㅇ 발목잡는 어떤애들이 벌써부터 이상한 법을 두달만에 다 통과시킴',
 '이게 마치 눈앞에 이득만 쫓다가 다같이 망하는거네 황금알 낳는 거위의 배를 가르는 격... 저런 기업 유치 해서 지방내수 돌아가고 지방세 걷는건 생각안하고..',
 '관광지 바가지 요금만 봐도 답이 나오죠.\n바가지 요금 때문에 외국으로 나간다니까 이제는 자기들 죽는다고 우는 소리하잖아요.',
 '지원안해주면 해외로 나가야 정신차리지',
 '삼성ㄷ이 본사를 미국으로 옮겨야 정신바짝차리지',
 '래퍼곡선...좋은거 배우고갑니다\n좋은 공부 하시는걸 느껴요\n다음 영상 기대 됩니다!!',
 '진짜 우리나라는 정치인들 싸움과 강성귀족노조때문에 몰락하고 있다.',
 '현실을 속시원하게 알려주는 좋은 영상입니다. \n국민성인건지 미꾸라지가 많은건지 모르겠지만 일상생활 곳곳에 남의 돈 뜯어먹으려는 사람들이 도사리고 있습니다.\n각종 환경단체, 시민단체의 최초의 목적의식, 순수성은 사라지고 눈가리고 돈 뜯어내는 스킬만 업그레이드 되고 있고\n지자체,  정치인들은 그들의 암묵적 수장 역할을 마다하지 않고 있는 현실입니다.\n안타깝지만 지금이 대한민국의 가장 찬란한 순간이라는 생각을 지우기 힘드네요',
 '그렇게 이기적으로 하면 다 같이 못살게 되는건데...심지어 우리나라는 가진 천연자원도 없는데...걱정입니다. ㅠ',
 '구내식당을 지역상권 운운하며 반대하다니..상권이 얼마나 발달했기에 직원들 밥을 모두 책임지나..\n직원들 회식은 생각안하는 멍청한태도가 아닌가..당연히 공장인근 식당들은 직원들이 이용하는 돈으로 생활이 가능할정도일텐데..삼성공장 한번 지으면 규모는 상상도 못할텐데..어디 쪼끄만 공장 생각하시나..너무 이기적이다',
 '삼성 평택캠퍼스 지을때\n기숙사 못짓게 해서\n직원들 외부 원룸 얻어서\n비싸게 월세 내고 삽니다.\n지방에서 오는 직원들\n첨에 월세지원하니깐 월세 폭등시키고 이제 신입사원 월세 지원 없습니다 ㅠ\n울 아들도 평택 월세 얻어\n살아요\n그놈의 주민들은 상생이 뭔지\n모르네요',
 '말해 뭐합니까? 국민성이 그냥 이게 한계인거죠. 제가 늘상 말하지만 바로 지금이 우리 민족 최고의 전성기이자 마지막 전성기가 될 겁니다.',
 '이게 지역 극단적 이기주의랑 노란봉투법 통과되고 더욱 가속화 되는중',
 '이런건 정치의 문제임 시스템의 문제고.... 사람은 세계 어딜가도 지 생각만함 그걸 견제해주는게 법과 체계인데 정치후진국에 서로 싸우느라 모든게 많이 늦고 후졌음 애초에 존나 작은 땅덩어리에 뭔 지역감정같은 소리나 하고 앉았겠음 그걸 하나 통합 못하는게 우리나라 정치다 이말임',
 '대기업은 지역 경제도 도와주고 정부는 필요시 대기업 도와주고~! 근데, 국회가 저따위니... 답이 없다가 정답!',
 '돈달라는 소리입니다',
 '이와중에 노봉법 ㅋㅋ 기업이 왜 남아있어야 하는거냐',
 '대학교 기숙사 못짓게 시위하는 지역주민들 생각이나네...',
 '현대차가 현명하네..',
 '삼성은 이재용회장이나 이부진 사장은 인성이 좋으신 분들이에요~~ 호텔 입구 차 들이받았을때도  손해배상도 안물리시는 맘넓은 클라스~~~이해관계가 1도 없지만  전 삼성을 애용합니다..  3류 정치나  정부의 불이익을 감수하고 1류기업이 대한민국에 남아서 기업을 하신다는 자체가 존경스럽습니다',
 '트렆프는 회사들에게 지원해주고 사업과 개발을 할수있도록 지원하는데 우리나라는 노란봉투법 등 정치인들 입맛에 맞춰 회사를 해외로 쫓아 내려하는 법을 만들고 있습니다 . 심각',
 '우리나라는 국민 수준 때문에 망하고있는겁니다... 세상은 수많은 보통 사람들이 바꾸는 게 아니고 진짜 희대의 천재 한둘이 바꿔가는거고 저희같은 일반인들은 그들이 만들어 내고자 하는 세상이 옳은건지 아닌지 정도만 판단할 수 있는 머리만 있어도 되는데 그것도 없는 사람들이 다 같은 목소리를 낼 수 있다는 게 문제인 건 같습니다.... 좀만 길게 보면 공장이 더 빨리 지어지게되면 임직원들이 더 빨리 들어와 구내식당 말고 외부 식당도 자주 이용하게 될텐데 말이죠... 당장 눈앞에 이익만 좇으니 정말 답답합니다...',
 '모든말씀  팩트입니다~~~',
 '제가 대기업 회장이면 한국에서 추가 사업 절대 안 합니다.\n작은 가게를 한 적이 있었는데 진짜 빨대 꽂는 놈들 천지입니다.\n자영업 하나 하는데도 이모양인데 대기업에는 별의 별 요구를 다 하겠죠.',
 '부동산에 미친나라',
 '그러네요 기업친화적인 나라가 되면 하네요 자원도 없는 나라인대 안타깝네요',
 '장사꾼 놈들\n지들 장사 잘되면 그돈은 철저한 자본주의의 논리하에 본인들이 전부 가져가면서\n장사 망하면 나라에서 대책 내놔라 지원금 내놔라 강제로 지역상권 이용하라면서 협박함\n천 민 자 본 주 의',
 '우리나라는 삼성이 잘되야 한다    이재용화장님  응원합니다',
 '나라가 미쳐 돌아가는듯 합니다',
 '지금 울진 원자력 현장도 근로자들 식당 안에 없습니다 지역과 상생하는 입장에서 건설사는 식당을 설치하지않았고 외부식당 이용 또는 배달로 근로자들이 먹고 있습니다 그런데 실제 식당주인들 울진군 주소를 둔 사람들도 아니 외지인들이 많다는 사실입니다 뜨내기들이 많이 있습니다',
 '해줘해줘의 나라',
 '삼성은 국민기업이다!!! 이재용씨 힘겨워도 더 많이 애써주세요!!',
 '감사를 모르니 참 끝도 없는 안타까운 일들 다들 해도 너무하네',
 '진짜저도그생각햇는데 솔까말로 미국이더나음',
 '민원도  들어줄만한것  들어주어야 하는데 왜  아니다생각되는것을  들어주는지 이해안됨\n 그리고 일하고 밥 회사안에서 구내식당을  이용하게 해야 밥을  빨리 먹고 쉬지. 왜 저런  이상한 하고 이기적인 민원을 들어주는지 이해안됨.  그리고 매일 밥 사먹는것도 돈이 많이 들고 시간도 많이 듬. 구내식당은 회사 크수록 필히 있어야 하는데 주변식당 반대한고 못만들게 하다니 구내식당을 꼭 만들어야 합니다. 반대해도요 왜 회사가 큰데 구내식당을 만든다고 반대해도 회사입장.일하는 사람입징에서는 꼭 있어야 합미다.이기적인 이고 자기이익만.생각하는 민원을 걸러야 함. 항상 기업이 살아야 일자리 있고 그렇게 해야 먹고살지 왜 기업들 괴롭히는 이해안됨 대모도 시위도 좀 전체로 보고 개관적으로 생각하는 뇌를 가지고 생각합시다.   밖깥에 식당은 이용하는 사람도 있고 구내식당도 이용할수 있게 해야함.\n정부는! ! !잘못결정을 해서면  바로잡자',
 '우리나라는 망하는게 맞는것같음 정치만 욕할게 아님 국민수준이 이런데 무슨 국민수준에 맞는 정치가들만 있을뿐임',
 '미국 가면 더 힘들어요 ㅎㅎ',
 '안성시..미쳤다..누가 먹고 살게 해주고 있는건지  모르나..환영받고 특혜줘도 모자를 판에..내쫏다니..',
 '태국과 중국에서 우리 기업이 철수하고 난 후의 남겨진 모습이 얼마나 처참한지 모르고 있지요 ᆢ 그나마 삼성이 얼마나 애국 기업이었는데~~이제야 말로 떠날 시간이 되어 가는군요 ᆢ  삼성 그동안 너무 수고했다 ~~',
 '구내식당 음식 지겨운날 밖의 식당에서 사먹을수 있는데\n그러면 동네 상권도 살아나는거 아닌가??\n만명 구내식당을 못짓게 하는건 참....\n우리나라는 기업체 하기 힘든 나라~~\n기업 기밀도 국회가서 다 까발리게 만들지도 모름...어느 당 때문에....\n차라리 미국 가라고 하고 싶다... 강성노조들도 문제고...\n지자체장들도 정신차려라~~기업이 살아야 대한민국도 산다~~\n돈뜯어먹을 생각말고 미국처럼 해봐라~~',
 '주민의 시위가 아니고 공갈범으로 법적 처벌을 해야 합니다.',
 '솔직히 말해라 우리동네서 공장운영하려면 돈 내라는것 아닌가 양아치도 아니고',
 '시민단체를 전부 없애고 정부 지원 다 없애라',
 '아니 근데 이걸 지역탓을 하는게 맞나? 솔직히 경기도가 아닌 다른 어떤 지역도 반도체 클러스터 들어온다고 하면 전격적인 지원에 지자체 협력이 이뤄질건데, 굳이굳이 용인에 그걸 세운다고 한건 삼성 아님?',
 '이들 투자기업들은 여러시들 중에서 혜택을 많이 준다는 시를 택해서 투자하는 것으로 알고 있어요!',
 '좋은말씀잘들었습니다\n감사합니다',
 '이런 내용을 더 적극적으로 홍보해야. 합니다.',
 '거지 근성입니다. 어떻게든 뜯어먹으려는 거죠. 식당 맛있게 하면 가끔 그 근로자들이 그 식당 이용하면 얼마든지 돈 벌수 있는데도... 지나치게 돈을 바라는 거죠. 강성 노조들에, 악법 만드는 정부 ,지역주민들까지, 참... 벼텨낸 우리나라 기업들이 대단합니다.',
 '집단이기주의 …..  문제입니다.  정말 비상식적입니다.  방송 감사합니다.   삼성 자랑스런 한국기업인데 지켜내야 합니다.   텍사스에서',
 '좋은영상항상잘보고잇습니다',
 '원래 삼성 무척 싫어했는데 경기도 남부 개발에 놀라운 공헌을 보고 호감 기업으로 생각이 바뀌었음. \n나라가 못하는 일을 삼성이 대신 해냈음.',
 '진짜 맞는말이다',
 '격하게 동의합니다. 우리나라 사람들 정말 너무 이기적입니다. 사내든 사외든 정말 요구수준이 끝이 없어요. 그러니까 자꾸 망하는 길로 갑니다. 근데 문제가 심각한것은 망해도 같이 망하자라고 망하는 길로 가는걸 두려워하지 않습니다.근데 이웃중 누가 성공하는 꼴은 보지 못합니다.  참 큰일입니다.',
 '5:55 한국이 구조적 타이타닉인 이유 중 하나라고 생각해요. 나라가 좁으니 지역입김이 너무 쌔서 기업들이 뭘 할수가 없음',
 '삼성이있어 우리나라가 잘살수있다는것을 알아야합니다.!\n기업을 대우하지않는나라는 망하는것이 진리입니다',
 'ㅋㅋㅋ 반기업 지방이네.\n기업 없이 뭘 먹고 살라고 우리나라 개인들 이기주의 대책없다.',
 '사내 식당을 이용하던 사와 식당에서\n식사하든 그건 먹는 사람 마음이다.\n하여튼 식당주들 심보가 시커멓다\n망하는 꼬라지를 싶다',
 '좋은 지적입니다  \n대기업이  시에  공장을 건설한다면  관련  시청에서  도움을 주어야  \n손발이  맞아  착공도 하고 완공도 순조로울 것이 아닌가요',
 '환경은 중요하죠. 더군다나 삼성은 반도체로 인해 암환자가 된 직원들을 오랫동안 무시하고 방치했었지요. 그렇지만 식당은 너무했네요. 지역상권도 중요하지만, 직원들의 복지도 중요하니까요. 1시간 이내에 왔다갔다 이동시간에, 주문시간에, 먹는시간까지 끝내려면 너무 지칩니다. 그걸 매일 하라구요? 직원들 하나하나를 무시하는 너무 이기적인 요구입니다. 그냥 냅둬도 지역상권은 이용하게 됩니다. 누가 매일 똑같은 구내식당을 이용하고 싶겠어요? 가끔 맛있는거 먹으러 나가고 싶지요. 그럼 깨끗하고 정갈한 메뉴를 개발해서 고객을 부를 생각을 해야지, 무조건 이용하라니요.. 너무 무책임한 상권갑질이라고 생각합니다!',
 '쟤네 간첩이나 분탕들 아니냐',
 '존경합니다 \n진짜 대단하십니ㅡㅏ',
 '저출산율1위의 나라, 늙어가는 나라, 경쟁력 잃어가는 나라, 집단 이기주의 나라, 파시즘나라 = 대한민국',
 '그러니까 말입니다. 지방은 무시하고 중앙 수도권만 사업성 높은 사업 독식하니 체한 겁니다.  그러니까 질 나쁜 사람들 보상은 끝이 없습니다.  부안 방폐장 사건이 우리의 일상이 되고 있습니다.   균형발전 무시한 정책의 대실패입니다. 몇 지연 몇 달 지연이면 기업은 죽음입니다. 지역주민은 답답할 것 없지요. 노조도 마찬가지입니다 최근 임명된 노동부장관님 말씀에 따르자면 파업이 근로일수에 폼함된다는 얘기를 듣고 깜짝 놀랐습니다. 기업에게는 국민들에게는 하루하루가 살얼음판이었는데  그들에게는 잔치였던 셈이죠. 파업도하고 돈도 받고, 그들에게는 국가도 기업도 투쟁의 대상?',
 '모든 개발을 수도권에 지으니까 문제죠. 수도권은 포화 상태인데 계속해서 수도권에 개발해서 문제입니다. 지방도 균형을 맞추면 안그러죠.',
 '삼성 Sk 감사합니다!\n애국 기업 입니다!',
 '간첩이...심각하다...',
 '기업하기 힘든 한국\n남이 잘 되면 배아파하는 심보가 깔려\n부자를 경계하는 심리가\n밑바닥있기 때문\n지금은 국민과 나라가\n기업이 잘 되도록 협력해야 나라가 산다\n위기의 한국을 살려내려면\n마음을 선하게 써야한다',
 '기업을 우습게 알고 내부 정보를 바치라는 정치인들. 기업을 유지하지 못하게하는 높은 각종 세금들.  한국은 유일하게 잘 할수있는게 제조업밖에 없어요. 그런데 그런 제조업을 말아 먹고 싶어하는 사람들이 많다는것',
 '좋은 내용 입니다.',
 '욕을 부자에게  하는것 보단 \n일부 그들이 하는 행동을 욕하는것이죠~\n\n기업이 이익 추구하는 것은 당합니다.\n단 기본과 상식에 벗어나면 안되겄죠~\n\n우리 나라가 기업 친화적인 국가는 아니라 생각하지만 그 이면에 그런 이미지가 외 생겼는지 생각해 봐야겠죠\n\n대를 이어 대표직을 물려준다던가 등등 너무많아 나열하기 힘들거에요~\n대를위해 소를 또는 상생이란 말로 모든 부분을 덥을순 없겠죠~\n\n하지만 깊이 고민해 봐야할 문재라고 생각되네요~',
 '기업이 봉은 아니지만 하나만 생각하는겁니다.지금 대기업으로 성장시켜준게 나라고 국민입니다.워낙 나라가 못 살아서 잘하는 기업들 몰빵시켜줘서 키워줬고 국민들도 하나같이 애국하자로 돈벌면 우리나라 기업제품만 소비해서 키워줬죠~또 미국하고 다른게 미국처럼 인구가 많은것도 아니고 기업이 내수경제를 무시하면 내수경제는 살아날수 없는 구조인데 주변 상권 무시해라?말도안되는 소리죠~지금 주요도시 근처 상권 만들어진게 다 정부에서 여기로 기업 옮겨서 상권 좀 만들어라 다 몰아줄게 이렇게 만들어진 나라임...',
 '공산 주의자는 반드시제거해야 한다 그리고 세금 걷는 거 기업을 어렵게 만드는 법은 모두 없애야 한다',
 '판교도 똑같았어요. 주변 상인들이 사옥내 식당 못 짓게 반대하고 난리 쳐댔죠.',
 '그게다 좌파.   문제다.  문가정부. 민주.이제명.',
 '삼성 화이팅입니다 ',
 '우리나라 정부는 진짜 정신 차려야한다!윤정부때 정말 값어치있는 나라였는데 가짜정부가 기업 망하게하는 지금 진짜 답없다!',
 '아무것도 없던시절에 기업세우고키우기위해 많은 노동자들 희생했습니다 정직한지도자들도 많이 계셨고',
 '정치인이나 국민들이나 중국따라 "공동부유"를 주창하니  식민지 백성으로 살기 딱 좋은 근성들이네',
 '간첩이 가득한 이 나라 상황을 보면 충분히 이해가 됩니다.',
 '이죄명당선되서 힘듬... 기업죽이기 하니까',
 '미국으로  가는것이  정답   우리는  차라리  미국의  한개의  주 로  남는것이',
 '한동훈이 삼성을 없앨려고 발버둥을 쳤지.  .그런데 삼성이 없어지면 우리나라 어떻게 될까?  SK는 중국몽',
 '그러니 지방에 공장지어야 됨',
 '지역 이기주의도 문제인데 국가 차원에서도 교통정리 좀 해줘야하는거 아님? 손놓고 나몰라라 하고 있는 동안 7조가 그냥 날라가고 있는데 ㅉ',
 '걍 다 해외로 가라\n우리나라 국민들과 정치인들은 다시 개처럼 굴러야된다.\n대충 50년에 한번씩 굴러야 되는데 지금이 딱 그시기다.',
 '국민이 국가에 의지하게 만듭니다.\n국민 주권 신장 이라는 명목으로 그들에게 이기심을 가르쳤습니다...\n자기만 아니면 된다는 생각이고, 좋은것은 가지고 싶은데, 겉으로 가증 떤다고 바쁜 것이 사람 입니다.',
 '진짜 이런 종류의 영상들 볼 때 마다 느끼는 건데\n우리 나라 사람들 특유의 나만 잘 살면 된다라는 이기적인 사고방식\n나무보다는 숲을 보라는 장기적인 안목,인식이 없는게 너무나도 안타까우면서 화가 난다\n다 좋은데 대한민국 기성세대의 이기적인 생각이 대한민국의 발전을 저해 한다고 봅니다\n양질의 일자리가 유지 되어야 그 지역의 경제도 활성화되고 인구가 유지 될건데\n한치 앞만 보고 있으니...',
 '군부대 옆 동네랑 똑같네.',
 '다 떠나 나라 망해봐야돼',
 '주민들이 왜 그럴까요? \n주민들만의 문제가 아닌 것 같습니다',
 '주변 민도수준이 참…',
 '중국이 대한민국 정친 시민단체을 움직인다',
 '작은 것 취하려다 대어 다 놓치는 . ㅎㅎ\n참 좋은 지적이여요. 삼성 고속도로. ㅎㅎ 멋져요. 이런 정신 ㅎㅎ',
 '기업이 성장하고 혁신을 이루려면 첫번째가 땅을 사는일이다. 그런데 이 땅을 사는데만 몇년씩 걸린다. 보상을 해야 하기때문인데 이래서 구입가가 천정부지로 뛴다. \n정부는 정부대로 지자체는 말할것도 없고 주변 상권에 연결된 주민까지 한목소리로 보상을 요구하고 해결하는데만 몇년씩 낭비하니 그자리에 공장짓고 허가받고 도로건설및 인프라구성하는데만 한세월이라 기업은 출발하기전부터 엄청난 마이너스선상에서 출발하게 된다. 100m달리기를 하는데 30m뒤에서 뛰는것이다. \n그러니 우리나라는 기업이 밥먹고살기 정말 어렵다. 기업이 공장을 지어야 일자리부터 세수증대 관련된 모든것이 살찌고 성장증대가 이룰수 있는데도 말이다. \n이걸 정부와 국회가 바꾸어야한다. 다 뜯어고치고 필요한것만 지원해주고 열어제끼면 된다. \n우리나라는 수출해야 먹고산다. 기업이 바르게 성장할수 있게 모두가 합심해야 한다\n이렇게 탄탄한 내실을 이루면 트럼푸 10놈이 나타나도 북어로 만들수 있다.',
 '대한민국 기업들 괴롭히는 인간들 대대손손 거지꼴을 면치 못할것이다.',
 '맛있게 만들어서 팔면 다 간다',
 '아~~~~~기업들이 해외로 빼는 이유가 있었내,,,,아~~이런거 까진 한번도 생각을 안해 봤내,,,,,,지역에 보상과 반대 시위등,,,,,중소기업 하나 없는 지역도 엄청 많은데,,,,다 이유가 있었내,,,,,,,',
 '지역이기심이다.  더 발전될 수 있는 \n기회를 발로 찬다.',
 '대한민국 민관이 합해서 기업을 내쫒는군요.  눈앞의 헛된 욕망으로 황금거위를잡고있네요.',
 '대기업은 반드시 필요합니다 서비스인프라정신에서도 다른 업체들하고는 상대가 안되죠 반드시 필요합니다 그래야 사회가 돌아가요\n하지만 나는 내가 맘에드는 제품을 산다.',
 '지주식이 아니고.. 자주식 주차장',
 '그냥모든기업미국가요.우리나라국민중일부가너무황당한짓황당요구.나쁜노조.진짜힘들겠다.우리국민위해있어주면좋은데.엉뚱한것들이과욕부리니외국가는게맞다',
 '구독 좋아요 했어요 ',
 '심각한 시기와 반목. 또 이기심.  그걸 부추기고 이용하는 정치권... 걱정만 늘어나는 요즘인것같네요',
 '상인들 극혐이네요..',
 '맞네 다 맞는 말 숲을 못보고 당장 자기 앞 나무만 보는 사람들',
 '기업들을  해외로  내 쫒는구나    기업이  들어와야   \n일자리가  생기고  국민이 부유하고    나라가 부유해진다  큰일이네요  앞으로   기업도   지역  공해부분도  신경쓰면서  잘하면  되지  않을까요',
 '정말 안타깝지만\n국민들 한국인들 정신 차릴 기회가 될겆이다',
 '삼성 이회장님 SK최회장님 한국에서 사업장 모두 폐업하고 미국으로 가셔서 본사를 옮겨 사업에 몰두하 시 길 바랍니다',
 '해외 이전이 답으로 보이네요.  그나마 핵심기술 만큼은 국내에서 가지고 가겠다고 해도 \n국내에는 이렇게도 뒷다리 잡는 이권단체가 너무도 많아,  빠르게 세계가 급변하는 \n이 시기에  기업이 도저히 따라갈 수 없는 환경이니 말입니다.',
 '기업이 있어야 일하는 근로자 자리도 보장됩니다.!! 기업이 운영하기 좋아야 그밑에 딸린 업체의 업체 ~~쭈욱 가는 거라구요~!!!!',
 '중공세력 ,남녀,기업,지역,세대 갈등 조성에넘어가지맙시다 그들은 서로 혐오하고 자국기업,나라 갈등일으킬라고 애쓰고있습니다',
 '호의가 계속 되면 권리인 줄 아는 전형적인 사례',
 '총체적 난국이군요 뭐하나 제데로 굴러가는게 없으니...',
 '우리나라에도 화성이랑 저기 아산 쪽인가 기업 이름 길 있는뎁',
 '1:00 한국에서 회사 하기 힘들다고 미국가면... 그냥 구속에 회사 박살인데요..',
 "화가 나네요. . .지금 사는 곳의 자동차 전용도로가 생길 때 램프 설치하면 조망권 문제 생긴다고 민원 넣어서 잘 자동차전용도로의  중간이 끊겨서 신호등이 생기는 바람에 직진도 신호를 기다렸다가 가야 하고  램프가 없어서  좌회전이 안됩니다.그 구간에서는 출퇴근 시간에 차도 많이 밀리구요. . .많은 사람들이 매일매일 스트레스를 받습니다. 좌회전을 하려하면 우회전 받아서 두 블럭 위까지신호 두 개를 지나 다시유턴을 받아야 하지요~! 이런 경위에는 소수 이기심편을 들게 아니고 다수를 위해 어느 정도의 보상을 해주고, 강제로라도 진행해야 하는 것이 맞습니다. 그런데 귀찮은 지 '그럼 불편하게 살아 봐~!' 하는 듯 저 따위로 만들었습니다. 공무원도 문제입니다.",
 '한국은 망조가 들었구만',
 '난기업상속세부터없애거나줄여야한다그리고바지사장경영반대임오너경영',
 '상인들은 망해봐야 정신차릴려나.대기업들이 얼마나 소중했다는것을 알아야돼.',
 '이재용 회장님 힘내세요!\n얼마나 힘드실까~~~~',
 '한국이 망하는 이유 개인의 탐욕이 극을 친다. 곳곳에 간첩이 많다. 곳곳에 친일파가 판친다.',
 '어느정도 밀어붙여야 하는데 정치인들은 표의식해서 못함ㅋㅋ',
 '호의가 계속되면, 그게 권리인줄 안다. 대한민국 최고 단점.',
 '안타깝네요,,,,, 그래도 기업가님들 홧팅!!!!!!!!!!!!!!!!!!!!!!',
 '이런 내용 너무 속터지네요. 이기심에 지발 지가 찍는 사람들',
 '반도체 공장을 기피시설로 지정했다는건 진짜 미친생각이네. 다른 지방도시들 상황을 알고도 저러나?\n다른 지방도시에서 이 소식 들었으면 전부 앞다투어 자기네 지역으로 공장 이전하라고 했겠다. 배가 불렀네 아주',
 '지금은 어떤지 궁금하네요 ? 저는 California 에 주거한지 오래됐는데 환경문제, 세금등등 으로 인해서 회사가 많이 타주로 이서가고 있어요 .',
 '기업으로 부터 나오는 세수가 무시못하는데, 시에서 나서서 교통정리 해줘도 모자랄 판에;;',
 '주변상인들과 상생을 해야져...... 사내식당으로 주변상인들 힘든건 안중에도 없나요 ㅠㅠ',
 '도대체어너지역의관리자들인가!',
 '대학에 기숙사 짓는다고\n주변 원룸업자들이 머리띠매고 기숙사건립반대\n시위하는 미친국가입니다.\n학생들과 학부모들 피빨아서\n호의호식하겠다는 사고방식',
 '지역구 표 때문에 주민편에 총대매고 앞장서는 정치인들은 반성해야됨',
 '동네 식당을 만명이상이 이용하다 밥 늦게 나오고 점심 시간에 못 먹어 배고프고 \n다 어떻게 할려고 그러시는건가요 ㅠ',
 '정부가 기업을 키워야 나라 경제가 살아나고 일자리가 늘어나야 하는데 기업을 죽이는 정책은 사회주의 정책아닌가',
 '삼성이나LG, SK....등도 본사는 한국에두고 생산지를 헤외에 두는것이 현명 한것 같다.',
 '우리나라 정치인들은 나라생각 기업생각보단 표심 생각해 시민들 눈치보느라 아무것도 안하죠 그냥 자급자족 기업들이 나라를 키운셈이죠',
 '맞습니다 일자리가 없으면 도시도 자영업도 마을도 사라지게 됩니다',
 '엇 오늘 처음왔는데 내가 sod가 아니네 ...',
 '상생이란 것으로 공짜 먹으려 하고 잘나가는 기업 때클 거는것',
 '기업이 살아야 나라가 부흥한다 ^^',
 '이게 우리 대한민국의 현실이지... 선동은 쉽게 되고 반기업정서에...\n이제 노오오오오란 봉투 법도 통과되었다.',
 '이거 각 기업 회장과 대통령도 보셨을까요? 꼭 보셨길 바랍니다.',
 '저녁 장사하면 되지 잘 운영해서 직원들한테 잘해주는 삼성한테 못뜯어먹어 안달...ㅠㅠ',
 '삼성 본사 경남고성 어때',
 '문닫으면 누가 손해일까??? 참 그놈의 사람들의 욕심이란~',
 '사람들이 욕심이 목구멍까지차서 /이 많은 청년들 일자리 없어지면 자식들은 뭐 먹고 살아야하는데/ 제발 미래를읽는 대통령이 나타나 기업이 잘되게 좀 힘이 되어주세요 삼성 sk 감사해요',
 '삼성 현대 없는   한국은  이미  베네수엘라  짝 났다.',
 '거의 공산주의나라네 미국 캐나다 이런곳에선 있을수 없는 일이다.',
 '구독이 안되네',
 '각자 사는게 힘들어지고 팍팍해지니까 더 여유가 없어지고 그렇게 바뀌는거 같네요.. 결국 무한의 굴레에 빠진 건 아닌가 싶어요',
 '세상 미친사람들이 너무 많다\n내 주변 보상 발언 하는 사람 있으면\n한소리 하고 사람취급 하지말자',
 'LNG 발전소요??? 탄소국경세는요??',
 '나라가  기업들을  망하게하려고  고사를 지낸다  ~ㅠ',
 '모두 정확히 맞는 말입니다.\n기업이 잘 되면 \n결국 국민 개인도 간접 혜택 많이 봅니다.',
 '저도요 삼성이 있어 대한민국이 행복합니다️️️️',
 '이기적 국민들 이득은 보고싶고 비들 사는곳은 안되고 \n언제가는 후회하응알이 올것이다 \n이번 기회에 해외 기업을 옮가는것도 좋을뜻합니다',
 '나라나 국민들이 하는 꼴보면\n이 나라 미래와 국민생활 뻔 !\n정치(政治)인은 망치(亡治)를',
 '그러니 누가 돈을 투자해서 기업하겠어요  법이 개법이지요',
 '기업 못하게 압박만 넣는데,\n나 같아도 오라고 환영하는 나라에 공장 짓고 싶겠다.',
 '문제가 심각하다 맛있고 질 좋은 음식을 먹는건 좋지만 싸지는 않겠다와 무엇이 다른가  반도체공장 들어서서 집값 오르는것 까지만 이라니 이정도면 나라의 발전을 저해하는 요인들을 국가 가 나서서 교통정리를 해야한다 가진자들의 이기심이 나라를 망조들게 하고있다',
 '감사합니다.',
 '기업이 구내식당 서비스를 제공하는 건 잘한일이라고 생각이 드네요. 주변상권의 상인들은 그에맞게 차별화를 하거나 경쟁력을 발휘해야지요. 서비스도 개선하구요.',
 '반도체회사 한국에 지을예정이면 예정후보지를 청해놓고  해당 지자체에 연락하여 조건을 사전에 물어 골라서 준공하세요',
 '너무 극단적인대 두기업 응원하는 사람이 더 많고 제발 재벌걱정하지 마시길 빕니다',
 '삼성동 현대 gbc 건설현장도 근처 상인회 반발로 현장내 한바집 금지라서 근로자들 근처 만원정도 주고 식당가야되요..참 이게 말이 되냐고..',
 '공동체가 무너져서 자신의 삶이 무너져야 공동체의 소중함을 알게 될겁니다. 지금 시대는 공동체를 잃어본 적 없는 오직 개인이 우선인 사회입니다. 결국 잃어야 알게될겁니다.',
 '알써,알써,,',
 '애플의 복지라고 봐도 대한민국 공무원만도 못함,,,대한민국사람들은 어느덧 복지의 허상속에 살고있음,,,\n또 다른 예로 최저임금을 보면 약자의 안전기준인데. 무슨 보편임금인줄 암,,,',
 '우리나라 사람들  진짜  너무한다   지역 를 만들어서  \n공장을  만들어야지',
 '구미가 망한 이유가 이것과 비슷하죠.... 앞이 안보이는 구미에서 살고 있는 1인입니다.',
 '성균관대학인가? 서강대인가?\n기숙사 지을때도 주변 숙박시설 업자들이 투표권을 빌미로 공기관을 압박한 적이 있죠~\n그때 학교에서 사용한 방법이... 학생들의 주소지를 새로짓는 학교 기숙사로 옮겨서 학생들을 주민화 한 겁니다.\n이를 통해, 성공적으로 기숙사를 지어서 학생들을 후원했죠.\n이 방식이 어떨까요?\n노동자들을 공장 주변으로 주소지를 옮기고 밀어 붙이는 겁니다. ㅎㅎ',
 '기업은 욕먹지 말아야죠.\n하지만 능력없는 불법세습 제벌들이 물러나야지 않을까요?',
 '이재용을 대통령하면 어떨지...',
 '맞아요',
 '지자체 장들이 보는건 오직 표만 보지요 저들에겐 국가는 없습니다 표만\n저런걸 보면 다 해외로 나갔슴 합니다',
 '삼성 미국행이 맞다ㅡ',
 '해당 지역 주민을 탓하기 보다는, 여기 계신 분들이 해당 지역 토지, 주택 구매해서 기업들 숙원 들어주자는 데 한 표 보태기 운동하면 어떨까요?',
 '대한민국 대기업 ceo 들은 정말 보살이다\n보통 사람이었으면 다 외국으로 떠난다',
 '자영업 자들도 정신 못차리네.. 망해도 정신 못차리지...다른나라는 눈부시게 발전하는 구나.. 부럽소',
 '교통정리 필요합니다. 이사람 저사람 다 만족시킬 순 없죠. 반도체 클러스터 얘기 나온지가 언젠데 빨리 추진되길 바랍니다.',
 '지역기업이 일어나야 지역경제도 살아날것임',
 '회사에서 밥주는게 이제는 당연시 된 한국인 근로자들ㅋㅋ',
 'SK 하이닉스 같은 회사가 한국에 있는게 기적이에요 SK 승승장구 해 주세요',
 '얼이서금의 이기심이요 \n일자없음의 이유이기도 하네요 ㅜㅜ',
 '과거 삼성과 현재 삼성은 많이 다르죠... 삼성이 국가와 사회에 좋은일 많이 한다는거 인정 해야 합니다.\n이런거 보면, 우리나라 지방자치는 명확하게 실패 한 제도 입니다.',
 '약간 27사단 확대판 같네요\n결국 27사단 철수 하고 나서 지역 상인들도 매출 감소로 이어지긴 했죠\n근데 이건 알아야 합니다, 결국에는 내수 시장에 돈이 돌려고 한다면\n\n기업이나 어디 소속에서 일하는 사람들이 돈을 어느 정도 벌고 \n식당에서 감당이 불가능한 수준의 인원이 밖으로 밥을 먹으러 다녀야 한다는 거죠\n\n근데 요즘 보면 기업들이 활황이니 여러 기업들이 그런 말을 하는 걸 보진 않은 것 같네요\n지금 관세 때문에 더 그렇다곤 하지만\n\n실질적으로 노동자들이 밖에서 사먹는 것 보다 구내 식당 이용해서 더 저렴한 식사를 해결 하길 바라니깐요\n사외 식당을 이용 한다고 해서 인센티브나 뭐 혜택이 있는 것도 아닌데',
 '대기업은 한국떠나야함 중요함을 느껴봐야됨.',
 '삼성 이재용 똑똑하고 인물도 인성도 좋음',
 '감사합니다',
 '진짜 직장없어 허덕거린다 하지말고... 대기업 공장 대한민국에 다 지을수 있도록 정부가 현행법 계정해서 대한민국 일자리에 기여해야함 미국인도 중국에갈 공장들 한국에 지었으면....,,, 우리나라 경제가 이렇게 될거 같냐?!~ 뇌없는 지역주민들과 지자체는 당장에 눈앞이익만 보려다.. 모든걸 다 놓친다.일자리 없다 하지말고 대기업 공장 대한민국에 지을수 있도록 규제도 미국수준으로 풀어주고 지원도 미국수준으로 지원해라',
 '이나라 국회를 해산 해야지',
 '02:09 전형적인 일반화의 오류입니다.',
 '근데 용인에 짓는 시설에 안성에선 오폐수만 떠안는 상황에서 징징댈만 하잖아',
 '반기업적인 따블당도 문제지만 지자체 장들이 진짜  문제네요',
 '근데 7만 전자 언제 오르냐... 사려고하는데 겁난다',
 '맞는 말씀 입니다.....!!',
 '삼성 다 찬성하는데 식당은 동네식당 쓰게 해주는게 어떨지 동네도 살고 국가도 살고 ~',
 '오~ 애국이란 이런거다  내가할수있는 범위내에 작은일부터 해야겠다',
 '우리나라 정부가 하는게 도무원가 중국 전기차에 보조금이라주고\n그냥 대통령이고 정부고 여당 아두것도 하지마라\n삼성 , sk, 가 국민일자리 만들어 준다. 그들 기업이 진짜 애국자',
 '기업을 살려야함 주민들 정신차려라 너무 이기적이야',
 '주변식당 이용하고  점심값 일부를 회사에서 지원해주면 되지 않나요 ?  상생이 그런게 상생이겠지요...',
 '욕심 때문에 나라 세금과인력이 외국으로 제발 생각 좀 하고 행동해라 멍청한 인간들이요 한순간에 행동으로 우리나라에 미래는 업다',
 '식당 주는 음식을 맛있게 만들 생각은 안 하고 날로 처먹을 생각만 하니 망하지 망해도 싸다 싸!',
 '미친 행정이네요.',
 'ㅎ ㄴ향우회',
 '진짜 정답입니다',
 '한국이 앞으로 살기가 힘들것',
 'ㅎ ㄴ향우회',
 'ㅎ ㄴ향우회',
 '능력이 넘치는 그대는 곧 미국가겠네...',
 '기업에게 친화적이지 않은 나라는 망할 수 밖에 없음…심지어 우리나라는 기술 하나로 큰 나라인데 그 분야를 홀대하니 당연한 수순임',
 '펄벅의 한국인 묘사를 보면 (세상은 아는만큼 보인다)\n1.한국인들은 땅의 전부를 농사 짓지 않는다.왜냐하면 많으면 관리들이 수탈 하니까 ㅋㅋ\n2.경제인들은 처절하게 목숨 걸고 싸우는데 정치인들은 관리들이랑 거기다 빨대 꽂고 어찌하면 더 빨까? 궁리 하는,예나 지금이나...\n3.경제인들은 나라를 발전 시키나 청치관료들은 나라를 퇴보 시킨다.\n4.지금 20/30 세대가 45/55 세가 되어야 개선 될것이라 본다.',
 '국민들이깨우치지못해자기욕심만생각해이재용회장님의큰일을막아지금같은일이생긴것같아요.우리국민들정치인들좀바낍시다.',
 'GM대우 처럼 현명한 선택을 하시길... 그 선택의 주인이 누가 될지는.............',
 '지역상권 살릴려다 직원들 속터지겠네. 일하다 말고 그 많은 인원들이 언제 외부까지 이동해서 밥먹고 돌아오니? 그리고 인원이 많으면 싸게 주던가. 한끼 가격이 미쳐돌아가네. 적당히 해라. 진짜. 개꼴보기 싫으네.',
 '경기도를  떠나서 충청도,  강원도, 경상도  쪽으로 가서  공장 지으면 환영힐텐데...',
 '삼성은 대졸 후 꼭 가고 싶어하는 기업으로 알고잇다 능력없는 사람만 나거기 싫어 말로한다 실력이 안되니까',
 '누가 대한민국에서 기업을 하겠어요',
 '5분 31초 나이스 대기업들 다 해외 이전하자 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ',
 '지역상인의 열에 아홉은 날로먹는 장사하거나 권리금 액시트를 꿈꾼다.',
 '굶어봐야 내가무슨짓을했나 알겠지요',
 '이러니 기업들이 한국을 다 떠나지... 다른지역가봤자 거기서도 똑같이 저러니',
 '솔직히 말씀드리면\n오너분들은 너무 깊은 애국심을 가지실 필요없습니다.\n이런 쓸데없는 스트레스를 감내하느니 주요 구매처가 있는 \n국가에 법인및 공장설립하여 운영하여 건승하십시요',
 '일단 채널주 집 앞에 오폐수 흘러가면 똑같이 말 할 수 있을지 궁금함 ㅋ :D',
 '부산 기장군에  원자력 발전소 있는데 부산시민들  전기 혜택 하나도 없다',
 '왜 수도권에 만 지으려 하나 지방에 지으면 되지~~~~',
 '손해도 좀볼줄아는 인간들이 되어라 \n내것은내것 남의것도내것 이기적인 국민성 절대로 안변합니다 망해봐야 알겠지요',
 '상인들이 미쳤네!!!!!!!!!!!!!!!!!!!!!!!!저러니 니들이 망하는거다!!!!!!!!!!!!!!!!!!!',
 '기업이 먹여살리는 거지 정치는 양아치 참 한심한 나라다',
 '도시락 싸갖고 다니며 참교육 시켜야한다.',
 '팩트) 구내식당이 있어도 맛있는집은 직원들이 알아서 찾아간다',
 '동네식당들이 삼성인원 수용할수있게 크게 맛있게  하면돼',
 '지주식 주차.가 아닌 자주식 주차. ㅎ',
 '기업들은. 미국으로.  가는게  답이다!!',
 '기업이 잘 되면 세금을 많이거드면되고 일을 잘할수있게 해주면 일잘이가 늘어날데고...당연한거아닌가???왜 기업을 못잡아 먹어서 난리인지...기업이 없으면 일자리가 없어지면 그때 후회 할건가...미ㅣ겠다..',
 '기업이 있어야 국민이있고, 나라가 사는거다 너무 법인세인상,상속세인상은 하지말고 국내기업에 더 특혜를 주어야 우리국민이 잘사는거지~~',
 '텍사스에 삼성이 투자하는 건 관세때문 아닌가요?',
 '어이가 없네~ 된통 당해도 쌉니다 저 식당들 ㅋㅋㅋㅋ 회사에서 좋은 질의 밥 싸게 해주시면 얼마나 감사한데~ 회사는 그냥 신경쓰지 마세요~',
 '재벌 친화적으로 법을 전부 고치다보면 부익부 빈익빈 심해지고 물질 만능주의로 인간 가치하락, 부자 감세로 세금은 전부 서민이 떠안게 되지요, 화성에 대기업 들어와도 직원들 대부분 서울 근교에 삽니다',
 '나라가 지역이 적극도와야 하건만 그 기없이 없으면 어쩔까요?\n점심이 아닌 다른걸로 수혜가 될수 있는건 왜 생각을 안할려고 하는지,,,',
 "'라도공화국'을 피해\n찾아 온 곳이 이 정도",
 '기업이 잘되야 노동자가 잘된다 기업을 조지면 노동자도 조져진다',
 '지역주민들이 문제가 아니고, 지역 주민이 왜 저렇게 까지 할까 생각해보면,\n결국 이 나라 정치판이 썩어 빠졌다고 밖에 볼 수 없다.',
 '도시 말고 차라리 지방 소도시 촌에 지으면 안되는가 미리 땅 확보해서 엄청 땅많은데',
 '모르는 국민은 많을까?\n실업자가 득실해봐야 알라나.\n에휴',
 '아오 이 미친나라..',
 '기업에 도움이 안되는 국가\n이러니 정치가 지들 돈만 챙기고 기업은 나몰라',
 '어휴..우리나라 사람들도 진짜 징한 사람들 많어......대기업 들어왔으니 어떻게든 뼈까지 발라먹어야지..이런 거지..대기업이 그 지역에 벌어주는 세금은 생각도 안해... 전문가를 인정하지 않는 사회는 퇴보하는 사회.',
 '노조도 미쳤고 상인들도 이기적이고 기업이 지역 살려주면 고마움을 느껴야 하는데 진짜 거지근성 같더라. 기업이 수준이 높아지면 사람도 그 수준에 따라가줘야 하는데 수준 미달이라 솔직히 외국으로 기업이 떠난다 해도 할 말이 없겠더라.',
 '질문이 생기는데요 대권 주자 인터뷰의 뜻을 잘 이해하지 못하겠어요. 재벌 기업을 비판하는 건지 우리나라 대학들은 목적을 줘야 연구한다고 대학을 비판하는 건지요?',
 '3분 50초 주민들 쌤통 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ',
 '한국 제조업들이 해외로 나가기 시작한게 어디 하루이틀인가요? 이제 남아있는 기업들도 여러 방면에서의 괴롭힘 때문에 해외로 나가겠죠. 서비스업이나 금융업 등으로 국가 경제를 성장시킬 수 있을 정도의 역량은 없는 한국이, 그렇다면 뭘로 성장해서 국부를 쌓아올리고 국민들의 지갑을 채울까요? 그저 복지로 나눠먹다가 점차 가라앉게 되겠죠',
 '밥먹으러 가면 최소 1시간 없어짐',
 '저런 동네상인은 동네상인아니고 소식듣고 돈벌려 주변에 함바낸 다른지역세력입니다. 전구간 개발사업지로 구성해서 싹 몰아내야 합니다.',
 '대기업이 우리동네에 들어오면 좋겠다',
 '삼성 SK 아니였으면 알지도 못하는 동네를\n동탄, 고덕처럼 땅값올려주는데\n원하는거 ㅈㄴ많네 \n그냥 농사나 지으면서 번화가 나오려면 자차로 1시간이상 걸리게 냅둡시다\n도로도 깔아준다는데 어이없다\n그냥 소나 타고다니면서 풀잎으로 피리나 불어',
 '식대 복지 포인트 늘리고 직원들은 계열사에서 복지포인트로 도시락 구매하게 지원하자\n도시락으로도 염병 떨진 않겠지?',
 '옛날 90년대에 미군부대에서 근무을 한적이 있는데 미군 부대에 일하신 분들이 외치는구호, 미군부대 철수! 그러면 그때 미군부대에 일하면서 월급받고 있으면서 그들이 철수하면 실업자 신세인데 어이가 없었읍니다.',
 '빡쌧편이 뭐에요?',
 '앞으로 삼성에게 온 국민이 빌어도 법인을 미국으로 옴길 것이다.',
 '관청들이 아직 정신못차렸네',
 '민주노총 산하 노조원들은 모두 정신차려야 합니다. 정치 파업이나 일삼으며 좀 먹는 놈들!\n우리 국민들도 정신차려야 하고요, 국가 기간산업을 유치하면 덕을 보는 것은 그 지역민인데\n달달한 건 좋아하고 쓴 것은 이기심으로 기업들이 일하기 힘들게 하는 우는 벌이지 말아야!',
 '뒤에서  \n변호사 와 국회의원들 들 \n돈돈돈 \n벌려는 인간들  입니다',
 '처인구 산골에 개발해줘서 땅값 떡상했으면 적당히 해라 대기업 들어가서 대박났음 감사하게 생각해라 욕심 작작 부리고',
 '열심히 일한 노동자들이 시원한 사내식당에서  실비로 맛있는 점심을 먹는게 당연하지 점심을 사먹으려면 사내식당보다 2배이상의 값을 지불하고도 더 나은 식사를 할 수 있다는 보장도 없고 더운날 추운날 밖에 나갔다 오면 일하는데 지장이 있지...이걸 요구하는거는 선을 넘었네.',
 '조아조아 ',
 '나같아도 미국 간다. 이래도 gr, 저래도 gr...\n여기도 포스코 덕에 잘 먹고 잘 살아왔던 동넨데, 포스코 옮긴다니 머리깎고 난리가 났다가 뭐만 하면 포스코에 기댐...나같애도 진절머리 날듯.',
 '인정할 건 인정해야지.. 비판할 것은 비판하고..',
 '지역 상권 살리자는 취지',
 '강남 한복판이 아니라 서초..',
 '용인반도체 공장 건설때 앞에 모든 가게들 떼돈 벌던데',
 '같잖은 지역 상권 요구 다 맞춰주면 편의 시설, 선진 유통 체계, 우수 식음료체계 다 없어지고 낮은 품질의 음식과 서비스를 불친절한 상인의 응대를 받으면 비싼 가격에 욕먹으며 사용하게 된다. \n\n우수한 대기업들이 기억과 이름만 남기고 다 해외로 가기 전에 주민들과 정부가 정신 차려야 한다.  모두가 대기업 털어서 살아가다 보면   당하다 당하다 결국엔 대기업들이 다 도망간다. \n열심히 해도 방해하고 맨날 정부에서도 괴롭히고 경영권 상속도 못하고  여차하면 감방 가라하고. 이것저것 시키고.   대기업이 애국심으로 우리나라의 이 환경에서 버티는것도 한계가 있다.',
 '하이튼 손해는 1도안보는건 좋지만서도 \n저런건 양보점 해줘야지 \n이기적인것들',
 '이런 지역 이기심 때문에 우리 대기업들이 엄한 외국인들 배터지게 먹여 살리고 있습니다',
 '지주식 주차는 뭐여?',
 '대한민국왜 이러죠?\n 대통령이 자격없는인간이 되니 나라전체가 엉망이네 지역주민들도 하나같이 지생각만하는거 아니냐 완전집단이기주의때문에 나라망하게생겼네 \n아 ~대한민국이망해야득보는거 누구죠?',
 '식당 입장에서 구내식당 싫겠지만 이게 기업애게 하라마라 요구할 수 있는 상황??!!',
 '집단이기주의 완전 양심도 없는 것들이네요. 어떤 회사에 큰 회사인데 구내식당을 두지 않나요?\n악마들이 따로 없네요 정부는 단호하게 처리하시길..',
 '그것도 모자라 노란봉투법 요거요거 기업을 쫒아내는 악법중 악법',
 '도대체 나라가 해주는게 옶어요. 기업발목이나 잡고....',
 '사내식당 금지랑 500조원이 한줄에 같이 있네 ㅋㅋㅋㅋ',
 '대학  기숙사증축할려니\n주변  원룸주인 반발하여  무산ㆍ\n결국  입학생다수는  비싼 원룸으로~~^^',
 '나라가 걱정입니다...',
 '직원들 죄다 도시락 싸들고 오면 자영업자들 뭐라고 GR 할까 ㅋ',
 '헐 겄다',
 '"사내식당금지 " 내용은?\n제목과 맞게 하면 좋겠다.',
 '대학교 주변 상인들이 기숙사 못 짓게 만드는 거랑 같네요',
 '맞는말만 하네 ㅜㅜ망해봐야 알지만 그때가서는 이미 늦었지~~경쟁자들 못따라가죠',
 '님비 현상이 나라 망하게 한다. 무지한 것들은 겁이 없고 경우가 없다.  \n저들이 무슨 권리로 타 회사 경영을 간섭하나 반대로 삼성이 저들 식당 영업 하지 말라. 한다면,',
 '그 기업의 역량이다',
 '동네 상인 동네 상인 하는데... 어휴',
 '삼성과SK모두\n미국으로떠나야합니다',
 '미국 인적자원이 한국만 못해요..',
 '정부는기업활동에 장애가 되는 민간단체, 정치조직, 정부조직은 국가발전에 해가 되니 고발하고 해체하라.\n언론은 공장증설 지연 원인, 결과\n를 완전히 공개하고 타국과 비교 분석하라',
 '걍 영농 기업해!! 아니다!!!\n농사 짓다 새참 가져가면 그것도 난리 치겠구나!!\n답이읍네ㅎ',
 '이게 정치가 기업을 지배하려하니 뭐든 잘못 되는 것',
 '구내식당 아무리 반찬 좋아도 몇달 지나면 물려요.. 인근 식당에서 경쟁력있게 메뉴개발 잘 하면 잘될텐데.. 사실 대기업다니는 사람들은 가격에도 크게 연연하지 않아서.. 좋은 상권이 될텐데..',
 '지금 보니 미국에서 공장 짓다가 뒷통수 맞고 ~~앞으로 지분까지 요구할 수도~~여차하면 꼬투리 잡아서 벌금 잇빠이 물리거나 공장을 아예 뺏길 수도 있을거라는 불안감이 드는데~~~외국 좋아하다 개털린답니다. 국내에서 잘 소통하며 해결방안을 모색해야',
 '삼성은 흥해라',
 '공장 지연ㆍ반대파들의 배후가  있지 않나( 중간쯤 가는 애들) 의심 스럽네요ㆍ항시 보면 1 등2등 보고 부러워 하는 애들은  꼴지는아님  ㅋ 갸들은 아예 관심도 안 가짐 ㆍ중간이  문제임  ㅋㆍ',
 '강원도에 대기업좀 들어오길',
 '기업이 활성화되어야\n그에 종사하는 종업원의봉급으로 가계가 부유해지고\n기업과 가계로 부터 세금을 거두어 정부재정이 든든해 진다.\n\n우리나라 강한 노주, 환경단체나 지역 반기업정서는 기업을 발전하지 못하게 한다.\n결국 이는 기업 공동화로 일자리를 잃고 가계가 피폐되고 종국엔 국가재정부족해 세금을올리고 가계는 더욱 피 폐,기업은 폭망의길\n이게 나라 망하는 길이다',
 '멕시코 마저 해외 기업 자산은 갱단들이 보호 할 만큼 해외 기업 유치를 최선으로 하는 멕시코 정부와 하나 같이 움직임',
 '대신 미국은 보조금 줄테니 지분을 달라고 하죠.\n기업들도 적극적으로 유치하겠다는 지역에만 공장 지어라.',
 '나라가 중요하지 그 밑에 일반 개개인 이게 중요하니? 이래서 안되는구나',
 '지역 이기주의 개인 이기주의 까막눈\n거기다 무식과 어리석음으로 큰댓가를\n받을날이 멀지않았구면',
 '공감 합니다\n맨날 지역 경제 살려야 된다 청년 일자리 늘려 달라 그러면서\n기업 들어 올려고 하면 저 난리 치고 \n그래서 해외로 나가 공장 지으면 또 비난하고\n대체 어쩌라고 나도 갑갑하다\n구내 식당 하지 말란 말도 어처구니가 없다\n삼성 없을땐 어떻게 먹고 살았는데? 어차피 회사원들 다 똑같지\n퇴근하고 술한잔 하고 가끔 회식도 하고\n그지역에서 쇼핑도 하고 카페도 가고 문화 생활도 즐기고 할텐데 보너스로 생각하지 못할지언정\n무슨 물에서 건져내니 봇다리 내놔라도 아니고 \n제비 다리 부러뜨리고 박씨 얻어서 도깨비 방망이로 얼마나 얻어터질라고 쯧;;',
 '왜 대한민국은 제살깍아먹는 짓을 잘하는거냐 ㅋㅋㅋ 이러니 나라가 성장할 수 없지',
 '미국은 세액공제 취소중이에요',
 '재용햄 만세..노란봉투법땜에 화가난다요!\n타국가셔도 응원합니다.\n삼성만세',
 '저런 모든 일의 배후는?',
 '그 발목잡는 뒷배들이 누구인지들은 다들 아시죠?',
 '그 변두리에 회사가 안 들어 가면 일자리가 있을까요?  저녁 장사라도 하면 되지 지역 상권 살리라고 구내식당을 없애라니 도움은 못 줄 망정 훼방은 놓지 말아야지 ᆢ',
 '우리나라 모든 분야가 더도 말고 자기 분야에서 삼성 만큼만 해라 지상낙원을 보게 될거다',
 '삼성전자의 힘. 국민기업이자 세계적 기업. 국가와 국민을 위한. 이들이 글로벌 경쟁력을 갖고 제 가치를 인정받는 것은 참 다행이다. 1000조를 넘어 2000조까지 아니 전세계 최대 기업으로 나아가길...이를 위해 국가와 국민도 동참해야.',
 '이래서 공대출신 아니 최소한 이과출신이 우두머리가 돼야 함',
 '보험만 안하면 되겠네ㆍ보험금 지급이나 제대로 하시오!',
 '극진한 대접을 하는 미국이 공장을 통째로 뺐으려는 심보가 지금 드러나고 있습니다',
 '애초에 나라가 기업에 적대적임.',
 '미국으로 다 나가야 정신 차릴 듯...',
 '저 공장 들어가서 생길 경제적 이익이 얼마나 큰데 별의별걸로 시비를거네 ㅋㅋ 멍청하다 진짜',
 '나중에 후회하면 의미 없는데; 제발 그지근성좀 버리고 기업 좀 살리자;;',
 '거위배 가르기 시전',
 '진짜 좋은 기업들 다나가고 정부한테 의지하는 대한민국이 될까봐 너무 무섭다.. 지금 정부가 원하는길로 갈까봐 무서워',
 '저게 자칭 사회적 약자 k-소상공인 입니다 ㅎㅎ',
 '국민들. 집주인 건물주. 들\n이기적으로나가다가\n경제망가져서\n기업떠나고. 유령도시되어서\n집값. 80%떨어져봐야\n철들까. 그땐 때가 너무늦지요',
 '미국투자해서 지금 어찌됐냐? 으이구 그놈의 미국',
 '중국이 뒤에서 돈주고 지자치 단체장들을 움직인다 안산시? 대한민국 맞냐? ㅎㅎㅎ 지자치제 없애야 된다 세금만 쓰고 임기 끝나면 밪으로 남긴다',
 '기업들 더 일찍 떠났어야함. 삼성일가가 부담한 상속세는 10조. 한국에 있으면 있을수록 기업을 강제로 국가에 뺏기는 구조',
 '유틸리티 인프라 완전 구비되어 도시로 옴겨요 해도해도 너무들 하네..',
 '그 지역에는 국가에서 예산 지원을 끊어버려야한다.이런 기업들이 국내애서 가장 효율적이라고 판단한곳을 선택했을텐데 지역이기주의가 국가발전을 저해하게 하는구나!물론 그 지역에 장단점이 있겠지만 해도 너무 하는것 아닐까?못된 사람들!',
 '사내식당 금지! 와~~\n역시 우리나라는 누가 뭐래도 삼성이다.',
 '한국은 경제성장의 속도에 비례해 국민의 의식 수준은 따라가지 못함.당분간 그대가를 치러야 할 것.',
 '호ㅏ성반도체매점에 판매하는 상품수를 좀 조절해야한다~주위상가가 삼성덕에 임대료가 높은데 공장매점에서 먹으걸 다 사서 나오니까 주위상가들이 빚만늘고 망해가고있어요!!!',
 '작은 소기업이 들어와도 동네주민은 돈 뜯어먹을려고 덤비는데 대기업이 들어오면 돈 많은 회사니까 이참에 왕창뜯어  먹을려고하지  이걸 방지하는게 행정 업무인데 이놈들 또한 같이 먹을려고 하니 문제지.',
 '유튜버보면서 건설적이고 참신한 말씀 \n감사합니다. 산업화과정에서 민노 금속노조가 발목을 잡았다고 생각않나요',
 '박명수 목소리...',
 '구내식당 안만드는걸로 뭐 감면이든 지자체에서 혜택을 주던가 이런 느낌으로 가야 기업도 ㅇㅋ 하는거지 저건 무슨 강도도 아니고',
 '오히려 주변상인들이 돈을 삼성과 하이닉스에 매달 상납하게 해야, 뜯어먹으려는 버르장머리 고칩니다.',
 '지금도 기업 빨아먹는 법안들 올리고 있다',
 '상인들이 착각하는건, 공인이 잘되야 상인이 잘되는데, 공인은 되든말든 관심없고 상인만 잘되야한다며 떼쟁이 마냥 떠든다는겁니다.\n특히, 소상공인 이라 말하지만 정확히는 상인기준이죠. 하이닉스든 삼성이든 기업이 잘 나가고, 직원이 많아지면 자연히 주변도 잘돌아가는데, 저런식으로 딴지나 걸고, 조건달면 좋던마음도 없어지죠.',
 '사내식당금지하지말고간편식&패스트푸드만공급하면해결된다는것을알려야하느니~~~',
 '역시삼성이야 이번에  유심침유출로  통신사 바꿀라고했는데 다시SK제계약해서 감니다',
 '큰일이네 기업들이 빠져 나가면. 우리나라는 뭐먹고사나. 망햐가겠네 욕심에심뽀들이 망하는 이기주의자들  정신차려라',
 '상생이라 쓰고 삥뜯기라 읽는다 ㅋㅋ이게 맞는거 같은데 ㅋㅋㅋㅋ',
 '삼성미워하는자들은 그 뿌리를 깊이 케어내 볼 필요가 있다.  스파이든지 적이든지 그에 포섭된 자들이든지 셋중 하나일것이다.',
 '이재용이 미국가서 제일모직이랑 그랬으면....ㅋㅋㅋㅋ',
 '지자체가 원하고 원전이 있어 전기 공급도 용이하며, 용수, 인적자원 충원도 가능한 대경, 부경쪽으로 제2의 반도체 클러스터를 생각해보면 안될런지? 그것도\n안된다면 sk 하이닉스는 모르겠고\n삼성전자는 미국으로 완전히 옮기는게 답일듯..',
 '지주식 아니고 자주식! 주차장을 법정이상 설치했는지는 좀 찾아보았으면 좋겠어요! 강남 삼성본사 전자 직원들이 수원을 가지 않았나요? 그러니 빈거 아닌가요? 법정 이상 풍성하게 설치한건 아닌것 같은데요!',
 '발목 손목 다잡죠.',
 '그냥 해외공장 짓길 바래요~',
 '!',
 '수도권에만 공장짓는다고 전력시설 짓는 지방은 식민지라는 인간들이 제일 신박했음 저따구로 생각할수도 있구나.. 전력시설이라도 지어주는걸 감사하게 생각해야 하지 않을까?',
 '돈많이 버세요 ㅎㅎ',
 '와 삼성 하이웨이, 삼성아 미국으로 날아가라',
 '저 저기서 일하는 근로자인데 현지식당에서 식사하려면 셔틀타고 주차장가서 차끌고 나와서 현지식당가야하는데 말이 아니라 막걸리죠 왔다갔다 끝입니다. 말이 안되는 억지죠, 그리고 어찌됐던 저찌됐던 근로자들의 생활권이 원삼이나 양지 이동등 다 근처인데 현지상가에서 돈을 쓰는데 도대체 뭐가 부족해서 그러는건지....',
 '무조건잘한다. 직원채용 국가내는 세금  얼마냐 난안듣겠다 화가 나서',
 '지주식이 아니고 자주식 ㅎㅎㅎ',
 '그럼 조리사 분들은 실직인가?????',
 '대한민국은 삼성이 국격이고 품격이란거 잊지마라~\n삼성 사랑합니다~ 난 그냥 아줌마다',
 '젊은이 청년들이 깨어나야 나라가 산다.  비상계엄 사유 내용 숙지하여 자유대한민국 위해 열심히 각자 노력합시다.  조건 반국가세력 척결 헌법수호 자유민주주의 실천',
 '잘했다 그리고 본사 이전이 노조 범죄 처벌한다',
 '쉬운방법 하나잇음 ㅋㅋㅋ 발전소나 오폐수처리장을 가진 지역은 이용료 혜택을 주고 그게 없는 지역은 더 비싸게 받아서 해주면됨\n예를들어 서울의 전기세 여러 생활용수 처리비용을 상당히 올리고 그걸 가진 지역에 떠다주는거지\n그럼 아무말도 안할거다 ㅋㅋㅋ',
 '삼성 현장내 웰스토리 먹어본사람은 알거임  가성비 지린다진짜...',
 '안성이 반대했다기 보다 공무원이 개발 정보 흘렸고 그걸 아무관계 없는사람들이 그 땅 사재기 했고... 정작 주민들이 반대한것',
 '다 죽어가는데 어딘가 크고 있는건 중국덕(?)이겠죠? ㅎㅎ 국가가 사라지고 있는판에...',
 '이재용회장님 최고 노랑봉투법 등 한국에서 기업이끌기 힘드심 그냥 다 외국으로 나가세요 제정신 가진 국민은 응원합니다. 상속세도 이중과세 거의 날강도 수준으로 받아내고 그러니 다들 해외로 가는 겁니다.',
 '삼성 웰스토리에서 도시락 공급해버리면 개꿀잼일듯 ㅋㅋㅋ\n소원대로 구내식당은 없잖아 ㅋㅋㅋ\n아득바득 기싸움에서 이겨줘야지',
 '외부 식당 이용하게하려고 지 멋대로 미국 시민들 두 치 앞을 못보고 공장 잘 되서 월급 받으면 음식ㆍ옷ㆍ 주거 등 경제가 풀릴텐데',
 'ㅋㅋㅋ보인다보여',
 '제발 한국 철수해라.  \n수없이 많은 나라들에서 보조금도 주고 법도 뜯어고쳐준다고 \n와달라고 하는 나라들 너무너무 많다. 인건비도 싸고',
 '우리나라 사람들 돈 욕심에 절때로 땅 안내줘요.\n업체 들어서면 들어선다고 난리\n폐업해서 나가면 나간다고난리\n심뽀들이 글러먹었어요',
 '지방제 당장 폐지해야한다 지방제 폐지안한다면 대한민국 발전 절대 못한다',
 '이상하다. 분명 같은 나라에 살고있는데 온힘을 다해서 나라를 망치고 있는거같아.',
 '그럼 구내에 식당 만들고 분양하면서 상생이라고 하면 되지..ㅋㅋ',
 '정말 걱정이네요 \n노조는 매번 파업에 \n시민 되지도 않는 \n억지 보상에 우리나라에 \n우리 기업이 회사를 \n지어서 일자리 창출한다는데 \n까다로운 규제 등으로 \n힘들게 하네요\n세상이 최첨단으로 바뀌어 \n편리해지면 뭐해요 \n살기가 더 팍팍한데',
 '맞있으면 먹곘지.......ㅠㅠㅠㅠㅠㅠ',
 '삼성 sk 직원 아니지만 우리나라 기업들이 세계로 뻗어나가길 바라는데 정부가 발목을 잡으니 기가찹니다 그것도 천문적인  세금을 받아쳐먹으면서',
 '염치가 없는 사람이 진짜 많아요ㅋㅋㅋㅋ 호의가 계속되면 권리인줄 알아요',
 'ㅋㅋ 전기없어서 안되요 !',
 '이천에 짓지.... 왜.. 안성과 용인에...',
 '세상의 시장경제의 이치는 낙수효과란 것을 잊어서는 안되는 그리하야 돈이 돌고돈다는 진리인데 부자들이 번돈을 똑같이 놔눠 먹자라는 \n지식이 결여된 머리가 나빠 몸이 고생하는 뇌가 모자란 인간들이 세상에 넘쳐나면서 벌어지는 조선시대에도 없던 대한민국의 기이한 현상 결국 이나라는 7080년대로 후퇴할 것',
 '이재용 삼성 최고 자랑스럽다',
 '구미로 갔다면..',
 '딱 군대 지역주민들 처럼  하는구나',
 '정치인들이 총알받이해서 막아줘야하는데...그거 해준다고 또 삼성에세 손 벌리고잇을겁니다.ㅋㅋㅋ 돈내놔~짓게해줄께!!!',
 '삼성.현대최고..두그룹이한국을만들엊다',
 '대기업회장님 존경합니다\n삼성 엘지 SK 현대 대기업이 잘돼야 \n대한민국 경제가 살고 고용창출효과도 극대화\n대한민국 사람들 진짜 너무한다\n대기업직원이 몇명인데 사내식당 밥을 못먹게 하는 주변상인들\n제정신인가??!!!!!! 니들도 대기업이 내는 세금으로 복지혜택 보는데도 이따위식 !!!!!!좌파정권은 기업을 죽이는처사 진짜 기업하기 힘든 대한민국이다 망해가는 대한민국',
 '우리나라는 개인주의가  너무 강해요  남이야죽어도 나만  잘살면 된다는생각',
 '관이 적극적으로 투자환경조성에 나서야 또 삼성이나 하이닉스 역시 유치 지역주민들 피해보상을 회피해선 안됩니다.\n안성에 lng시설이라니 주민들 반발은 당연한겁니다. 당신 아파트옆에 그런 위험시설이 들어와도 쌍수들어 환영할겁니까.\n한국 반도체클러스터는 미국이나 일본처럼 땅덩이크고 소지방 인구밀도\n낮은곳 아닙니다. 절차복잡하고 비용발생하는건 당연한 겁니다. 관민기업이 잘 협력해야 합니다.',
 '우리 동네로 와 주지...',
 '판교 근처 식당들 보면 퀄리티가 김밥 천국이랑 비비거나 못 하는데 가장 싼 음식이 1.2만원임 ㅋㅋㅋ 양심 터졌어',
 '한국 미국 중국 일본 동남아 아프리카 유럽 다 비슷해요 환경규제 지역 정치인들  삥뜯기 산업 스파이  세상 사람 다 똑 같아요  지역 이기주의 주변 상인들  삥뜯는  제발 기업이 살아야 나라가 산다. R&D살리고  기업들 살리는 정책 규제 정리해주시길 ',
 '진짜 한심스럽다.',
 '어떻게 저런 요구를?... 한심한...',
 '지역사회에 도움을 주는 뭔가가 있어야죠. 지역주민을 직원으로 쓰는건 아니잖아요.  세금을 낸다면 그걸로 지자체가 먼가를 지역사회가 느끼게끔 해줘야죠. 집값만 올라가는건 사실상.가난한 지역주민은 혜택과 멀지요..지역주민들과 소통해야할것 같아요',
 '이런 걸 보면 뭐가 잘못되도 한참 잘못됐다.\n쪽박을 차고나서야 후횔할 나라와 국민성~\n우리나라는 기업이 일꾼이고 가장인데~',
 '행정 미숙에 님비가 아니라\n애초에 수도권에 사람들 많이 사는곳에 원주민들 무시하고 어떻게든 수도권에 지으려고 하니까 그렇지; 안그래도 몰림도 심하고 원주민도 많이 사는곳에 하는거면 당연히 대화와 소통으로 풀어가는거다, 미국은 기업 환영한다고? 거기는 땅덩어리 넓고 남는땅에 지어서 그런거지',
 '무시할만 하니깐 무시하는거지....ㅋㅋㅋ',
 '불과한10년만에 나라의 분위기가 완전 바뀌어버려서 반기업 문화가 만연해져서 부자는 적이고 성공한기업가를 적대시해버리니 어지간히 성공한 기업은그냥 탈한국하는게 일상화되어버릴거 같네요 결국 황금알을 낳는거위의 배를갈라버리려하니 한국은 다시 예전의 가난한 경쟁력없는 빈곤국가로 다시 돌아가버리게 될거같네요 이나라에서 어떻게 살아가야할지',
 '사내 식당 찬성!!',
 '나라를 살리는 대기업 그런뒷통수를 정부가치는중.. 스파이들이 너무많다',
 '기업 망하든 말든 잎단 내게 돈만 주면 된다니까.',
 '한때 세계시장을 장악한 유럽기업들이 어떡해 그나라을 떠난는지 한번 알아보세요\n우리도 이제 그 길을 가고 있습니다 멀리 볼것 없다 지방들 기업없어서 망해가고 있다\n이것이 지방에서 끝날까? 아니다 이제 지방탈출에서 국가 탈출한다 기업인들 죽일듯이 법와 세금으로 쪼으면 \n누가 기업하고 경제을 발전시킬지 모르겠다 결국 정치인들이 표받기 위해서 복지정책 난발하고 그돈을 만들기 위해서\n기업이랑 부자들 쪼으면 결국 나라는 망하는 것이다 한국은 망해가는 단계에 들어갔다 \n언제까지 기업들에게 애국심으로 한국에 붙어있어 달라고 말할 것인가? \n사람의 마음은 똑 같다 내가 도망가고 싶으면 기업가도 같은 마음이다',
 '학교 급식소도 짓지마라고 하지 그러냐?ㅋㅋ',
 '지들 입맛대로  하겠다네!',
 '사내 공장을 이용하라고 시위하는 사람들은 지역주민이 아니라 공사장 따라다니면서 함바집 운영하는 사람들 일 것 같은데.\n지역주민들은 원래 공사 인부들 대상으로 장사하던 곳도 아니고 지역주민들이 찾아주니 상관 없을텐데\n 함바집은 딱 공사인부들한테 단기로 뽑아먹어야 하는데 그게 안돼서 시위하는 듯',
 '대기업 몆개 업는데 \n발목만 잡고 있네',
 '전세계어딜가도 천연계곡에 상깔이놓고 장사하고 자기네꺼라고 못들어가게하는나라 이기주의적인나라',
 '제일 위험한건 기업을 옥죄려고만 하는 정치 세력과 특히 기업을 돈벌이 수단으로만 생각하는 환경단체들입니다. 저런것들이 없어져야 비로소 기업이 제대로 발전하고 경제가 성장할 수 있는데 앞으로도 이런식이라면 역성장은 피할 수 없는 결과일거 같습니다..',
 '이민가라는거 장난같지?\n진짜 가라\n전쟁나면 우리나라 무조건 진다',
 '국가 시스템개혁',
 '공장을  경기도에만 짓을려고 하냐  아래지방  전남 여수산단 율촌공단 옆 순천 서면공장단지 같은 부지도 많은데   경기도만 고집하는 기업도 문제지',
 '기업은 국민을 먹여살립니다. 정치는 말썽만 일으키고 돈만 뜯어가여',
 '간접 경험이 이런건가 봐요. 시장조사를 철저히 하라.잘못 판단했다고 생각들면 빨리 접어라.',
 '발상이 놀랍다\n그럼 세탁기도 못만들게 햐 죄다 빨래방으로 들고 가게',
 '삼성 및 대기업들 응원하셔야 합니다 우리나라 대기업 없어지면 나라 폭망하면 후진국 전락 한다',
 '서울대병원 구내식당 가서 뭐해유',
 '파주에는 엘지로 있습니다. 엘지로~~',
 '남의 것이 왜 탐나는지 모르겠다.참 칼만들면 도둑이랑 뭐가 다른가?',
 '삼성을 대체 누가 욕 을하냐\n절 을 해야지!!!양심 좀 있어라!',
 '한국 탈출은 지능순',
 '정치만 삼류가 아니라 그곳에 삼성 때문에 먹고 살 안성시 주민도 삼류인가요, 청년들 일자리 많아지면 도시가 발전합니다. 도시가 발전하면 그곳에 사는 안성시 주민들도 여러가지 혜택이 커질 것인데 당장 피해만 생각하거나 근시안적으로만 보지 마시고 누구를 위한 반대를 하는지 잘 생각해봤으면..... SK, 삼성등 대기업이 한국을 발전시키는데 크게 기여하였습니다. 이런 대기업을 항상 응원하고 더 발전하도록 규제를 풀고 제도와 환경도 뒷받침해야합니다.',
 '그러지 말고 강릉으로 오세요.',
 '주사파 정치꾼 놈들이 대기업을 재벌로 타도의 대상으로 생각해요. 이 마인드를 지역주민에게 심어주고 있죠. 미친 놈들입니다.',
 '1찍이 만드는 신세계 ㅋㅋㅋㅋ',
 '그지 땅이 작긴 하지. 허허 벌판에 짖는다면 알빠노지만. 우리나라는 어디다가 지어도 누군가의 동네지',
 '안성시: 돈 더 내. 기부 안해?',
 '대통령 국회의원 국민이 썩은내나는 대한민국 벙법이 없다',
 '더 지방으로 내려와도 될듯한데,, 수도권에 뭐 그리들 모여 있는지....\n\n\n그러고 보니 다 경기도에 모여들 있네....',
 '지자쳬 웃기는 동리 주민들 때문에 못해먹겠다',
 '한국떠나는걸 추천드립니다 우리나라는 답이없습니다',
 '삼성 응원합니다',
 '정피인 봐야할 영상이네',
 '반도체공장 인근에 화력발전소나 공업용수를 쓰는 것을 쉽게 말하고 경제효과 위주로 말하는데 동의하기 어렵다. 막대한 공공시설 투자인 것이다. 우린 정전이나 단수를 잘 경험하지 않는 전력과 물이 풍부한  나라여서 쉽게 생각하지만 이명박정부때 정전사태나 미국 텍사스 정전사태 찾아보거나 대만 가뭄 문제를 찾아보면 간단한 문제가 아니다. 주변 상가나 지역 경제 이게 본질이 아닌데 쉽게 소상공인들 매도하지 마라.',
 '그것이  청치하는  놈들 때문에  우리 나라는  망했어요',
 '집단이기주의.\n극우발상.\n무엇이 미래이고 내 자식을 위한 것인지보다 당장의 내 주머니에 몇천원,몇만원 더 들어오는것을 추구하는 우매함',
 '부동산에 춰한 나라의 결말',
 'ㅋㅋㅋㅋㅋ',
 'ㅋㅋㅋ영상이후 리짜이밍이 대통령이 돼버림 ㅋㅋ',
 '우리가 망하고 있는 나라야?몰랐네 일안하는 젊은이들이 망하는거지 나라가 망하니 자신의 문제를 다른데서 찾지말고 치열하게 살아라 온실에서 컸으면 그 에너지로 튼실하게 살아라 신체는 건강하지만 정신은 나약한 청년들아',
 '우리나라 대기업은 국민들 쥐어짜서 만든거다. 미국가서 아양떠는꼴보면 참 사대주의 오진다.',
 '눈앞에 이익만 쫓는건 어리석다는걸 좀 알았으면 좋겠다. 특히 식당 음식에 정성 쏟으면 사람은 찾아온다. 맛과 질은 떨어지고 가격만 올리니 어려워지는거지. 그럼징징~ 정신들 차리기를~ 노조들 기승에 대기업들이 해외로 발길을 돌린 부분도 있을거임.',
 '지방으로 오세요\n삼성 . 하이닉스. 지방에 일자리좀 만드세요. 너무 경기도에 너무 몰빵됨.',
 '타이밍이 지금은 아님',
 '칭창  아니고 아부같다',
 '구내식당을 이용하라니...ㅋㅋㅋ무슨 자격으로?ㅋㅋ 참 우리나라 기성세대들은 주머니에 돈좀 쥐어주면 상대를 호구로 보고 가격올리고 등쳐먹을생각밖에 안하는거같다',
 '오히려 가성비로 밀고 나가면 될 거 같은데\n사람들이 사내식당이 질리면 가끔은 밖에서 먹기도 하는데 가성비인데 맛도 있다? 단골로 만들 생각은 아예 못하는건가',
 '식당은 찬성, 어디 현장을 가던 구내 함밥집 진짜 최악임. ㅋㅋㅋ 전부 현장 쪽이랑 커미션으로 들어온 사람들이라 무조건 싸게 하다 보니 ㅋㅋㅋ 개판임.',
 '기업이 잘 되어야 민생이 돌아가고 민생이 돌아야 상권이 삽니다, 상권이 산다는 의미는 개인이 여유 있다는거고 너무 여유가 있으면 잡 생각이 많이 생기고 나쁜 짓을 하게 되어 있습니다 . 결국 또 원상 복귀 되는 것입니다, 직장인은 실업 되고 돈을 쓰지 않으니 상인들은 죽어가고.',
 '삼성은 세금젤많이내고, 기부도많이하고, 진짜 애국기업입니다..... 우리나라는 산유국이 아니라서 기업의 세금이 국가를 운영한다해도 과언이 아니라봅니다... 기업이 잘되게해야 나라가 흥한다!!!!!  정신차려!!!! 정치인/국민들!!!!!',
 '지역주의와 개인주의가 나라를 망치고 있네요..기업이 잘되어야 국가가 흥하고 국민 개개인이 잘 살게 되는것, 어린애들도 아는 것 아녜요? 정말 한심한 사람들입니다. 지금의 어려운 난국이 남의 탓이 아니란겁니다. 지금이라도 구국정신으로 뭉쳐야 합니다!',
 '용인은 버려야 해...',
 '식당주인들 어이없네',
 '트럼프는 미국에 공장 짓는 대신 삼성전자 주식 가져가고 경영에 참여해 첨단기술 뽑아가 미국기업들에 뿌린다는데 그런 말씀 하시는지? 결국 삼성전자 미국 가고 난 후 몇년안에 뽑아먹을거 다 뽑아먹고 망하게 할려는건데 ',
 '우리나라는 고생도 많이냈는데 너무이기적 정치가잘못',
 '기업이 있어야 일터가 있지\n미련한 인간들',
 '우리나라는 쫄딱 망해야 합니다. 비참한 현실을 몸소겪은 다음 무엇이 소중한지 배워야 합니다.',
 '평택 보세요 아파트값 2억정도 하락 빈상가로 인해 손님 뚝 ~ 세사ㅇ에 옆집 안성주인들이 미쳤구만~',
 '지난 10년 전의 삼성과 TSMC 규모와 지금의 삼성과 TSMC를 비교한번 해봐라.   세계적 기업과 지도자는 그런 감상에 젖어 있을수 없다는 거다.',
 '삼성 잘 하는거 그걸 누가 모르는게 아닌데요 ㅎㅎ\n저도 삼성 응원합니다. 그런데 에스오디 님은 \n삼성이 잘하는것만 80% 못하는거는 거의 10% 정도만 \n방송을 하시니까 편향된 잣대라고 보여 지거든요 ㅎㅎ\n삼성은 잘하는 국가적인 보은 사업 많이 하는거 압니다. \n그런데 잘못된것도 많아요 \n방송의 편중을 고르게 하심이 좋을듯 보이네요 ㅎㅎ',
 '좌측은 나라가 망하든 기업가들이 떠나든 표만 잡으면 된다 생각하나',
 '식당들만 기업 피 빨아먹는게 아니라\n대기업도 하청업체 지독하게 피 빨아먹는게 똑같음\n그냥 국민성이 그런거지 ?',
 '그럼 한강 상류에 반도체 공장 짓는다고 하면 서울 사람들 난리 날텐데....안성같은 경우 용인에 하이닉스 유치 반대했던 이유가 식수와 관련되어 있는데(수혜는 용인이 보고 거기서 내려오는 물은 안성시민들이 쓰고 혜택은 없고)...그리고 수원에 삼성과 용인의 하이닉스는 비교자체가 안됨 수원은 전철과 기차가 지나가며 경기도 지역 중앙에 위치하는 지리적 요건이 있지만 용인 처인구쪽은 그냥 시골임 하이닉스가 들어온들 인프라 부족으로 기존 주민들은 수원과 다르게 어떠한 부동산 혜택같은걸 볼 수 없음 이천에 있는 하이닉스만 가봐도 그냥 아무것도 없음',
 '우리동네도 시맨트회사 어떻게 하면 이장.환경시민.개발의원 등등  어떻게하면 돈을받아낼지  인간들 많아요 지뿔도 모르는 인간들이',
 '정작 나라 망하게 하는 주범이 누군지 생각해 보면\n가장 혜택을 많이 본 놈들 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ',
 '트럼프가 좋아하겠네요 ㅋ자국에다 지어주면 각종 해택준다는데',
 '빨리 미국 가시길.',
 '주식으로 따지면 급하강 포인트지~묙심은 욕심으로 망한다~~~망해라',
 '국민이 바보다..정치인들을 국민들이 모두 선거에서  뽑았으니  그책임이 국민에게 돌아오겠죠.삼성,sk 중국기업에 매각되면 한국인들은 중국기업에서 일해봐야 정신 차랄듯..그렇게 되면 때는 늦었음.. 기업도 없는데 무슨파업이나 하는노조..',
 '동네상권 키울생각 말고 구내식당에 취직하면 되지..어떻게든 빨대 꼽으려는 인간들은 그냥 망해야됨',
 '우와 답답하네 ᆢ',
 '중국하고 비슷하군.. 글로벌 대기업 철수하게되니 일자리도 없어지고 부동산도 똥값되던데..',
 '주변 식당들 밥값 올리지 마라',
 '지금일부사람들은  중국이 한국 설계하고있다라는 소리까지 나옵니다 다른나라들은 반도체 나라에서 밀어주는데 우리나라는 왜 못내보내서 난리인지 삼성도 외국 나가서하면  당분간은 이득이지만 핵심기술 유출등을 이유로 반도체만큼은 해외로 안나가고 국내에서 버티는거같은데  왜 정부는 공산당놈들과 같은생각을하고 규제해서 기업들을 힘들게 하는지 이해가 안감 중국이 판짜놓고 국내에 있는 반국가 단체나에 오더 주고 있는거 아닌가 싶음 딱 그림이 보이자나 스마트 폰은 이미 베트남 인도 생산임 ~',
 '지 혼자만 잘 살라는 시대가 있으니 이렇게 망하는거임',
 '대기업욕하는 사람들은 정치하는 놈들에게 다 배운거다. 우리나라 발전 뒷다리 잡는 자들',
 '2020년대판 님비현상.',
 '한국은 어쩔수없음 적국이 간첩보내서 조직적으로선동하고 성장을 저해시킴 폐쇠적이고 강압적인 상대로 자율적이고 개방적인나라가 이길방법이 없음',
 '삼성,sk등이 동네 작은 공장 크기냐?\n그 넓은데서 밥먹으러 밖에 나갔다오라고?\n그럼 왔다갔다하고 밥먹을람 식사시간 2시간줘야한다.\n뇌가 있고선 저런 생각을 못하지.',
 '자기네들이 5900원에 준하게 런치 짜서 팔면 될꺼아녀. 그럼 직장인들이 런치 먹으러 가것지. 일끝나고 저녁에 배고프면 저녁도 먹으러 갈꺼고.',
 '국내에 삼성처럼 사회환원 많이하는 회사는 없습니다. 미국 일본 중국 기타 어느 나라라도 국가가 나서서 보호하고 도와줄텐데 정부 하는 꼬라지를 보면 기업이 망해야 괴롭히지 않을듯~',
 'ㅋㅋㅋㅋㅋ 도와줘도 시원 찮을판에 다 망하고싶나??',
 '밥줄 스다 날새겟다',
 '원삼면 가보셨나,\n원삼면에 가면 한참 토목공사 중인데,\n인근에 식당이 변변한게 없다.\n골프장에 접해있어, 부자?들이 가는 나름 맛집이 있는데, 그 곳은 노동자 보고 장사하는 곳은 아닌것으로 보인다.\n식당 몇 군대 가 보면, 평택 고덕의 나름 질이 낮은 식당보다 쬐끔 나아 보이긴 하는데,\n노동하는 일꾼들이 먹기엔 터무니 없어 보인다.\n그나마 식당이 몇 군대 없다.\n저렇게 동네 팔면서 지역 식당 이용하라고 외치는 이들은 누구인가?\n외지인 아니면, 외지인과 결탁한 지역민으로 보인다.\n원룸은 어떤가 천에 90이다. 조건은 한 사람. 한 사람 추가되면 월세가 30%에서 50% 추가된다.\n지금도 원룸이 없어서 일하러 가려는 사람들이 많이 포기하고 있다.',
 '저렇게 잘해줘요 노조가..',
 '하나는 알고 둘은모르내.  지벙내려가면 모시려고 줄섯다. 왜수도권 문재만고 돈만이드는대에 공장지의려고 난리인대\n지방에내려가라',
 '한국을 뜨는게 답임...  한국 대기업들 모두 미국으로 오세요....  그 지역 사람들은 감사하게 생각할거에요...  기아차 공장 지역 사람들은 모두 기아차만 탄다고 하더군요...  그리고 공장 지어서 고맙다고 까지 하는 인터뷰까지 하고..',
 '거제에 지어요',
 '물많고 전기남아돌고 땅 값은 거의 공짜인 새만금으로 오세요!!!',
 '그냥 삼성 SK는 한국을 떠라',
 '어차피 구내식당 밥 먹다가 종종 나가서 먹고 그럴텐데ㅋㅋㅋ\n너무 근시안적이고 이기적이네요.',
 '한국은 시스템 자체가 망하는 구조예요.\n대기업이 해외로 나가야 살수있는 구조로 결국은 많은 일자리가 사라질수 있는 상황이라 생각합니다.',
 '7775 댓글 달았는데, 그 댓글은 다 어디갔낭?',
 '김포로 와\n쓰레기 매립지만 주려 하지 말고 \n김포공항 강화도 땅뺏어 가지 말고\n반도체 단지 좀 주라\n두손들어 환영이다',
 '야 웃기네\n3000명이 한번에 쏟아져 나오면 40분안에 전부 밥먹일 식당이 있긴하니....\n말도안되는 헛소리들이야...',
 '군부대 빠져도 상권 다죽고 난리더만 군바가지 등쳐먹기.. \n대기업앞이라고 다를까  동네상권 눈치를 왜보나.. 상권에서 경쟁력재고할 생각은 없고 주서먹고 싶어서 드글드글',
 '지방도시살리기 일자리창출 새로운인프라형성 뭐하나 손해볼게없는데  여기서 제일 무서운건 선진국들이 왜 삼성,하이닉스 어서오세요 하겠어요?세계제일의 기술공유가 가능하니까 레드카펫깔고 환영하죠',
 '죄명아 기업죽이면 국민이죽고 노조가 설치면 기업이망한다',
 '능력안되면 자영업 제발하지마라 망했다고 세금축내지말고 남믿에 일해',
 '이런거 볼때 마다 우리나라 정부는 세금은 기업들한테 다 받아처먹으면서, 기업이 투자하는데 교통정리는 커녕 행정절차 드럽게 지연시키고, 정작 그 시간에 포퓰리즘에 묻혀 돈이나 풀고 정체도 모르는 민간단체에 지원이나 할줄 알지….대한민국 정부는 정신차리세요',
 '하긴 소각장이랑 쓰레기장 각종 중금속 업체는 쉽게 승인되고 주민반발도없고.. 삼성 하이닉스만 타겟이 되버렸어.. 문제는 돈 내놔라는 심리라는것이지',
 '오면 안됩니다',
 '그래서 미국이 쇠고랑 채운거냐',
 '돈도 안되는 식당 자영업 왜케 많이들 하는지 자영업자 이들이 지역 대기업 유치 반발하는 빌런',
 '이사람 삼성 광고하네',
 '시기가 코시국이네? 아 규제할만할때였나? 이래서 선거잘해야함 \n정치이념떠나서 나라경제가 어려울땐 경제학적으로 국가발전을 시키려는 쪽이 정권을잡아야 극복할텐데 대기업만 작살내고있으니 \n내가 그룹사총수여도 한국뜬다',
 '우리 나라는 뭐 한다하면 어떻게든 돈 뜯어낼려고 우르르 몰려가서 떼쓰고 드러눕고 그럼. 근데 그게 한번 허용이 되면 사람들이 학습을해서 당연하게 여기고 계속 악순환반복. 이기주의 끝판왕.',
 '외국에 다뺐기면 우짜노',
 '지무덤 파고 있다는것을 모르니깐 욕심이 목구멍까지 차니깐 하는짓꺼리지',
 '역시 삼성정자',
 '양평과 서울등 어느곳을 잊는 초전류 트라이엥글 산업이 있었는데. 전자파 때문에 우리 후손 다 죽는다며, 양평에서 극구 반발.. \n교류는 서로 엉겼을때 상호 간섭으로 인해 증폭되는데.  이를 초전류로 만들어 서로 순환되도록하면,  영화 아이먼맨 처럼 삼각 트라이 앵글 에너지 장치가 만들어 진다. \n이를 크개 만들어 삼각트라이 앵글  전류 보전및 증폭라인을 크게 만들면,  효율성이  아주좋다.  \n걔다가 . 라인 근처의 사람들에은 전자이온이 흐를 때 자동충전되는 기기를 만들어 주면, 무료 충전이 될 수 있다. \n30년이상 고전압 전기 기술자로 사신분의 이야기를 들어보면, 전자파가 있었으면, 자신은 벌써 죽었다고 한다.. \n전혀 문제 없다고, 확답을 하는데. 지역 이기주의와 땅값의 문제등의 이해 관계가 심각하다. \n\n니네가 우리동내에 뭘 지으려면, 일단 우리를 먹여살려야 한다. 라는 개진상 마인드가 전쟁을 일으킨다.',
 '참 해도해도 너무한다 한국은 망할거야',
 '이제는 새만금 이전 이 지랄 하는 중...',
 '창업주 이병철의 고향 의령에서 고따우 짓 하다가 딴동네  대박 나는거 구경만 하고 있다 ,주민들 날파리짓 하면 떠라',
 '중국 북한 일본 스파이는 대한민국 경제 사업을 무산되게하고 그냥 밀어붙여라\n주민들 무시하고',
 '주제파악 못하는 인간들 참 많아요/\n이런인간  있는 마을들 다 망해야 그때?\n홈프러스 바라 발악하 노조 ?\n문딷고 더나니 그제야 무릅긇고?\n뻐스는 이미 지구를 떠난는데요,',
 '사토시다',
 '그렇게 날뛰다가 삼성,하이닉스 국내를 떠나야 그때 아이고 하겠지.\n지방에서 수도권으로 옮긴 기업의 판단도?',
 '국민성이 4류 5류인데\n매일 데모하고 농성하고 이기적인 국민들 때문에\n발전할수가 있을가??',
 '이건 아니지',
 '구내식당이용하는이유는 만명넘는사람들이 우르르 점심시간에나가면 나가는데20분 식사주문 기다리고 먹고오는데 1시간 들오는데20분이라 회의많은 회사에선 직원들다 굶게될꺼임  그래서 사내식당운영해주는거임 요즘은테이크아웃으로 세입먹고 점심때우는사람도 많음 사내식당운영은 이건희가 너무잘한거임',
 '미국에 오기을 잘 했다',
 '제목사내식당금지그런데왜경제이야기',
 '지자체가 없어져야됩니다',
 '이건희회장님뵈면박정희대통령님새악나괜히눈물이납니다\n살아있는박정희대통령님을뵙는것같은기분은왜일까생각해보니눈물이납니다그리워서그리워서내아버님같아요저는부산사는70할매입니다',
 '노동일하는 사람들은 가깝고 싼밥먹지 공사현장에서 나가는대 시간도 오래걸리고 밥먹고나면 점심 시간끝이다 당신들 돈벌자고 노동자들 힘들게하지말어라',
 '진짜 배가 불렀다\n진짜 러시아로가라',
 '인간들이 그냥 너무이기적임',
 '삼성이 외부인 주차를 조건부 허용하는게 무슨 사회공헌처럼 말하네.. ㅋㅋㅋㅋㅋㅋㅋㅋㅋ',
 '주차장으로 무신 좋은일한다고 하냐 ㅡㅡ',
 '삼성 나라 버려라 그래야 기업이라도 살지 나라랑 같이진흙탕에 빠질수없지',
 '특히 현대 노조',
 'ㅎㅎ  간철수 지지했던 모양이군..',
 '아직도 한국에 돈은 싫다면서 누구보다 돈을 좋아하는 이중적인 착한 사람 컴플렉스 아재들 꼰대들 많아서 싫다\n착한 척 옳은 척하면서 민폐 끼치는 인간들',
 '삼성은 장학제도에도 엄청 기여합니다\n또한 비 전공 IT인재 육성 사업등 기업이 나라를 운영하는 정도예요\n\n이재용회장님께서 대통령 출마하셨으면 \n좋겠네요\n\n아주 이상적인 나라를 만드실 것 같은데',
 '역시 한국의 발전은 지역주민이 망치네. 이게 민원의힘이 너무 강해서 그럼… 기업발전서부터 사람의 생명까지 민원하나로 좌지우지 하는꼴이 말이됨?',
 '요거 하나만큼은 애국기업이라뇨..우리나라 경제의 대부분을 기여하고 있는데.  일부러 내려치려고 저런 표현을 쓴건 아니겠지만, 요거하나만큼은 이라고 말하기엔, 자원이 대기업이 전부인 나라에서 요거하나만큼은 이라니 ㅋㅋ',
 '그래서 불법 체류자로 다 잡아넣으려고 했나 이 엔병할 게트보야',
 '삼성 대박',
 '아니 노동자들에게 기업이 복지제공해야한다고 그렇게 난리난리 쳐놓고 \n이제와서 복지중의 가장 중요한 부분인 사내식당을 이용하지말라구요??? 진짜 미쳤네요. \n이게 다 미디어가 싼 똥의 결과입니다. 한참동안 그렇게 노동자위하는척 재벌까고 대기업 무작정 까면서\n내놔내놔하는 노조들 편을 들어주더니 이제와서 저런 소리하는 주민들은 왜 방치할까요? \n한쪽에서는 자영업자 살리자고 민생지원금풀고 한쪽에서는 노동자를 위해서 대기업이라면 온갖 복지를 다 해줘야하는게 당연하다고하고 뭐 어쩌라는건지...',
 '저 기업이 없으면 아예 이용자가 없을 건데 뭔 소리여.자기네가 저 기업이 차리라고 해서 식당 차렸나.뭔 이상한 욕심을 부리나',
 '다 버리고 떠나야 되겠네',
 '어차피 지방소멸은 가속화되고있다.\n정부가 그냥 지방에 인프라 구축해주고 지방이 먹고살수있게 해주면 좋겠다.',
 '지역 주민때문에 미국에 공장을 짓는다고? 이 뭔 개소리야… 세금때문에 가는거지 그리고 일본 TSMC도 구마모토시와 서로 싸우고 있는데, 참…',
 '그래 그래서 잼프가 기업하기 좋은 나라로 만든다자나. 왜 지역 상권의 미친 이기심을 들먹이냐? 그리고 세금은 원래 내는 거인데 그건 내기 싫어?',
 '전주로 와라 용인은. 배가 불러 그렇다 전북으로 오면. 대환영이다.',
 '이정도면 문재인과 코로나가 자영업자들 소독한거임',
 '정치만 없으면 잘사는 나라는? 대한민국 입니다',
 '민원만 넣으면 다냐...미친...ㅋㅋㅋ 그 많은 사람들식사를 다 어떻게 감당할껀데...그리고 그 많은 사람들이 언제 이동해서 밥먹도 다시 들어감...피곤하게 만드네...바가지 할라고 별...ㅡㅡ+',
 '지역 이기주의이다. 함께 협력적으로 ㅈ2ㅣㄴ행해야 할 판에 이기심만 가지면 안됨. 주민들 각성해야 함.',
 '그냥 전국에 원하는 곳이 있으면 그리로 가면 안될까요?\n님비도 문제이긴 하지만 일본처럼 5년공사를 20개월만에 끝내는 것도 좋아보이지 않네요.',
 '더이상 늧추면 안될듯 새만금으로 가라',
 '에이 정말~~~ 위생이나 가격부터 챙기지... 대도록이면 식당 안간지 몇년됨.  드럽고 불친절하고 ... 사는동안 갈일이 없을것 같아요.  당장 들어오는 쩐 챙기느라 나라 경제 안중에 없는 노조들이나 장사꾼들 땜에 정말 정 떨어져.',
 '잘해줄때 잘하길....좀만 도와주면 뒷통수 칠생각하고 관광지도 그렇고 자본주의 자본주의 거리면서 금액 겁나 올려받고 \n우리나라 사람들도 은근 앞뒤 다름 앞에서는 삼성최고 이러고 뒤에서는 욕하고 인간은 더러워..',
 '근데..공장을 꼭 인구많은 도시에 지ㄴ니 ...미국에서도 그러면 마찬가지아냐.. 텍사스 사막에 짓는거랑 다르지.. 지방에 지어바라 ... 수도권말고.. 좋아할기다...ㅉㅉ 업체도 지역주민도 다 생각좀바꿔야지...',
 '사내식당 금지인데 뭐가 주민들이 미쳣단 소리인가  주변 식당들이 활성화될건데',
 '마국가서대접받아보시든지',
 '주믹들이진짜반대하나요?왜?',
 '정부가 당근과 채찍을 잘 사용해야되는데 무조건 버티면 돈준다는 인식을 박아버려서 저런거 보면 답답할뿐입니다 지역상권이고 나발이고 다 자기 이름걸고 사업하는 사람들일텐데 왜저러는건지',
 '99% 맞다라고 볼께!그런데 1%가 99%보다 더욱 중요한것 대한민국이 망한다고?\n틀렸다,지금도~~ 영원토록 앞으로 10,50,100년 200년이지나고 천년이 지나도\n한국은 1등나라다,아니면  내목 내놓을께?! 한생명이 천하보다 귀하다고 하늘은 말씀하셨지,',
 '왜 외국애들이 자기들 나라에 공장을 지으라고 하는것일까요?   자기나라에서 돈쓰고 고용하고 하라고하는것이지요. 그러니 대한민국에서 장사하면서  동내 식당과 상생하는게 뭐가 나쁜진 모르겠 습니다.   자국민들끼리 서로 먹고사는게 남 나라 사람들 먹여살리는것보단 좋을듯 합니다.',
 '지역식당들도 제발 철좀들지\n점심 밥. 구내식당 먹고\n그지역에서. 인프라 구축되면\n슈퍼든 저녁외식이든\n돌아갈텐데\n지자제도. 멍청해\n기업을. 유치해야지\n못버티고. 나가게하면\n지역도. 국가도. 국민도 다망하는데',
 '나라가 망해야 ..국민들..정신차리지..\n근데..역사를 보면.아무런..죄 없는게..국민인데..\n정치하는 인간들이..나라 말아 처먹고 잇지..\n국민은 늘..그 자리에 있는데..\n정치하는 인간들이..늘..나라 팔아 처먹고 있지...',
 '망하는게 답  그동안 배가 너무 불렀지',
 '5:10 망하고 있는 나라잖아요? 아무리 그래도 그렇게 말씀 하시면 안돼요.\n.\n.\n.\n.\n.\n.\n.\n피해의식에 쩔어사는 국민들과 치매노인들만 정치를 하기 시작한 순간 부터 이미 망한 없는 나라이죠. 없는데도 나라라는 표현 하는건 왜인지 아시죠? 삼성제국ㅋㅋㅋ 아 진짜 이분 뭐하시는 분인데 팩트만 딱딱 꽂아주지? 알고리즘으로 우연히 보다가 소름까지 돋네ㅋㅋ',
 '다른나라 에서 사업한다는건 한순간에 망할수도 잇다는걸 잘 아셔야 합니다~~~네이버 일본,중국 삼성 하이닉스 대기업들~미국도 지금 여러모로 문제가 계속 생기고요~~~  다른 선진국도 노조 및 환경법등등 쉽지 않습니다...물론 한국 자국민들의 삼성 엘지 하이닉스 등등 대기업들 모두 자랑스러운 우리의 회사들이지만......다른나라에서  처음에는  이리저리 꿀발라  오라고 하지만~ 해외에서 순간 망하는건 순간 입니다...',
 '삼성괴롭히는거 우리나라사람아니라고 하면 전부 이해됨',
 '??? 그럼 식사시간 한시간인데 언제 나가서 먹고 들어옴? 휴게시간에 밥 잘 못먹고 낮잠못자서 사고나면 주변식당들에서 책임지나??',
 '왜 지연 됐냐면\n윤정권때 아무생각이 없었어',
 '삼성이 얼마나 갑질을 하는지 모르는 순진한 청년이야',
 'ㅋ',
 '대한민국 쯔쯔쯔 아니지 소한민국도 아니지 지금 세계정세를 보라\n얼마나 불안정한가를 그리고 우리는 이미 서서히 침몰하고 있다는걸\n명심해라 기업이없으면 국가는 침몰하게 되있다\n제발 정신들 차려라',
 '그러면 들을수록.\n캣츠 보내',
 '다 이잼대통 덕분이라해라!',
 '땅이나 파고 살던 인간들이 ... 세상 좋아졌네',
 '제가 어떤사람한테 단통법으로 50조를 줬구요  특혜로 사면까지 시켜줫고요 전세사기당해도 금융권 우선순위로 친구  돈다가져가서  양보해줫구요 친구가 공장땅짓는다고 부모님 농사짓던 땅까지 헐값에 팔고 실업자되셧는대 이젠 친구가 ai기술발전시켜서 일자리마저 없애고 혼자 더꿀빨겟답니다   그사람은 저의 친구가아니라 그냥 이득충이였던것이죠  그래놓고 머라고할까바 \n그간이득으로 언론사사서 선동하고 떳떳한척이나 합니다 \n살다보면 알게되요  그친구인줄 알았던 그는  친구가 아니였다는것을 적보다 못한 존재였다는것을\n여기서 나는 국민이고 친구는 기업입니다',
 '이러니까 해외에 공장을 차리지 우리나라에서 기업을하면 세금에다 인금 지역구와의 갈등 나같아도 외국에다 공장 지어서 한다 그러니 일자리 줄고 지역경제 힘들어지고 어디까지 갈려는건지 .\n삼성.현대.LG.SK.S오일등 대기업들이 해외공장으로 나가는거지 반성해야할듯',
 '애국기업 어쩌구 렉서스?',
 '좁은 땅에 지자체 선거하니까 이모양 이꼴 되는거임.',
 '이런말하면 내란당이라하던데 ㅋㅋㅋㅋㅋㅋ',
 '기업 만들어 일자리 창출한다고 해도 반대하네 이제 뭐 가난을 원한다는 거네. 그래 50년전으로 돌아가보자 다 가난해지면 무지 행복하겠다? 이런 걸 원하냐.',
 '지방으로 가야지',
 '정말 너무하네~ 주변 상권들 만명이면 회식자리만 따져도 장난아닌데. 뭐라할 입장은 아니다. \n그러면 단가를 맞추던가~ 그것도 아니지? 나도 현작직이지만 회사에서 나오는 식사비 맞추느라  라면에 밥말아먹어.\n이정도야~ 현실이 이래. 본인들 단가는 유지 하고,  그라면에 뭐가 더들아가는줄아냐?! 그냥 물, 스프, 계란 살짝그리고 면 이게다인데 5000원이다.\n밥말면 6000원 왜 이런걸 먹냐고 다른건 비싸거든. 한달을 맞출수가 없어. 내가 식당하고 싶을정도다. 제발 양심좀 가져라.\n먼저 말한 회식자리만해도 당신들은 돈을 벌어~ 욕심이 지나치잖아. 평택처럼된다. 제발 자초하지마~ 정도를 알아라.',
 '노란봉투법 법인세인상 ㅠㅠ 주민들 지랄, 노조 지랄, 뭐하나 메리트가 없지 나같아도 해외로 나간다 ㅠㅠ 어쩌려고 이러는지,,,,,',
 '자유국가에서 소비자들이 선택하는거지. 서비스나 경쟁이 더 좋다면 소비자들이 자연히 동네 식당을 선택한다.\n저런게 공산주의식 사고방식이다.',
 '점심시간 1시간인데 그 많은 인원이 언제 먹고 언제 휴식하고 오후 작업 준비하나? 지역사회는 너무 욕심부리지 말라. 대학 주위에 학생용 기숙사 짓는 것 반대하는 지역주문들과 비슷하네.  그런데 그렇다고 못하고 안하는 대학이나 기업도 이상하다.',
 '나라 진짜 망하기 직전...',
 '듯다가보니',
 '삼성무너지면 한국도 조짐',
 '이건 거의 가면을쓴 채널이네\nㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ',
 '외국 35년 살고있는데 미안한 말이지만 소수의 한국국민들  문제가 많이있읍니다',
 '이런것들이 나라 팔아 먹는 젊은이들....내 댓글도 날라가겠나?',
 '너무하네 진짜',
 '중국 욕할거 없다',
 '한국은 투표가 국민직선제라 미국하고 달라서 주민들 눈치를 안볼수가 없음. 그러니까 님비현상없는 반도체는 널널하고 님비현상 없는 새만금에다 짓는게 낫다고 본다.',
 '이재용 회장님 \n인성이 좋으셔서 자식분도 잘자랐네요\n김영삼.이명박.윤석열 이놈들 전부 사지 멀쩡한놈들이 \n군면제인데\n이회장님 자제분은 해군장교지원입니다',
 '아니 엔테테이먼트 구내식당 이용은 칭찬하는인간들. ㅋㅋ',
 '한국은 알아서 망하겠다잖아 내비둬',
 '기업은애국 정치는 깡패 도독',
 '삼성 들어온다고 엄청 상업부동산 많이 져서 그거 보전해달라는거임... 틀린 소린 아닌데 그렇다고 맞는 소리도 아니고 한국에 현금흐름의 맥의 정부 지자체 대기업에서 에서 시작되니 모든분야가 \n재발이랑 정부한데 돈이 몰리니 세종도 공무원 상대가 믈가 비싸고 암튼. 감나무밑에서 입벌리고 기다리는중임... 재발대기업과 정부뿐이라. 다같이 다하고 있는 이유임',
 '평택 고덕 봐라.\n아파트 입주율이 반밖에 안됨\n삼성이 지역경제를 좌지우지하는 건 사실임. 삼성우월주의때문에 싫어했지만 요즘은 삼성이 지역에 미치는  지대한 경제적영향력에  존경심을 가지게 되었다.',
 '대한민국에서 나온 삼성이 대한민국때문에 망할수도있겠구나,',
 '말 할 때 입이 왜....이준석인줄',
 '도움이 하나도 안되는데 \n당연히 떠나는게 이치 아닌가',
 '삼성은 문제가 없어요. 오너일가가 문제지.',
 '말이  되는 일로   건의  하시기를  ~\n그많읁 인력이  어디가서  밥을  먹나요~~~\n진짜  가지가지   억지  부리지 마세요~',
 '심통질 상인들의 횡포는 소비자들의 외면을 받을것이다 갈수록 이기적인 억지 논리가 극심해지는 꼬락서니 보기싫다 삼성 물러서지 마시길 응원합니다',
 '지들이 뭔데 구내식당을 불허해 저런 자영업자들 망해야함',
 '저런거 들어줘야하나 ? 즈네 장사 시켜주기위해서 사내식당 금지 ? 이기주의네 그럴거면 가격인상 금지도 법제화해야지요',
 '직원 10000명 중 1/100의 입맛에 맞았다 하더라도 점심 1시간 사이에 수용은 가능하니??? \n맛도 떨어지면서 촉박한 시간에 건물 밖으로 굳이 나와서 먹으라는건 무슨 심뽀?',
 '우리나라는 망하는게 맞아.',
 '삼성이 못한게 뭐죠???',
 '삼성이 이재용건가?\n삼성 대주주 기관아냐?\n적당히 해. 국민연금 삥땅친게 이재용아냐?',
 '사내식당이 뭔소리여? 원래금지아니여 내가 yg가서 먹고싶다하면 해주나?',
 '이과가 나라 살리면 문과가 나라 망친다',
 '세금도 못 낼 수준이면 지역에서 욕먹는 거 당연한 거 아님 ? 물론 올해부턴 좀 내겠지. 이럴 때 구내식당 차림 되는 거 아님 ?',
 '말도 안되는 소리 안성시민, 하이닉스\n회사원들 구내식당\n서 식사 하는것을 반대한다고 ? 주민들식당 이용하라고 ? 그래갖고 회사 일 되겠냐?  주민들이 미쳤네 욕심을 너무부려',
 '대기업다니는 이유가  제 시간에 밥 제대로 먹고 일하는거 아닙니까. 그것마져 상권이라며 막으면 회사가 없어지고 경제도 없어지고 상권도 없어집니다',
 '시정부가 혜택을 보는데, 정부가 지원하는 게 맞는가?! 시정부와 시민이 알아서 해야지!',
 '미국가봤냐',
 '에휴... 대한민국이 선진국이라는 헛소리는 대체 누가 하는건지 모르겠다.',
 '대학교에 기숙사 짓지 말라고 하는 것부터가 시장 경제 무시한 공산주의 사고방식',
 '솔직히 식당하는 주인들 이기적인 마인드 가진 인간들이 대부분임 \n갑질 투성에',
 '삼성 애국자\n지방자치 다 없애라',
 '지방으로 와라 지을때 많다',
 '구내에서 밥먹는 삼성 떠나면 더 폭망할텐데  그나마 그동안도 삼성있어 장사잘하고 잘먹고했으면서 이시국에? 냄새난다 누가시킨거냐  조사해봐라',
 '삼성만 하더라도, \n천안,평택,수원, 용인등 \n충남과 경기남부에 끼치는 영향이 크고, 화성은 말이 신도시지, 초반엔 미분양뜨다가 삼성덕에 공급풀되고, 동탄을 계속 확장하고있고, srt에 gtx수혜받고, 덩달아 토지상승도 한몫잡고 이득보고 있자나?\n용인, 남사에 하이닉스와 삼성이 들어오면, \n젤낙후된 처인구도 얼떨결에 같이 개발되고, 인구유입에 세수까지 확보되니, 1석2조인데, 그깟 동네식당?\n안그래도, 트럼프가 반도체관세 100%때리는데\n행여나, 삼성이 미국으로 모두 이전때리면, 최대 클러스터라 관망대던, 남사는 하이닉스만 남을진몰지만... 그냥 깨지면서, 유보금 낭비안하고 그돈 잘간수하고 좋은대우받으며, 미국에서 사업확장이 더 이득이될거란건, 애도 알고잇을 셈법인데...\n암튼, 삼성으로 덕보며 확장되는 도시들은, 삼성빠져나가면, 경제 부동산 모두 쪽박찰생각하고, 좋은관망보다, 현정부의 경제논리와 국제정세를 생각해보고, 좀더 넓고 리스크대비하고, 후폭풍대비하는게, 이로울듯하다.\n\n현정권은, 기업을 살리고 일자리창출과 정반대경제행보를 보이고잇고,\n수출로 먹고사는 대한민국의 한강의기적도 이젠 상승기회는 줄어들고 하락할일만 남았기에, \n고삐풀지말고 제대로 정신줄 잡아야 큰타격에서 그나마 살아남을거라 생각듭니다.\n\n대기업을 적대시하면서, 경제발전, 일자리창출, 세수확보란건... 그냥 꿈속에서나 많아하고 현실은 그반대이니, 대한민국을 위해서라면, 제발....\n정신좀차리자.',
 '시설물 짓기전에 허가 확실하게 받아놓고  시작하세요. 가능한  인간들 없는  논바닥에 지으세요.~ 열받아서 못보겠네요',
 'ㅋㅋㅋ 삼성 하이닉스 다니는 사람은 뭐 돈이 썩어나냐.. 저렴하게 팔것도 아니면서 사내식ㄷ당을 이용해라 마라야',
 '삼성, LG가 구미시에서 수원으로  옮겨가고 난뒤 구미 경제는 땅바닥을 메고 있습니다.\n지자체나 주민들도 반성해야 됩니다.\n세계 경제가 어떻게 돌아 가는지 등잔불인데  노조나  반대하는 지자체를 보면 한심하기 짝이 없네요...',
 '중국한테 쎄쎼한 죄',
 '대통령 한명 잘못 뽑은게 죄임....으휴...ㅜㅜ',
 '공업대학 활성화가 중요한다 무슨 법대 의대냐 한국은 이미 늣었다 법대 의대가 돈을 더많이 버는 특이한구조 전세계는 엔지니어 임금이더 놉다',
 '미국으로 가서 지어라',
 '대기업이 와도 환경만 망치고 지역상권에 도움이 안되는거 사실입니다. 대표적으로 춘천에 네이버 데이터 센타가 있는데 춘천시에 서는 처음에 잘 모르고 네이버 온다고 엄청 좋아했습니다. 그런데 현재는 녹슨 방치된 듯한 풀숲에 건물만 덩그라니 있습니다. 지역 경제에 아무 도움도 안 되죠. 지금 춘천 사람들은 네이버에 속았다고 말하고 있습니다',
 '이런 어거지로 들어주지 마라!...',
 '삼성은 미국 떠나도 욕 안한다. 사실 재용이형이 애국자라 지금까지 참고 국내에서 버티는거지 사업가 입장에서 봤을때 왜 아직도 미국으로 안갔는지 그게 더 의문일 정도였다.\n나였으면 깜빵 나와서 치킨 시켜 먹을때 미국으로 넘어갈 결심 했을거다.',
 '이대로 가면 10년안에 한국 망할것 같네요 좋은기업 다 외국으로 나가면  안그래도 일자리 없는데',
 '삼상빤다고 고생이 많다 ㅋ 뭐 그게 쉽게 버는 방법인가?ㅋ',
 '여태 그리 많이 해먹었으면 됐다 싸게해줘도 뭐라 하면 군인들 안가는 양구 꼴나는거지',
 '노조들의 불법에 대해 엄벌에 처해야 하고, 노조들의 월급을 내려야 하고, 성과급을 중단해야 한다. 노란 봉투법으로 기업이 떠나면 국민이 피해보는 것이고, 청년들의 일자리가 없어지는 것이다.  범죄자 이죄명 하루라도 빨리 제거해야 대한민국 정상화됩니다.',
 '정치가 바뀔때마다 항상 삼성못살게 글구. 정말 미국가서해라',
 '상간남 최태원은 언급하지말았으면....',
 '나라망하겟네',
 '사내 식당을 지들이 뭔데 금지해?',
 '그냥 지들이 싫다는데 하지마라. 미국가면 관세 받을일도 없고 님비도 없고 노조도 합리적인데. 굳이 개고생하면서 왜 짓냐',
 '그럼 하나 묻자~\n삼성이 잘몼하고 있는것이 뭐꼬?  이재용부회장이 잘못이 뭐꼬?',
 '뭘 모르는 사람이 유시민 선생님 말씀하시길   아버지 자식이 말해도 이익이 없으면  기업은 말을 안듣죠. 기업하기 좋은 곳',
 '용인 처인구 10년살고 안성이사와서3년거주하고평택고덕으로 출퇴근하는 1인임\n\n그런데 안성은 이해해줘야함\n용인 하이닉스지어봤자 안성 혜택 없음 오히려 오폐수만 떠안게됨\n차라리 안성용인 걸쳐 지었어야함\n집값이야기하는데 용인원삼쪽만 혜택봤고 안성은 없음 어차피 원삼면접경이 안성고삼면인데 사람 별로없어 집값혜택이 없음\n\n하이닉스보고 용인경전철이나 이천여주부발선이 안성으로 연결되면 또모르지\n\n평택고덕은 이제 건물 거의 다 지어 건물값떨어짐',
 '삼성 수원사업장 직원들의 회식 담당들이 회사 근처가 아닌 다른 지역으로 회식을 추진하는데 그이유가 삼성전자 근처의 중앙문 상가의 물가가 비싸서 그렇습니다\n웃기는건 가게들이 오래 못버티고 나가는데도 가격이 비싸요, 그러면서도 사내 식당 때문에 장사 안된다며 사내 식당 패쇄하라고 한적이 있었음 삼성단지가 생긴 후 나중에 식당촌이 생겼는데 그딴식임 지들은 LG 에어컨 TV 냉장고 쓰면서 삼성에게 먹여살려달라고함',
 '미국가서환대받으며하라\n이나라 노조가사라지기전엔안된다',
 '미래를 아는 사람은 없다고 본다. 다 뇌피셜 수준..',
 '대한민국은 겨우 중진국에서 선진국이 된 최초의 나라이면서 선진국이 되자마자 바로 몰락하는 첫번째 나라가 될겁니다. 내 말년에 무너지는 대한민국을 보고 있는 현실이 너무 안타깝습니다. 그동안 어떻게 고생하며 여기까지 왔는데...나라는 이제 완전히 2조각으로 쪼개질 상황이고, 국제정세든 급변하는 경제환경 같은 건 안중에도 없고 오로지 알량한 권력을 위해서 도무지 상식조차 없는 듯한 여당과 야당...중도는 침묵하고 양극단의 극렬한 의견만이 지배하는 정치. 한쪽은 사면해서는 안될 사람까지 사면하며 자충수를 두고 있고, 기업은 망하든 말든 자신이 젊을 때 가졌던 이데올로기를 살펴볼 생각없이 그저 노동자를 위한다는 명분으로 기업죽이기에만 몰두하고 그래서 결국 일자리 줄어든면 그 피해는 노동자가 보게 만들고 있고. 또 한쪽은 계엄을 옹호하는 두 사람이 대표자리를 두고 선출되도록 그 당원들이 압도적으로 뽑아줬고. 이 나라는 이제 외부의 상황은 안중에도 없고 정말 우물안 개구리가 되어서 서로 죽이자고 달려들고 있다. 한국이라는 나라의 문화는 각광받고 있을지 모르지만 우리 국민들의 의식은 다시 천박해지고 있다. 국민연금, 노령연금을 받아 생활비가 걱정없는 노인들이 매일매일 시위에 나서고, 고정된 일자리를 가지지 못한 젊은이들은 인터넷으로 돈벌기에만 몰두하고, 도대체 이 나라에 혁신과 미래가 있을까? 침몰하는 대한민국에서 권력을 차지하겠다고 쇼를 하는 정치인들....아직 채 얼마 살지 않은 우리의 아이들이 너무나 정말 너무나 불쌍하디. 우리야 이제  나이 들만큼 들었지만...',
 '삼성을 한국기업이라고 착각하는 한국인들 많네. 다국적기업이다. 걍 생까고 해외로 가버리면 못잡는다.',
 '언제부터 지역주민들이  깨시민이 되었나? 노조에 지역민에~  싨어진다.',
 '저는 AI 생체실험 피해자이자 생존자...이제는 증언자입니다. 자세한 사항은 제 유투브에...',
 '노조는 사라져야할 1위 대기업 아니면 대한민국  과연 성장 할 수 있었을까!',
 '배부른 짓 하는거지. 국가가 우선이지 개인이 우선인가. 국가가 유능해야 개인이 잘 사는거지 개인이 유능하다고 다른 개인이 잘사나? 그놈만 잘살지.국가에 국가기간산업을 대입해봐라. 저것들이 얼마나 소탐대실 하는지.',
 '정부가 열심히 발전하고 살아남으려는 사람, 기업들 지원은 안하고 맨날 말도안되는걸로 징징거리는 인간들 달래는데만 집중하고 있으니…. 나라가 망하지',
 '재용아.  미국가라',
 '내가 이재용 회장이면 한국에 더 이상 투자 안한다,,,,정치인들 괴롭히고 지역구 이권단체들 반기업정책….으이구…한국 망해라',
 '기업한테서 돈을 띁어서 가난한 사람한테 떼주려고 하는 이유\n어느나라던 부자보다 돈없는 사람층이 두텁죠\n그래서 정치꾼은 표심을 모으려고 돈없는 사람의 지지를 받는게 쉽다는걸 직감하죠\n그렇게 수준떨어지는 정치꾼들은 부자의 돈을 10뜯으면 2를 챙기고 8을 뿌리고\n이짓을 반복하면서 뜯은 2로 정치자금을 계속 늘리는 쓰레기 짓을 반복하는거\n그러다가 기업이 떠나려고하면 매국노라는 프레임 씌우고, 가난한 사람들은 거기에 동조하죠\n지적능력과 부는 어느정도 비례한다고 봐도 무방함\n지금 민생지원금 뿌리는것도 우리를 뽑으면 꽁돈이 생긴다는 신앙심?? 뭐 그딴걸 세뇌시키는거죠\n그 민생지원금의 출처가 단순히보면 세금이지만 깊이 들어가면 기업들한테 삥뜯은걸 겁니다.\n이놈의 정치꾼들은 작정하고 이짓을 하는거라 그러지말라고 타이른다고 먹히지도 않습니다.\n예방주사 차원으로 대기업들이 여기를 줄줄이 떠나서 초토화가 되어야 이 모든 화살이 정치꾼들한테 돌아가고\n그제서야 뒤늦게 정신차리는 척 하는게 늘 겪는 한국의 모습이죠\n곧 그때가 찾아올겁니다.\n기업인들 집합시키고 이리저리 끌고다니고 하는 깡패들이나 하는 그 버릇이 어디 가겠어요',
 '삼성비난하는 사람 우리 국민\n아닙니다',
 '사기성 광고가 아무렇지도 않고 무제한으로 반복되네요.\n얼마나 많은 사람들이 피눈물을 흘리고 있는지 모로겠네요\n이런 광고에 출연한 사람들도 죄가 있으면 강력하게 처벌했으면 좋겠습니다.',
 '우리나라 사람은 절대 남 잘되는 꼴 못보는 사람들이 너무 많은거 같음ㅋㅋ\n이러다가 국내 기업들 다 해외로 떠나면 그땐 좌파 우파 정치질 하다가 끝에는 한국인이라는 이유로 매국노 식으로 비난 하겠지 ㅋㅋㅋ',
 '이기적인 민원,  이에 대한 입법이 있어야 합니다. 개인의 손톱만 한 이익을 위해 국익이 말살되는 일이 없도록 이재명 정부는  나서야합니다..',
 '미국에 가도 마찬가지다, 처음에는 세제혜택을 주고 어쩌고 하지만 시간이지나면  마찬가지다, 공화당 정권이 하는게 그렇다. 나중에 야당과 시민들이 반대한다.',
 '이재용이 미국가서 사업 했으면  감방에서 못 나왔지 ㅋㅋㅋㅋㅋ',
 '남한은망하고야 만다',
 '극 이기의 표본.   누가 와서 식당 하랬나? 자기들 선택으로 와선 ....',
 '악법만 골라 만드는 민주국힘 연합당만 없으먼 우리나라는  살만한 나라,,,',
 '안성시.용인시민  지역이기주의는 자녀들 먹거리를 막는행동이고 나라를 망치는짓입니다.\n일자리가 없어지면 자손들이 손가락 빨라고 멍청한짓 하나요?\n생각좀하고 삽시다.  그리 이기적이어서 잘사나요? 나라가 잘되야 국민이사는겁니다.\n세계적인 대기업이 외국으로나가서 외국인들한테 직업주고 이득주는게 좋습니까?\n심각한 상황이네요. 지역주민들 이러면 망합니다. 제발 기업들좀 놔둬요. 도와준것도 없이 방해하지말고.!!\n그지근성 좀 버려요. 챙피하지않습니까?  한심하네요.\n그지역 아파트값 왕창 떨어졌다는데. 좋습니까.  제발 선량한 시민들 피해주지말고 가만히나 있어요.',
 '삐끼 마인드 갖고는 안뒈~~~ 사내식당 구내식당 등 특수단체급식 식당 메뉴얼은 한국은 아직 한참 떨어졌다고... 사내보안을 요하는 직원들 대화나 몸짓이 나가기만 해도 정보 다 새나가는  세상 임돠~~~ 경쟁을 독려하는 회사 성격상 회사가 지원하는 복지부분인 구내식당 사내식당은 반드시 필수 입니다~~~ 해외에 나가서도 마찬 가지 입니다.... 사내식당 카페 가지고 있는 회사는 일류 회사죠~~~~ 군대식당도 마찬가지... 훈련과 업무마친 군인들에게 당신들 밥 나가서 먹구와??? 이건... 세상 돌아가는거  모르는거에요~~~  50년전 중동건설사에서도 사내식당 운영은 현지 비지네스에 필수 였습니다!!!!! 이는 또 최첨단 물류와도 밀첩한 관련이 이습죠~~~~',
 '삼성전자 나팔수냐 ㅋㅋㅋㅋㅋ',
 '이재용, 최태원이가 미국에서 사업했으면 이미 구속 됐다. 미국 가고 싶으면 가라 해라.',
 '대기업들이 여기저기 공장 짓고 수출해서 달러벌어오는거 안 좋아할 대한민국 국민들이 누가 있겠냐만은...결국은 자기 집 근처에는 어떠한 기피시설들이 들어온다고 하면 집값 떨어진다고 드러눕고 반대 할 거면서 무슨 80년대마냥 나라를 위한 것이니 지역주민들은 입닫고 있어라 하는 마인드네요 그럼 그에 따른 보상이 있어야 할 것이고 그게 어떻게 진행되는지 정말 지역주민들의 이기주의에서 나오는건지 확인을 하고 컨텐츠 제작을 해야 하지 않을까요? 80만 구독자를 보유하고 있다면 영향력도 엄청날텐데 대기업이 공장 짓는거에 지역 이기주의 때문에 해외로 공장들이 나간다고 보여지는 영상이라 안타깝네요',
 '까스화력 말고 원자력을 잦자고 하세요~ 수원과 서울 사이에~ 응?  당신 동네에 꼭 지으세요~ SMR 은 안전하다며? 오케바리~ 고고고~ 꼭 알리세요~ 응?',
 '동네식당이 하이닉스 때문에 영업이 안되는가 ? -> X\n그럼 왜 동네식당을 이용하라고 ㅈㄹ 하는가 ? -> 본인들 매출 올리려고 \n사람이 서있으면 앉고 싶고 앉으면 눕고싶은게 사람인데 \n사내에 있는 식당을 굳이 밖에나가서 ? 내돈내고 ? \n장사를 하려면 음식과 서비스도 물론이지만 생각을 해야하는데 왜 무지성으로 징징 거리기만 하는지 \n우리나라 언제부터 징징징 만 하는곳이 되었는가...',
 '앞니좀 어떻게 해봐라. 앞니 때문에 입이 안닫혀서 발음이 제대로 안되고 뭐라 그러는 건지 숨쉬기도 힘들어보임.',
 '잘하는 니가 정치해라',
 '쇼부 같은 일본어는 방송에서는 사용하지 않았으면 합니다.\n승부라는 우리 말이 있습니다. 해방된지 80년인데.. 아직도 일본어를 써야하나요?',
 '지방으로 와라!!  수도권에서 사람 많은곳에 피해 보상 해줄돈으로  사람 없는 면 단위 지방으로 내려와',
 '한국인 국민성은 공산주의 가 맞습니다',
 '지금 정권 하는 짓을 봐라 기업 죽이기에 안달이다 나라 자체가 위수지역이다',
 '기업이사라지면 세금걷을곳이\n없어지고 경제는마비되고\n자원도 없는 나라\n강성노조가 설치는 우리\n나라는 망합니다',
 '자기생각을 얼마 안되는 지식으로 그럴듯하게 말하는 사람들이 많긴합니다',
 '말 하는데 입모양이 ..왤케 얄밉;;',
 '삼성이 잘못 한것 있나요.',
 '조선국 자본파업',
 '멀 잘 모르는거 같은데?',
 '거기에 문쟈인 이재명 이 대통령이면 대기업은 그냥 해외가야지뭐 미친나라',
 '삼성이 떠나면 중국기업이 들어오고 속국이 됩니다.',
 '주민들미첬네.말도안되는소리다?회사가있으면.당연이사내식당있는거지?억지부리지마라?',
 '대구일베',
 '댜 떠나라 답이다',
 '삼성은 이제 미국으로 가십시요 한국은 더 이상 희망이 없소 뭘 그렀게 자국에 신경을 쓰시나요 국민들에게 돈을 벌 수 있는 직업을 주려고 하는데 이내저래 말이 많으니 그냥 미국으로 가서 미국 국민을 먹여 살리시요',
 '어떻게 저렇게 이기적인지... 옛날에 도대체 뭘 어떻게 배운건지.. 이해가 안감.. 진짜 짜증남..',
 '외국가면 기업 세습하기 어렵지 않음? 우리나라야 불법이 아닌 교묘하게 세습해서 상속세 적게내고 대기업 꿀꺽하는데 ㅋ 미국이었으면 세금폭탄이나 깜빵에서 죽을때까지 살겠지 \n이거부터가 말이 안되는 기업가의 이기심이지 \n공장 중남부 지역에 지으면 말릴사람 없을껄? 굳이 경기도권에 지을라고 전부 투기꾼들이 보상이나 땅값올릴려고 쑈하는거지 한숨만 나오네',
 '삼성 이재용이 대통령 되면, 나라가 살아날걸?',
 '노조나 상인회 다 처망해라 진짜 그만좀해라 대한민국 에서 정부에서 노란봉투법인가 뭔가해서 기업 다나가는대 아직도 저러고있네',
 '손좀 쓰지 마시오 정신사나와요.   수화를 하는것도 아니고.',
 '민노총은 기업 위에서 군림하고 온갖 행패를 부리고 있는데, 또 노란 봉투법으로 기업을 옥죄니?',
 '조카는 조카네',
 '화성은 망해야합니다! 다른 나라나 다른 지역으로 눈돌리세요. 뭐가 아쉽나요? 창원 야구장 nc 상황이랑 비슷하게 보시면됩니다.',
 '박대통령 덕에 살아갈뷴.\U0001fae1',
 '사대주의에 젖어 있는 댁이나 미국가쇼............이 사람의 현옥하는 말에 속으면 안되지',
 '그쪽이 좌파가 우세한 지역. ',
 '정알아서 정권논치보기',
 '삼성 문닫게할려고 문제인정권때 구속시키고  그많던 영업력무력화하고 지금의 경쟁력의 삼성을 있게 햇다  참으로 멍청한짓을 하고 그기에 삼성다니는 직원들조차도  어이없는 짓을하고  또 노조를 집어넣고 지금도 많을 월급을  받으면서 노조가 대모하고있다  글로벌경쟁력을 밀어줘도 시원찮을 판국에  지금와서  정부에서 후회한들  이미경쟁력이 뒤처진상태인데  그래도 잠안자고 고민하는 삼성맨들이 있어  이나라가 굴러간다',
 '세종대왕 정조대왕 재명대통령',
 '참 자유민주주의, 자본주의 사회에 대한 이해도가 딸린 수준이 요즘 최소 4,50대부터임\n2030은 선진국 시대의 국민이라 수준이 높음',
 '반기업 정서를 계몽하고 피해의식 주입하는 매국노들인지 한국 망하게 하려는 중국인인지 많아요... 결국 경제를 이끌어가고 양질의 일자리를 만들고 국가 인프라를 짓고 천문학적인 법인세를 내고 사회에 공헌하는건 결국 기업인데.. 어떻게든 기업 싫어하고 착취니 뭐니 개소리 하는 인간들이 매국노죠. 일제강점기때 친일파랑 다르지 않음.',
 '대기업 빌붙어 기생충 처럼 연명하면서 전과자 뽑는 클라스',
 '이재명대통령님삼성기흥공장에 사원들이먹고일할 식 당도짓고 사원들이 자고일할 숙소도짓고 올해안에 완공이 되게 도와주세요.기흥의 이기 적인 사람들도인해 6월에 완 공이되야할 기흥의공장이 완 공이 되지않아서 너무걱정입 니다. 우리나라 대기업이 우리나라에 투자를많이하여 문닫았던 중소기업들이 다시 문을열고 활발하게일하고 직 장을잃은 사람들이 다시직장 에 들어가서 활발하게 일하게 도와주세요 ~  제발요~ 기흥지역에 관리자들이 더럽 게일을못해요.시민들말만듣고 개떡같이 일해서 기흥의 이기적인 사람들로인해 우리 나라경제가 퇴보될까봐 걱정, 고민,슬픔,화가납니다.',
 'ㅣ찍그모야이지',
 '삼성이 국내공장 없애고 미국가는게 그렇게 떠들 일인가? 유튜버들 문제 정말 심각하다',
 '이분 삼성에서 칭찬받으시겠네 아님 뭐 이미 받으셨나.. ㅋ 댓글들도 하나같이 뭔 삼성알바들이냐. 참 희한하다니까. 이나라 국민들은 희한해 거지들이 부자 세금 걱정하고 대다수가 힘이없는 노동자들임에도 이리도 기업걱정들을 하는 거보면. 전부 기업광고로 먹고사는 언론에 세뇌가 된건지 파시스트들을 양산하는 이나라 공교육시스템 때문인건지. 이게 바로 기득권들이 기득권을 유지하는 가장 좋은 방법이지(약자들끼리 싸우게 만드는거.)\n이분말씀대로 저지역 주민들이 문제가 있다. 인정. 어딜가나 저런 몰상식한 인간들이 있기마련이다. 저렇게 생떼쓴다고 언제 그렇게 눈하나깜짝했다고 그래.. 그렇다고 저게 삼성이 이나라에서 기업을 못할 이유가 되나? 가라그래 ㅋ 가서 그렇게나 기업하기편하고 좋으면 진작가지 왜 안갔냐. 이재용이 미국에있었으면 분식회계에 삼성물산 합병시 벌인 불법편볍적일들로 주주들에게 엄청난 손해를 입히고도 지금처럼 아무일도 없이 경영권승계하고 풀려날수 있었을까? 재벌들기업총수들에게 얼마나 관대한 사법시스템을 가진 대한민국이 얼마나 기업하기 좋은 나란데 뭔소리를 하는거야. \n쟤들이 공장지을때 국가나 지자체로부터 아무 혜택도 안받냐? 마치 엄청 지원을 못받는거처럼 이야기하네. 삼성등 이나라 대기업들이 국가지원 행정혜택 국민세금없이 만들어졌나? 앓는소리좀 그만하지',
 '잘 생각해보자. 왜 국짐당이 30프로 정도 지지를 받겠냐? 이런거에 질린 사람들이 국짐당 찍는거라고,',
 '박정희대통령의 방식이 맞는거 같다\n민주화도 벌어먹고 살아갈 일자리가 있고 나서지 굶어 죽어도 민주화냐',
 '제가 느끼는 사회 분위기도 그렇고 제 주변사람들도 그렇고 부자가 되는걸 욕하는 사람도 언론도 본적이 없습니다. 반칙을하거나 범죄를 저지르는 걸 비판하는 건 봤지만요. 그럼 부자가 되기위해 반칙하고 범죄를 저질러도 눈을 감아줘야 하는 걸까요?',
 '지적옳긴한데 왜 윤검찰땐 이나라 언론 죄다 입꾹하다 진보정권들어서기만 하면 개소나 봇물터뜨리냐',
 '진짜 폐급이다 이 유튜버 안일구가 보인다',
 '이재용 제보한적있는데요 \n지문ㆍ안구인식 해제안하고 사용 가능한가요 ? 노노노 그럼 대신 출ㆍ퇴근은 찍힌다 ! \n내란범죄가 옆에있다! \n\n검사하는곳? \n봉투가 왔다리 갔다리 급여가 어디로? \n\n갈까요?\n해제신청안하면 자동 해제가되야죠?\n급여가 그대로ㅡ세금이 그대로 건강보험도적용되어 나오고 있다는 점입니다 \n아시나요?\n이재용 또는 삼성일가의  급여가 제대로 내려진다?\n\n글쎄요?현장투입해서 잠시 일해봤어요!\n헌데?\n엉망입니다~  \n나는 뭐 숨은 보안관역활을 한 셈이죠~  근데 업체는 모르고 다 해먹고 장비안주면 장비값 또 \n급여가500만원 300만원 주간만이라도 그렇다 ! \n근데 급여에서 수수료 교육비 준다는 교육비가 안나왔네?\n또 실제로 다니고있는지?\n확인을  잠시 해보셨나요?\n\n뜯겼다! \n\n삼성이 ㅡㅡ 그리고 실제로 다니지않는데 ? 다닌다 출ㆍ퇴를 찍을수 있는곳이 어디게?\n정답 ㅡ입니다 \n\n범인이 어디서 어떻게 나올까나?\n누구에게 잘보이면 돈이 출ㆍ퇴근이 찍혀요~~~!!!!!\n알것지요 \n정보 주면서 내급여 제대로 일한 노동값 못받아서 고용센터에 신고했는데 ?\n전화가 끊겼다요?\n전화비 없거나 전화기종의따라서 등록이 되고 안되고 합니다, \n왜일까요?\n통신비에서 핸드폰을 구입또는 업체 서비스 또는 그곳의 반장ㆍ소장ㆍ에드토 같은 업체? 등등 에서 같이   쉽게 사람들 등처먹는다.이말씀 \n삼성 기업은 돈주고 했는데? 중간 사람들이 문제 인부분 그리고 시스템 입니다 \n수기 작성은 아무나 가능하죠잉? 그럼 여기까지 제가 신고자입니다.\n\n에드토회사가 웃겨요~ \n쉽게 사람 보면 절대로 안되는이유 뭘까요?\n\n나는뭐 숨은그림자 아닌가벼? \n당하기만합니까?\n법적대응 필요없어요 \n\n본게 전부이고 실제로 급여가 어떤지 보라ㅡㅡ\n특수요원도 아니고 정보수사관도 아니고 \n나는 그냥 숨은 그림자 라서 요렇게 당했네?\n\n식당이 문제일까요?\n절대로 아니고요 \n식당이 멀어요 \n화장실 멀어요\n다리가 아파요 \n하루하루 관리해야합니다\n그럴시간충분한가요?\n버스타는곳 멀어요 \n시간이되면 출발 ㅡ없으면 걸어가?교통비 걸어가도 준다며 체크가안되요! \n그래서 거짓이 많다!\n교통비 교육비 어떻게 줄건디? \n탓는지 확인가능한 명찰있다며ㅡ 그것도 버스회사에 간단하죠 ?\n시스템은요 그렇게 쉽게 가져간다 남의 돈을 주위하시길 바란다.\n\n내돈 내놔\n장비는 월래 지급이 되는거라고 명시 되어있어요 .\n다시 바꿔서  서류꾸며대지마세요!\n돕는마귀 화난다~~~!! \n\n이재용 아닌 누구의짓? 정보가 삭제된이유가 뭘까요?\n제일큰이유 쿠팡ㆍ삼성이야~ 범인검거 힘들게 삭제 또는 정보가 도둑맞았어요! \n누가 했을까요?\n\n억울합니까?\n아니면 \n진실을 은폐하려한걸까 ?\n노동값따위 안주고 건설사편든건지?\n아니면 내란수사 막은거야?\n앙?\n\n여기 경기도경찰서도법원도 난리가 났다\n만약에 삼성이 한일이거나\n건설사쪽 집안에 공무직이 있는 소장ㆍ반장ㆍ등에서 있으면 니들입니다 \n수사 완료 해서  일한거와 장비값은 내놓거라~ \n얼마안된다 웃기지마라! \n우습게 보는 너희 건설사 소장ㆍ반장ㆍ그 위 정직원분들 조심하세요! \n항시 ㅡㅡ 특히 저런 업체 열심히 잘 서울에서도 하고계시죠?\n딱 걸렸네?\n\n에퉛ㅣㅡ토 \n요기가 주가 되어서  ㅡ 그 나머지 업체 회사들 싹ㅡ다 수사해요 \n\n제가 방송크게 해야 안전수칙 지키더라ㅡ작게하면 작업무시하고한다.\n캔디주고 ㆍ잘부탁드립니다 방송 규칙어기는데 그런사람 계속 더 쓰더라~~!! 적절한가 보세요\n규칙은 규칙인데 \n나는 사탕아닌 핸드폰사용해서 오기전에 싸인을 주는데 무시당했다 \n그래서 에라이ㅡ 핸드폰 소리쳐서 정지 먹었는데 \n나를 노려보더니 안부른다?\n\n그런업체들이야ㅡㅡ \n공사장이 일을못하고 방송하면 싹다 도망가요 \n피해주시는 일이 있고 \n등등 .\n저있는곳 잘지켰다 가 아니라 ? 그곳 바로 안에 다들 범인들야 ㅡ \n웃겨 죽것네?\n표씨를 제보했다 우짤래?\n체포는안하고 방송타냐?\n그래서 제보좀했다 내란수사좀 같이 해주셈요 \n했더니 위험해서 무서워 뒤지는줄알았네 했더니 나를 왜 교도소보내냐,\n일주일씩이나? 밥도못먹고 몰라 나한테는 이주야 2주 \n왜 날 보내? \n미쳤나 ?\n\n죄질도 죄명도 이상하게 해놓고 ~~ \n관우도령측근이좀 ᆢ 이상하게 꼬인게 아진이네라고 이씨ㅡ 원정리 라고했지 ? \n\n원정리 몰라?가르쳐줬더니 근내리ㆍ고덕ㆍ청양 이라고 분명히 했다잉.\n너 그렇게 한 사람잡아라잉 엄한집 잡지말고잉?\n\n그라고 \n삼성은 내핸드폰 안되는데 카톡도 막 봐 그거아는가?\n\n아니 단체톡 사진등등 \n어디로 정보유출이야 \n\n조심조심 난 받아야합니다.\n돈 내놓으세요 !\n삼성 ㅡ \n만신이 함께 내란수사 할지 받을지 모르겠네?\n\n범인이  누구게? \n도울지 아니면 보상받아주실까요?\n\n누구편 ?\n다 삼성편 들었다?\n혼자싸우고 있었다 \n그리고 조용히 한 몇분은?\n내편에서서 쭈우욱 수사협조했다,\n나도모르는사이 ㅡ 메롱',
 '힘들게 해도 한국을 떠나지 않는 삼성 문재인 때는 이재용 회장 교도소 보내놓고 문재인 얼마나좋아 하던지',
 '뭘 잘 모르고 떠들고 있구나...용인 안성 살아봤냐? 난 용인 2년 넘게 살았는데 반도체 클러스터 처음부터 반대했다. 용인의 아름다운 자연환경 다 황폐화시키고 그 터 위에 반도체 공장 지어 주변 환경 오염시키는 게 잘 하는 짓 같냐? 돈이 무조건 최고인줄 아냐? 반도체 공장에 반대하는 안성 및 용인 시민들 입장은 생각해봤냐? 그들이 단순히 반대를 위한 반대를 하는 것일까? 반대편 입장에서도 한번 생각해보길 빈다. 대기업 입장만 대변할 게 아니고',
 '이준석 모해? 삽질말고 이런거나좀 하지',
 '좌파시민단체및각종 사이비단체들이 시민들에게 선동하는게 문제애요.\n\n삼성.현대.SK..등등 모두 해외로 나가야 정신차림',
 '뭐 어디 차리고, 주변 상권 죽인다며 시위하는건 짜친다 라는\n말 까지는 OK 이해함, 근데 한국만큼 세금 덜 거두는\n선진국이 있나? Nope, 세금을 덜 거두자는 멍청한 소리부터 ..\n아~ 이 사람 개소리하는 구나 느낌.. 한국이 그렇게 세금\n과하게 거뒀으면 이미 해외로 다 나갔지 왜 한국에 있겠음?\n제발 생각좀 하고 영상좀 만드시길..',
 '들으면 들을수록 쓰레기네',
 '한국에 투자하지마라. 한국은 정치와 노조로 인하여 안된다.특히 민주라는 탈을  덮어선 놈들 때문에  망한다.',
 '삼성이 미국에서 태어나고 미국에서 기업이 시작되었으면 한국에서처럼 성공했을가?삼성이건 sk이건 현대,엘지 모든 한국의 재벌기업들이 성공한 바탕은 근로자들의 고혈을 짜내서 이룬부분이 분명 존재하고,국민의 세금이 지원되었고,부패한 정권과 뇌물거래가 오갔었고,그런건 생각 못하나? 미국에서 탄생했으면 아예 지금까지 한국에서 해왔던 경영방식으로는 진즉에 망했던가 성장하지도 못했음,무노조경영을 자랑스러워하는 기업이 미국에서 무노조경영을 할수있을거라고 생각하나? 극우파쇼들이 독재를 찬양하면서 민주주의를 지키겠다는 허무맹랑한 주장과같은 앞뒤안맞는 주장임,독재가 민주주의인가?그러면 또 이렇게 말함,독재했지만 장단점이존재하고 공과가 있다고,독재했지만 경제를 사렸다고 그래서 독재했어도 괜잖다고,그러면 중국도 중국헌법에 공산당1당독재를 명시하고 독재를 해오면서 세계경제2위 G2국가로 만들었는데 그럼 중국공산당도 독재했지만 경제대국만들어서 잘한건가? 다카키마사오 유신독재는 잘한거고 중국공산당독재는 나쁜건가?그 차이점이 뭔가? 독재는 독재일뿐,독재는 민주주의의 적이고 반대일뿐,독재에 좋은독재 나쁜독재가 존재하나? 아무리 나쁜 민주주의라도 아무리 좋은 독재보다는 좋은것임,',
 '시대에 역행중인 재명이',
 '문정권  윤정권에서 정리했음  됐을걸  안했으니  시장  잘 뽑아요  시민들도  잘 협조하시구요~~~ 그리고  국뽕 기업으로  재도약 하세요!!',
 '삼성 떠나라\n정치 기러기들 반성하라',
 '미국에서는 허허벌판에 짓습니다. 아파트 단지 근처에 짓지 말고 멀리 떨어진 지방 벌판에 지으세요. 너같으면 집옆에 공장 두고 싶나요? 방송하려면 좀 정보를 잘 알고 하길.',
 '철저하게 기득권 보수의 주장과 친재벌기업 관점을 전파하면서 돈을 버시는 분이.. 과학기술 채널이라는 소리 좀 그만 했으면 좋겠네',
 '큰거를 보지못하고 작은것에만 집착하는 소탐대실에 현옥되는 무지한 자들이다.자칭 보수라는 우익파시즘들이 세뇌교육 시키 결과다. 그나마 다행인건 진보가 정권 잡을때 이루어 논 자유 민주주의로 인해 이정도까지 발전한것이다. 정치가 경제인들을 길들이는 정책을 쓰는건 어제 오늘 틸이 아니다. 다까끼때부터 대머리 ,쥐박이,닭그네,윤개통까지 경제인들에게 사리사욕 안채운 놈이 있었는가를 보면 알수 있다. 물론 진보를 개검들이 공작으로 몰아 기소 되었던 적은 많았지만 극우처럼 몇천,몇백억이 아닌게 그렇게까지 공작 할수 없었기 때문이다.',
 '좀 더 배워라 ....재벌의 역사 알아보고 지껄여라....코인팔이 제대로 해....재벌들이 검찰 언론 장악해서 힘 없는 근로자, 안테세력으로 범법자 역사나 챙겨보고 지껄여..........이제 국민들이 들고 일어나니 하는 척 하자나...그깟 주차장?',
 '미국의 실정을 잘 모르면서 여러가지 문제를 이슈화 하는 것은 그냥 뇌피셜입니다.\n하다못해 삼성 SK측에 왜 미국에 공장을 짓는지 그 이유를 인터뷰하고나서 영상을 만들었어야 했습니다.\n한국의 노조가 힘들게 한다구요? 미국의 노조는 더 강합니다.\n미국 자동차 공장들이 문을 닫고 해외로 나가는 이유가 이것이죠.\n힌국에서는 왜 안전관리에 투자하거나 관심을 두지 않습니까?\n미국에서는 안전사고로 단 명만 사망해도 징벌적 배상을 명하기 때문에 큰 회사가 흔들릴 정도입니다. 이런 우리가 잘 모르는 내용을 캐내서 기사화 해야 팩트를 전달할 수 있을 겁니다. 참고로 저는 미국에서 40년 살고 한국에 나와 지내고 있습니다.\n미국이 삼성에 땅을 그대로 주고 길 이름도 삼성 현대로 지어준다구요?\n그야 아무 것도 없는 허허벌판을 개발해서 공장지으니 길도 내주고 이름도 지어주는 것이죠.  한국에서도 시골 벌판에 그렇게 한다면 왜 안해줄까요?\n한국에서 공장을 왜 대도시 근처에 지으니까 항상 문제가 되는 것이겠죠.',
 '이과적인 마인드를 가진 사람이 이렇게 친재벌주의적인 영상을 만드는 걸보니 참 어이없고 기가 찹니다. 신문기사에 나오는 몇줄의 잘못된 팩트를 벗어난 찌라시를 사실인양 호도하지 마세요.',
 '제발 무식한 소리좀 그만해요\n외국은 유치하려고 애쓰는데 한국은 님비현상이 있다고?\n\n하 정말 ᆢ\n그래서 그 유치하는 미국지역 가본적이 있나?\n제발 삼성 쉴드치려고 나온 삼성맨 아니면 그냥 입닫고 계세요\n\n전기풍부하고 용수풍부하고 행정 인허가해줄 지역 넘쳐납니다\n\n새만금하고 비교해라\n경기도 요충지역하고 비교하지말고',
 '솔직히 대기업에 대한 걱정은 일반 국민이 할 필요없습니다. 알아서 잘 합니다.',
 '이재용아나 최태원이었으면 미국간다고\n어지간히 빨아라 \n그렇게 한국에 공장 짖고 싶으면 수도권 말고 \n지방도 땅 많은데 왜 수도권만 고집하는지는 \n다루지 않냐?',
 '내란당 굥두환 집권할때는 쫄아서 한마디도 못하더니....그니까 투표잘해라... 이렇게 시급하고 중대한 시점에 억지로 떡볶이 먹인넘은 진짜...',
 '공정하게 자료를 가지고  비판해라   삼성 하이닉스  물고 빨고하지말고 ......',
 '호남으로 가면됫ㅇ 전력도가능하고 남는땅도많음 ㅋㅋ지역발전도되고 저리반대하는데 굳이 ㅋㅋ',
 '그런데 지역식당 이용해주면 지역주민들한테 도움이되니까.\n가능하면 이용해주면좋죠',
 '대기업들 사내식당 운영하지 않았으면 좋겠습니다.',
 '극좌 정치인들의 무능함이 대한민국을 병들게 합니다... 오염수 외치며 반대한 지금의 여당과 이무것도 막지 못하는 야당에 국민들만 털리는 현실 나 같아도 한국에서 기업 않한다...\n\n현대차가 아틀란스 공개하고 경고해도 노조는 파업 하는 나라...노봉투법도 통과인데 누가 한국에 투자하나??자기 살려는 쎄쎄 하는 수장인데...',
 '근데, 이재용 재산 상속하라고 국민연금 어마어마하게 손해 봤고, 주식 양도세도 100억으로 올려주고, 전기세등등 파격적으로 싸게 해주지 않나?',
 '주식 10% 달라잖아 미국이',
 '미국가면 거져 사업하니?',
 '누가 용인에다 공장 지으래? 나중에 여차하면 땅값이라도 건질생각으로 한거잖아? 힘들게 용인을 고집한 이유가 뭐야? 호남지역 땅 남아 도는데, 지어달라고 해도 안했잖아. 공장이 수도권에 있어야할 이유가 뭐야? 인재유치? ㅋㅋㅋ 공장이 무슨 인재유치야? 연구소라면 그럴 수 있지만… 지금와서 뭔소리여.',
 '아무것도모르는사람인제가보기엔  미국이   한국의  삼성을  조금씩   강탈하는것으로보여지는건  저만느끼는것일수도있죠 \n남녀관계로보아도  잘해주는  남자쪽으로간건  여자의잘못일까요 못해주 남자잘못일까요\n잘해주는 남자쪽으로가는  여자를욕하는  바보가되기전에  잘해주는 남자로변하면 정들고잘해주는쪽에서 행복해지지않을까요',
 '극우도문제지만\n좌파들때문에 나라먕해간다',
 '삼성매장만 가봐도 알죠. 직원들 그냥 고객 관심 없습니다. 이상한게 as 직원들 자기업무 아니라고 그냥 보내고 방문서비스로 돌려요. 방문서비스 직원 말하길"와, 이런 문제를 방문서비스로 돌려? " 얘네들 돈을 왜 주는지 몰라..',
 '한국대기업들한테 세금으로 얼마나 많은 지원을 해줬는데 이런 헛소리를 ᆢ해대고 있네\n\n미국에 못이기는 척 진출하는것은 또다른 목적이 있어서 못이기는척 기어나가는 것이지\n\n용인 크러스트하면서 금강이남으로 사업지 유도를 하니 인력들이 금강이남은 안갈려고해서 힘들다고 개소리를 하면서 그 먼 미국땅으로 파견가고 돈 조금 더 주면 중공까지 가서 기술유출하고 자빠지는데 ᆢ\n\n먼 헛소리를 이리도 장황하게 나열하는지 ᆢ ㅉㅉㅉ',
 '잘못한게 더 많아요 잘한것보다. 다 나열해 볼까여? 밤새요 \n잘한걸 칭찬할수 있지만 대기업이 지금의 대기업이 된것은\n국민들 덕입니다. 그건 변하지 않아요 \n님께서 그렇게 칭찬안해도 알건 다알아요 하지만 \n그렇게 대놓구 이야기 안해도 안다는거 그렇게 무엇인가를 찾아서 \n칭찬해주는것보다 오른손이 하는일을 왼손이 모르게 하는게 \n더 멋져 보인답니다 대놓고 아부하는것처럼 보여서 좀 그러네요 ㅎㅎ',
 '광주로 보내요.전기도 물도 땅값도 저렴하고 .\n정부에서 정리를 하는 것이 첫재 표가 문제지요.\n아마도 지방에서는 온다고만 하면 그보다도 더 후허게 맞이 할 것 입니다.',
 '그럼에도 이재명 뽑잖아??? 15만원 지돈으로 내는 줄 모르고 배급문학 써대면서 치킨 뜯는 동안 관세 협상 조지고, 노란봉투법 날치기 통과시켜서 외국 기업은 물론 국내 기업도 떠나게 생겼고 ㅋㅋㅋ',
 '경제 망하고나야 손 비비면서 와달라고 애걸복걸하겠지.',
 '안타깝지만 이것또한 대기업이 그동안 해왔던 것에 대한 업보라 생각 한다. 어렵고 힘들때 국민들 피빨아 먹고 성장 했으면 너무 징징 대지 마라.',
 '노란봉투법에 비하면 전부다 귀여운 수준이네']

In [10]:
import re

# 첫줄 광고 지우기
data = [c for c in data if '리포트' not in c and '뉴스레터' not in c]

for i in range(len(data)):
    # 개행 문자(\n, \t) 공백으로 대체
    data[i] = re.sub(r'\n', '', data[i])
    
    # 이모티콘 및 특수문자 제거
    data[i] = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', data[i])

    # # 양 끝 공백 제거
    # data[i] = data[i].strip()

In [19]:
data

['회사 문닫고 떠나면 장사안된다고 징징거릴 상인들',
 '구구절절 맞는 말씀입니다 기업이 살아야 나라가 살고 국민이 삽니다',
 '감사합니다 정확한 지적이십니다 삼성 현대   sk 등 우리 기업들과 우리나라를 응원합니다',
 '이재용회장님을 언제나 응원합니다또한 대한민국 국민의 한 국민으로서 그 노고를 참 치하합니다 저는 할머니입니다',
 '국내 투자하는 기업에 우대해줘야 합니다그래야 일자리가 생기죠',
 '삼성 SK응원합니다 겨우4명 먹여 살리는 초초소기업 운영하는데도 이리 힘든데 삼성SK 같이 몇십만명 몇백만명 먹여 살리는 두분보면 존경스럽기까지 하다',
 '너무 좋은 내용입니다 기업을 키워야 국가가 부흥하지요응원합니다',
 '부산살다가 청주산지 10년 되었습니다  전국이 다 불경기인데 청주는 하이닉스 때문인지   호황기인거 같습니다  월세가 없습니다  전국에서 몰려온 하이닉스 공사인부들때문에   원룸  모텔아파트까지 월세를 구할수가없을정도입니다  대기업의 역활이 얼마나 큰지 세삼 느낍니다  한국이 계속 경제가발전하는데  모든국민이  노력해주는게 맞는거같습니다 결국은 나라가 결국은 국민이 혜택을 입는것 같습니다',
 '저 지역주민들은 굶어봐야 정신차리지',
 '이런방송 잘하고 있어요우리가 미처몰랏던 사실들 알려줘서 고마워요 ',
 '구내식당 만들려고하면 주변 식당가들에서 시위하는게 짜치는게 전 회사가 신사옥 지을때 구내식당 만들려다가 구청장 까지와서 주변 식당이랑 상생하라고 만들지 말라고 압박하더니 결국 구내식당 없는데 주변식당 가격들 한끼에 1000012000원 수준 판교에서도 주변식당가들 직원식대 올랐다고 식당 사장님이 이번에 식대 올랐다면서요 물어보고 오른만큼 가격올린것도 있고 뭔 주변에 죄다 남 피빨아 먹으려는것들밖에 없음',
 '이재용 회장님  힘내세요대한민국  화이팅올바른 사람이  한명이라도 많으면  그것이  승리  가  될것임',
 '군부대 떠나는 강원도 한도시의 업자들이 생각나네요선을 넘는 인간들은 어디든 있죠',
 '이런방송 정말  멋진 

In [20]:
ko_stopwords = [
"아","휴","아이구","아이쿠","아이고","어","나","우리","저희","따라","의해","을","를","에","의","가","으로","로","에게",
"뿐이다","의거하여","근거하여","입각하여","기준으로","예하면","예를 들면","예를 들자면","저","소인","소생","저희",
"지말고","하지마","하지마라","다른","물론","또한","그리고","비길수 없다","해서는 안된다","뿐만 아니라","만이 아니다",
"만은 아니다","막론하고","관계없이","그치지 않다","그러나","그런데","하지만","든간에","논하지 않다","따지지 않다",
"설사","비록","더라도","아니면","만 못하다","하는 편이 낫다","불문하고","향하여","향해서","향하다","쪽으로","틈타",
"이용하여","타다","오르다","제외하고","이 외에","이 밖에","하여야","비로소","한다면 몰라도","외에도","이곳","여기",
"부터","기점으로","따라서","할 생각이다","하려고하다","이리하여","그리하여","그렇게 함으로써","일때","할때",
"앞에서","중에서","보는데서","으로써","로써","까지","해야한다","일것이다","반드시","할줄알다","할수있다",
"할수있어","임에 틀림없다","한다면","등","등등","제","겨우","단지","다만","할뿐","딩동","댕그","대해서",
"대하여","대하면","훨씬","얼마나","얼마만큼","얼마큼","남짓","여","얼마간","약간","다소","좀","조금","다수",
"몇","얼마","지만","하물며","또한","그러나","그렇지만","하지만","이외에도","대해 말하자면","뿐이다","다음에",
"반대로","반대로 말하자면","이와 반대로","바꾸어서 말하면","바꾸어서 한다면","만약","그렇지않으면","까악","툭",
"딱","삐걱거리다","보드득","비걱거리다","꽈당","응당","해야한다","에 가서","각","각각","여러분","각종",
"각자","제각기","하도록하다","와","과","그러므로","그래서","고로","한 까닭에","하기 때문에","거니와",
"이지만","대하여","관하여","관한","과연","실로","아니나다를가","생각한대로","진짜로","한적이있다",
"하곤하였다","하","하하","허허","아하","거바","와","오","왜","어째서","무엇때문에","어찌","하겠는가",
"무슨","어디","어느곳","더군다나","하물며","더욱이는","어느때","언제","야","이봐","어이","여보시오",
"흐흐","흥","휴","헉헉","헐떡헐떡","영차","여차","어기여차","끙끙","아야","앗","아야","콸콸","졸졸",
"좍좍","뚝뚝","주룩주룩","솨","우르르","그래도","또","그리고","바꾸어말하면","바꾸어말하자면","혹은",
"혹시","답다","및","그에 따르는","때가 되어","즉","지든지","설령","가령","하더라도","할지라도",
"일지라도","몇","거의","하마터면","인젠","이젠","된바에야","된이상","만큼","어찌됏든","그위에",
"게다가","점에서 보아","비추어 보아","고려하면","하게될것이다","비교적","보다더","비하면",
"시키다","하게하다","할만하다","의해서","연이서","이어서","잇따라","뒤따라","뒤이어","결국",
"의지하여","기대여","통하여","자마자","더욱더","불구하고","얼마든지","마음대로","주저하지 않고",
"곧","즉시","바로","당장","하자마자","밖에 안된다","하면된다","그래","그렇지","요컨대",
"다시 말하자면","바꿔 말하면","구체적으로","말하자면","시작하여","시초에","이상","허","헉","허걱",
"바와같이","해도좋다","해도된다","더구나","와르르","팍","퍽","펄렁","동안","이래",
"하고있었다","이었다","에서","로부터","함께","같이","더불어","마저","마저도","양자","모두",
"습니다","가까스로","즈음하여","다른 방면으로","해봐요","습니까","했어요","말할것도 없고",
"무릎쓰고","개의치않고","하는것만 못하다","하는것이 낫다","매","매번","들","모",
"어느것","어느","갖고말하자면","어느쪽","어느해","어느 년도","라 해도","언젠가",
"어떤것","저기","저쪽","저것","그때","그럼","그러면","요만한걸","저것만큼",
"그저","이르기까지","할 줄 안다","할 힘이 있다","너","너희","당신","설마","차라리",
"할지언정","할망정","구토하다","게우다","토하다","메쓰겁다","옆사람","퉤","쳇",
"힘입어","그","다음","버금","두번째로","기타","첫번째로","나머지는","그중에서",
"견지에서","형식으로 쓰여","입장에서","위해서","의해되다","하도록시키다","뿐만아니라",
"전후","전자","앞의것","잠시","잠깐","하면서","그러한즉","그런즉","남들","아무거나",
"어찌하든지","같다","비슷하다","예컨대","이럴정도로","어떻게","만일","위에서 서술한바와같이",
"인 듯하다","하지 않는다면","만약에","무엇","어느","어떤","아래윗","조차","한데",
"그럼에도 불구하고","여전히","심지어","까지도","조차도","하지 않도록","않기 위하여",
"때","시각","무렵","시간","어때","어떠한","하여금","네","예","우선","누구",
"누가 알겠는가","아무도","줄은모른다","줄은 몰랏다","하는 김에","겸사겸사","하는바",
"그런 까닭에","한 이유는","그러니","그러니까","때문에","그들","너희들","타인","것","것들",
"위하여","공동으로","동시에","하기 위하여","어찌하여","붕붕","윙윙","엉엉","휘익",
"오호","어쨋든","하기보다는","놀라다","상대적으로 말하자면","마치","아니라면","쉿",
"그렇지 않으면","그렇지 않다면","안 그러면","아니었다면","하든지","이라면","좋아",
"알았어","하는것도","그만이다","어쩔수 없다","하나","일","일반적으로","일단",
"한켠으로는","오자마자","이렇게되면","이와같다면","전부","한마디","한항목",
"근거로","하기에","아울러","않기 위해서","이 되다","로 인하여","까닭으로",
"이유만으로","이로 인하여","이 때문에","알 수 있다","결론을 낼 수 있다",
"으로 인하여","있다","관계가 있다","관련이 있다","연관되다","에 대해","여부",
"하느니","하면 할수록","운운","이러이러하다","하구나","하도다","다시말하면",
"다음으로","에 있다","에 달려 있다","우리들","오히려","하기는한데",
"어떻해","어찌됏어","본대로","자","이","이쪽","이것","이번",
"이렇게말하자면","이런","이러한","이와 같은","요만큼","요만한 것",
"얼마 안 되는 것","이만큼","이 정도의","이렇게 많은 것",
"이와 같다","이때","이렇구나","것과 같이","끼익","삐걱",
"따위","와 같은 사람들","부류의 사람들","왜냐하면","중의하나",
"오직","오로지","에 한하다","하기만 하면","도착하다",
"까지 미치다","도달하다","정도에 이르다","할 지경이다",
"결과에 이르다","관해서는","하고 있다","한 후","혼자",
"자기","자기집","자신","우에 종합한것과같이","총적으로 보면",
"총적으로 말하면","총적으로","대로 하다","으로서","참",
"할 따름이다","쿵","탕탕","쾅쾅","둥둥","봐","봐라",
"아이야","아니","와아","응","아이","참나","년","월","일",
"령","영","이","삼","사","오","육","륙","칠","팔","구",
"이천육","이천칠","이천팔","이천구","하나","둘","셋","넷",
"다섯","여섯","일곱","여덟","아홉",'의','가','이','은',
'들','는','좀','잘','걍','과','도','를','으로','자','에',
'와','한','하다','에서','입니다','있습니다','아니라', '이다'
]

len(ko_stopwords)

634

In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize
from konlpy.tag import Okt

okt = Okt()

preprocessed_text = []

for comments in data:
    sentences = sent_tokenize(comments)
    for sentence in sentences:
        tokens = okt.morphs(sentence, stem=True)        # 토큰화 및 어간 추출
        tokens = [token for token in tokens if token not in ko_stopwords]   # 불용어 제거
        tokens = [token for token in tokens if len(token)]
        preprocessed_text.append(tokens)

TypeError: Okt.nouns() got an unexpected keyword argument 'stem'

In [ ]:
preprocessed_text

[['회사', '문', '닫다', '떠나다', '장사', '안되다', '징징거리다', '상', '인들'],
 ['절절', '맞다', '말씀', '기업', '살다', '나라', '살', '고', '국민', '사다'],
 ['감사하다', '정확하다', '지적', '이십', '니', '다', '삼성', '현대', 'sk', '기업', '우리나라', '응원'],
 ['이재용', '회장', '님', '언제나', '응원', '대한민국', '국민', '국민', '노고', '차다', '치하', '할머니'],
 ['국내', '투자', '기업', '우대', '해주다', '래야', '일자리', '생기', '죠'],
 ['삼성',
  'SK',
  '응원',
  '4',
  '명',
  '먹이다',
  '살리다',
  '초',
  '초소',
  '기업',
  '운영',
  '이리',
  '힘드다',
  '삼성',
  'SK',
  '몇십',
  '만',
  '명',
  '백만',
  '명',
  '먹이다',
  '살리다',
  '두',
  '분',
  '보다',
  '존경',
  '스럽다'],
 ['너무', '좋다', '내용', '기업', '키우다', '국가', '부흥', '응원'],
 ['부산',
  '살다',
  '청주',
  '산지',
  '10년',
  '되어다',
  '전국',
  '다',
  '불경기',
  '인데',
  '청주',
  '하이닉스',
  '때문',
  '인지',
  '호황',
  '기',
  '인거',
  '월세',
  '없다',
  '전국',
  '몰려오다',
  '하이닉스',
  '공',
  '인부',
  '때문',
  '원룸',
  '모텔',
  '아파트',
  '월세',
  '없다',
  '정도',
  '대기업',
  '역활',
  '크다',
  '세',
  '느끼다',
  '한국',
  '계속',
  '경제',
  '발전',
  '모든',
  '국민',
  '노력',
  '해주다',
  '맞다',
  '나라',
  '국민',
  '혜택',
  '